In [ ]:
import sys, subprocess
print(sys.version)
print(subprocess.run(['git', '--version'], capture_output=True, text=True).stdout)
print(subprocess.run(['pip', '--version'], capture_output=True, text=True).stdout)

In [ ]:
import json
WORKER_SRC = "\"\"\"Validate one SWE-rebench task in a plain online environment (used by the G7 spike notebook; dry-run locally first).\n\nrun_task(task, work_root) -> dict with a verdict:\n  valid             FAIL_TO_PASS all fail/error before the gold patch and all pass after; PASS_TO_PASS all pass after\n  p2p_partial       FAIL_TO_PASS behave, but some PASS_TO_PASS tests do not pass after (usable with those tests dropped)\n  f2p_passes_before at least one FAIL_TO_PASS test already passes with only the test patch (task does not test the fix)\n  f2p_fails_after   the gold patch does not make FAIL_TO_PASS pass in this environment\n  install_failed / clone_failed / patch_apply_failed / no_tests_ran / timeout\n\"\"\"\n\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nALIASES = {\"yaml\": \"pyyaml\", \"PIL\": \"pillow\", \"cv2\": \"opencv-python-headless\", \"sklearn\": \"scikit-learn\", \"dateutil\": \"python-dateutil\", \"bs4\": \"beautifulsoup4\", \"attr\": \"attrs\", \"OpenSSL\": \"pyopenssl\", \"jwt\": \"pyjwt\", \"git\": \"gitpython\", \"magic\": \"python-magic\", \"serial\": \"pyserial\", \"usb\": \"pyusb\"}\nSTATUS = re.compile(r\"^(PASSED|FAILED|ERROR|SKIPPED|XFAIL|XPASS)\\s+(.+?)(?:\\s+-\\s+.*)?$\", re.M)\n\n\ndef _run(cmd, cwd, timeout, env, shell=False, inp=None):\n    t0 = time.time()\n    try:\n        p = subprocess.run(cmd, cwd=cwd, env=env, shell=shell, capture_output=True, text=True, timeout=timeout, input=inp, errors=\"replace\")\n        return p.returncode, (p.stdout or \"\") + (p.stderr or \"\"), time.time() - t0\n    except subprocess.TimeoutExpired as e:\n        return -999, ((e.stdout or b\"\").decode(\"utf-8\", \"replace\") if isinstance(e.stdout, bytes) else (e.stdout or \"\")) + \"\\n[timeout]\", time.time() - t0\n\n\ndef _tail(s, n=900):\n    return s[-n:] if s else \"\"\n\n\ndef _statuses(out):\n    return {m.group(2).strip(): m.group(1) for m in STATUS.finditer(out)}\n\n\ndef run_task(task, work_root, keep=False):\n    iid, repo = task[\"instance_id\"], task[\"repo\"]\n    d = Path(work_root) / iid\n    shutil.rmtree(d, ignore_errors=True)\n    d.mkdir(parents=True)\n    rdir, vdir = d / \"repo\", d / \"venv\"\n    res = {\"id\": iid, \"repo\": repo, \"python_requested\": (task.get(\"install_config\") or {}).get(\"python\"), \"times\": {}}\n    binname = \"Scripts\" if os.name == \"nt\" else \"bin\"\n    env = dict(os.environ)\n    env.update({\"PIP_DISABLE_PIP_VERSION_CHECK\": \"1\", \"GIT_TERMINAL_PROMPT\": \"0\", \"PYTHONDONTWRITEBYTECODE\": \"1\"})\n    for k, v in ((task.get(\"install_config\") or {}).get(\"env_vars\") or {}).items():\n        if v is not None:\n            env[k] = str(v)\n    try:\n        # ---- clone at base_commit\n        rdir.mkdir()\n        for cmd in ([\"git\", \"init\", \"-q\"], [\"git\", \"remote\", \"add\", \"origin\", f\"https://github.com/{repo}.git\"]):\n            _run(cmd, rdir, 120, env)\n        rc, out, dt = _run([\"git\", \"fetch\", \"-q\", \"--depth\", \"1\", \"origin\", task[\"base_commit\"]], rdir, 900, env)\n        if rc != 0:\n            rc, out, dt2 = _run([\"git\", \"fetch\", \"-q\", \"origin\", task[\"base_commit\"]], rdir, 1500, env)\n            dt += dt2\n        if rc == 0:\n            rc, out2, dt3 = _run([\"git\", \"checkout\", \"-q\", \"FETCH_HEAD\"], rdir, 300, env)\n            out += out2\n            dt += dt3\n        res[\"times\"][\"clone\"] = round(dt)\n        if rc != 0:\n            res.update(verdict=\"clone_failed\", reason=_tail(out, 300))\n            return res\n        # ---- venv + install\n        # Kaggle's image has a broken `ensurepip` (python -m venv fails), so use virtualenv, which brings its own seed wheels\n        rc, out, dt = (1, \"forced\", 0) if os.environ.get(\"G7_FORCE_VIRTUALENV\") else _run([sys.executable, \"-m\", \"venv\", str(vdir)], d, 300, env)\n        if rc != 0:\n            shutil.rmtree(vdir, ignore_errors=True)\n            rc, out, dt = _run([sys.executable, \"-m\", \"virtualenv\", \"-q\", str(vdir)], d, 300, env)\n            if rc != 0 and \"No module named virtualenv\" in out:\n                _run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"virtualenv\"], d, 300, env)\n                rc, out, dt = _run([sys.executable, \"-m\", \"virtualenv\", \"-q\", str(vdir)], d, 300, env)\n        if rc != 0:\n            res.update(verdict=\"install_failed\", reason=\"venv/virtualenv: \" + _tail(out, 300))\n            return res\n        venv_bin = vdir / binname\n        env[\"VIRTUAL_ENV\"] = str(vdir)\n        env[\"PATH\"] = str(venv_bin) + os.pathsep + env.get(\"PATH\", \"\")\n        vpy = str(venv_bin / (\"python.exe\" if os.name == \"nt\" else \"python\"))\n        rc, out, dt = _run([vpy, \"-m\", \"pip\", \"install\", \"-q\", \"pytest\", \"pytest-timeout\"], rdir, 600, env)\n        ic = task.get(\"install_config\") or {}\n        steps = list(ic.get(\"pre_install\") or []) + ([ic[\"install\"]] if ic.get(\"install\") else [\"pip install -e .\"])\n        install_out = \"\"\n        for s in steps:\n            rc, o, dt2 = _run(s, rdir, 1500, env, shell=True)\n            dt += dt2\n            install_out += f\"$ {s}\\n{_tail(o, 600)}\\n\"\n            if rc != 0:\n                res[\"times\"][\"install\"] = round(dt)\n                res.update(verdict=\"install_failed\", reason=f\"`{s}` rc={rc}: {_tail(o, 400)}\")\n                return res\n        pkgs = [x for x in (ic.get(\"pip_packages\") or []) if x]\n        if pkgs:\n            rc, o, dt2 = _run([vpy, \"-m\", \"pip\", \"install\", \"-q\", *pkgs], rdir, 900, env)\n            dt += dt2\n            res[\"pip_packages\"] = {\"n\": len(pkgs), \"rc\": rc}\n        res[\"times\"][\"install\"] = round(dt)\n        rc, out, _ = _run([vpy, \"-m\", \"pip\", \"freeze\"], rdir, 120, env)\n        res[\"n_packages\"] = len([l for l in out.splitlines() if l.strip()])\n        res[\"freeze\"] = out.splitlines()[:400]\n        # ---- tests before (test patch only)\n        for name, patch in ((\"test_patch\", task[\"test_patch\"]),):\n            rc, o, _ = _run([\"git\", \"apply\", \"--whitespace=nowarn\", \"-\"], rdir, 120, env, inp=patch)\n            if rc != 0:\n                res.update(verdict=\"patch_apply_failed\", reason=f\"{name}: {_tail(o, 300)}\")\n                return res\n        f2p, p2p = list(task[\"FAIL_TO_PASS\"]), list(task[\"PASS_TO_PASS\"])[:150]\n        base = [vpy, \"-m\", \"pytest\", \"-rA\", \"--tb=line\", \"-q\", \"-p\", \"no:cacheprovider\", \"--timeout=300\"]\n        added = []\n\n        def run_with_deps(ids, timeout):\n            \"\"\"Run pytest; if collection dies on `No module named X`, pip install X and retry (max 4 times). This mimics a shared wheel set that holds test deps.\"\"\"\n            for _ in range(5):\n                rc_, o_, dt_ = _run(base + ids, rdir, timeout, env)\n                miss = re.findall(r\"No module named '([A-Za-z0-9_\\-]+)\", o_)\n                miss = [m for m in dict.fromkeys(miss) if m not in added and m.split(\".\")[0] not in added]\n                if rc_ == -999 or not miss or len(added) >= 8:\n                    return rc_, o_, dt_\n                for m in miss[:3]:\n                    added.append(m)\n                    _run([vpy, \"-m\", \"pip\", \"install\", \"-q\", ALIASES.get(m, m)], rdir, 300, env)\n            return rc_, o_, dt_\n\n        rc, out_b, dt = run_with_deps(f2p, 1200)\n        res[\"times\"][\"tests_before\"] = round(dt)\n        sb = _statuses(out_b)\n        if rc == -999:\n            res.update(verdict=\"timeout\", reason=\"tests before the patch timed out\")\n            return res\n        if not any(i in sb for i in f2p) and not sb:\n            res.update(verdict=\"no_tests_ran\", reason=_tail(out_b, 400))\n            return res\n        passed_before = [i for i in f2p if sb.get(i) == \"PASSED\"]\n        # ---- gold patch, tests after\n        rc, o, _ = _run([\"git\", \"apply\", \"--whitespace=nowarn\", \"-\"], rdir, 120, env, inp=task[\"patch\"])\n        if rc != 0:\n            res.update(verdict=\"patch_apply_failed\", reason=\"gold patch: \" + _tail(o, 300))\n            return res\n        rc, out_a, dt = run_with_deps(f2p + p2p, 1800)\n        res[\"extra_modules_installed\"] = list(added)\n        res[\"times\"][\"tests_after\"] = round(dt)\n        sa = _statuses(out_a)\n        if rc == -999:\n            res.update(verdict=\"timeout\", reason=\"tests after the patch timed out\")\n            return res\n        f2p_ok = [i for i in f2p if sa.get(i) == \"PASSED\"]\n        p2p_ok = [i for i in p2p if sa.get(i) == \"PASSED\"]\n        res.update(f2p=len(f2p), f2p_pass_before=len(passed_before), f2p_pass_after=len(f2p_ok), p2p=len(p2p), p2p_pass_after=len(p2p_ok))\n        if passed_before:\n            res.update(verdict=\"f2p_passes_before\", reason=f\"{len(passed_before)}/{len(f2p)} F2P already pass with only the test patch\")\n        elif len(f2p_ok) != len(f2p):\n            bad = [(i, sa.get(i)) for i in f2p if sa.get(i) != \"PASSED\"][:3]\n            errs = [l for l in out_a.splitlines() if (\"Error\" in l or \"error\" in l) and \"PASSED\" not in l][:4]\n            res.update(verdict=\"f2p_fails_after\", reason=f\"{bad} | {' / '.join(errs)[:500]} | {_tail(out_a, 200)}\")\n        elif len(p2p_ok) != len(p2p):\n            res.update(verdict=\"p2p_partial\", reason=f\"{len(p2p) - len(p2p_ok)}/{len(p2p)} P2P not passing after the gold patch\")\n        else:\n            res.update(verdict=\"valid\", reason=\"\")\n        return res\n    except Exception as e:  # never let one task kill the spike\n        res.update(verdict=\"harness_error\", reason=repr(e)[:300])\n        return res\n    finally:\n        if not keep:\n            shutil.rmtree(d, ignore_errors=True)\n"
TASKS = json.loads('[{"instance_id": "cdent__gabbi-303", "repo": "cdent/gabbi", "base_commit": "a8d200bdfcaabb2351b2ccb8f9a2b4de693b2b81", "patch": "diff --git a/gabbi/case.py b/gabbi/case.py\\nindex 6b7823c..c1c0dec 100644\\n--- a/gabbi/case.py\\n+++ b/gabbi/case.py\\n@@ -595,8 +595,12 @@ class HTTPTestCase(unittest.TestCase):\\n                 data = self.replace_template(data)\\n                 data = dumper_class.dumps(data, test=self)\\n             else:\\n-                raise ValueError(\\n-                    \'unable to process data to %s\' % content_type)\\n+                if content_type:\\n+                    raise ValueError(\\n+                        \'unable to process data to %s\' % content_type)\\n+                else:\\n+                    raise ValueError(\\n+                        \'no content-type available for processing data\')\\n \\n         data = self.replace_template(data)\\n \\n", "test_patch": "diff --git a/gabbi/tests/test_data_to_string.py b/gabbi/tests/test_data_to_string.py\\nnew file mode 100644\\nindex 0000000..d3150ba\\n--- /dev/null\\n+++ b/gabbi/tests/test_data_to_string.py\\n@@ -0,0 +1,54 @@\\n+#\\n+# Licensed under the Apache License, Version 2.0 (the \\"License\\"); you may\\n+# not use this file except in compliance with the License. You may obtain\\n+# a copy of the License at\\n+#\\n+#      http://www.apache.org/licenses/LICENSE-2.0\\n+#\\n+# Unless required by applicable law or agreed to in writing, software\\n+# distributed under the License is distributed on an \\"AS IS\\" BASIS, WITHOUT\\n+# WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the\\n+# License for the specific language governing permissions and limitations\\n+# under the License.\\n+\\"\\"\\"Test handling of data field in tests.\\n+\\"\\"\\"\\n+\\n+import unittest\\n+\\n+from gabbi import case\\n+from gabbi import handlers\\n+\\n+\\n+class TestDataToString(unittest.TestCase):\\n+\\n+    def setUp(self):\\n+        self.case = case.HTTPTestCase(\'test_request\')\\n+        self.case.content_handlers = []\\n+        for handler in handlers.RESPONSE_HANDLERS:\\n+            h = handler()\\n+            if hasattr(h, \'content_handler\') and h.content_handler:\\n+                self.case.content_handlers.append(h)\\n+\\n+    def testHappyPath(self):\\n+        data = [{\\"hi\\": \\"low\\"}, {\\"yes\\": \\"no\\"}]\\n+        content_type = \'application/json\'\\n+        body = self.case._test_data_to_string(data, content_type)\\n+        self.assertEqual(\'[{\\"hi\\": \\"low\\"}, {\\"yes\\": \\"no\\"}]\', body)\\n+\\n+    def testNoContentType(self):\\n+        data = [{\\"hi\\": \\"low\\"}, {\\"yes\\": \\"no\\"}]\\n+        content_type = \'\'\\n+        with self.assertRaises(ValueError) as exc:\\n+            self.case._test_data_to_string(data, content_type)\\n+        self.assertEqual(\\n+            \'no content-type available for processing data\',\\n+            str(exc.exception))\\n+\\n+    def testNoHandler(self):\\n+        data = [{\\"hi\\": \\"low\\"}, {\\"yes\\": \\"no\\"}]\\n+        content_type = \'application/xml\'\\n+        with self.assertRaises(ValueError) as exc:\\n+            self.case._test_data_to_string(data, content_type)\\n+        self.assertEqual(\\n+            \'unable to process data to application/xml\',\\n+            str(exc.exception))\\n", "problem_statement": "If content-type is not set, but data is, the error message is incomplete\\nIf you try to write `data` without setting a content-type header, there\'s no content handler and the error message is not very useful:\\r\\n\\r\\n```\\r\\n      File \\"/mnt/share/cdentsrc/nova/.tox/functional/local/lib/python2.7/site-packages/gabbi/suitemaker.py\\", line 95, in do_test\\r\\n        return test_method(*args, **kwargs)\\r\\n      File \\"/mnt/share/cdentsrc/nova/.tox/functional/local/lib/python2.7/site-packages/gabbi/case.py\\", line 94, in wrapper\\r\\n        func(self)\\r\\n      File \\"/mnt/share/cdentsrc/nova/.tox/functional/local/lib/python2.7/site-packages/gabbi/case.py\\", line 148, in test_request\\r\\n        self._run_test()\\r\\n      File \\"/mnt/share/cdentsrc/nova/.tox/functional/local/lib/python2.7/site-packages/gabbi/case.py\\", line 524, in _run_test\\r\\n        utils.extract_content_type(headers, default=\'\')[0])\\r\\n      File \\"/mnt/share/cdentsrc/nova/.tox/functional/local/lib/python2.7/site-packages/gabbi/case.py\\", line 587, in _test_data_to_string\\r\\n        \'unable to process data to %s\' % content_type)\\r\\n    ValueError: unable to process data to \\r\\n```\\r\\n\\r\\nIf content_type is None, we should provide a different message.", "hints_text": "", "created_at": "2021-09-02 17:18:57", "FAIL_TO_PASS": ["gabbi/tests/test_data_to_string.py::TestDataToString::testNoContentType"], "PASS_TO_PASS": ["gabbi/tests/test_data_to_string.py::TestDataToString::testHappyPath", "gabbi/tests/test_data_to_string.py::TestDataToString::testNoHandler"], "environment_setup_commit": "a8d200bdfcaabb2351b2ccb8f9a2b4de693b2b81", "version": "2.2", "license_name": "Apache License 2.0", "install_config": {"python": "3.9", "install": "pip install -e .", "pre_install": null, "pip_packages": ["pytest", "pytest-cov", "pytest-mock"], "reqs_path": ["requirements.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 0, "files": 1}}, {"instance_id": "dag-hammarskjold-library__dlx-402", "repo": "dag-hammarskjold-library/dlx", "base_commit": "5bdb082b8296696afc9c7398822bce8438f59b1e", "patch": "diff --git a/dlx/scripts/init_indexes.py b/dlx/scripts/init_indexes.py\\nindex 7d99357..ad89d0e 100644\\n--- a/dlx/scripts/init_indexes.py\\n+++ b/dlx/scripts/init_indexes.py\\n@@ -56,7 +56,7 @@ def run():\\n                 collation=Config.marc_index_default_collation\\n             )\\n         )\\n-        indexes.append(col.create_index(\'_record_type\', collation=Config.marc_index_default_collation))\\n+        indexes.append(col.create_index(\'_record_type_collated\', collation=Config.marc_index_default_collation))\\n \\n         print(f\'creating text indexes...\')\\n         indexes.append(col.create_index(\'words\', collation=Config.marc_index_default_collation))\\ndiff --git a/dlx/util.py b/dlx/util.py\\nindex 37de978..4d2c016 100644\\n--- a/dlx/util.py\\n+++ b/dlx/util.py\\n@@ -61,7 +61,12 @@ class Table():\\n     def set(self, rowx, field_name, value):\\n         self.index.setdefault(rowx, {})\\n         self.index[rowx].setdefault(field_name, {})\\n-        self.index[rowx][field_name] = value\\n+\\n+        if existing := self.index[rowx].get(field_name):\\n+            # if this is a repeated subfield, append the next value \\n+            self.index[rowx][field_name] += f\'||{value}\'\\n+        else:\\n+            self.index[rowx][field_name] = value\\n \\n         if field_name not in self.header:\\n             self.header.append(field_name)\\n", "test_patch": "diff --git a/tests/test_marcset.py b/tests/test_marcset.py\\nindex 21bac69..d7ff4b8 100644\\n--- a/tests/test_marcset.py\\n+++ b/tests/test_marcset.py\\n@@ -168,7 +168,7 @@ def test_to_csv(db):\\n     bibset = BibSet.from_query({})\\n     bibset.records = list(bibset.records)\\n     #[bib.set(\'001\', None, str(bib.id)) for bib in bibset.records]\\n-    assert bibset.to_csv(write_id=True) == \'1.001,1.245$a,1.245$b,1.245$c,1.520$a,2.520$a,1.650$a,1.710$a\\\\n1,This,is the,title,Description,Repeated subfield,Header,Another header\\\\n2,Another,is the,title,,,Header,\'\\n+    assert bibset.to_csv(write_id=True) == \'1.001,1.245$a,1.245$b,1.245$c,1.520$a,2.520$a,1.650$a,1.710$a\\\\n1,This,is the,title,Description,Another description||Repeated subfield,Header,Another header\\\\n2,Another,is the,title,,,Header,\'\\n \\n     # comma and quote handling\\n     bibs = BibSet()\\n", "problem_statement": "Handle repeated subfields in CSV/TSV export\\nCurrently, in cases of fields with repeated subfield codes, only one of the subfields makes it into the serialization. This is due to the tabular nature of the serialization.\\r\\n\\r\\n- [ ] Concatenate the values of repeated subfields into one string joined by a separator i.e. \\";\\" or \\"||\\"", "hints_text": "", "created_at": "2024-10-21 15:10:39", "FAIL_TO_PASS": ["tests/test_marcset.py::test_to_csv"], "PASS_TO_PASS": ["tests/test_marcset.py::test_mocked", "tests/test_marcset.py::test_init", "tests/test_marcset.py::test_iterate", "tests/test_marcset.py::test_from_query", "tests/test_marcset.py::test_from_ids", "tests/test_marcset.py::test_from_table", "tests/test_marcset.py::test_from_mrk", "tests/test_marcset.py::test_from_xml", "tests/test_marcset.py::test_to_mrc", "tests/test_marcset.py::test_to_mrk", "tests/test_marcset.py::test_to_xml", "tests/test_marcset.py::test_xml_encoding", "tests/test_marcset.py::test_to_str", "tests/test_marcset.py::test_from_aggregation"], "environment_setup_commit": "7bc9c436adc657b48945eafa70a190381864e26c", "version": "1.4", "license_name": null, "install_config": {"python": "3.9", "install": "pip install -e .", "pre_install": [], "pip_packages": ["pytest"], "reqs_path": ["requirements.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 2}}, {"instance_id": "pypa__setuptools_scm-220", "repo": "pypa/setuptools_scm", "base_commit": "632b02b8ad1f10c79d36c03502069d74c9686c73", "patch": "diff --git a/setuptools_scm/hg.py b/setuptools_scm/hg.py\\nindex 9322fb3..0ba1774 100644\\n--- a/setuptools_scm/hg.py\\n+++ b/setuptools_scm/hg.py\\n@@ -8,8 +8,14 @@ FILES_COMMAND = \'hg locate -I .\'\\n def _hg_tagdist_normalize_tagcommit(root, tag, dist, node):\\n     dirty = node.endswith(\'+\')\\n     node = \'h\' + node.strip(\'+\')\\n-    revset = (\\"(branch(.) and tag({tag!r})::. and file(\'re:^(?!\\\\.hgtags).*$\')\\"\\n-              \\" - tag({tag!r}))\\").format(tag=tag)\\n+\\n+    # Detect changes since the specified tag\\n+    revset = (\\"(branch(.)\\"  # look for revisions in this branch only\\n+              \\" and tag({tag!r})::.\\"  # after the last tag\\n+              # ignore commits that only modify .hgtags and nothing else:\\n+              \\" and (merge() or file(\'re:^(?!\\\\.hgtags).*$\'))\\"\\n+              \\" and not tag({tag!r}))\\"  # ignore the tagged commit itself\\n+              ).format(tag=tag)\\n     if tag != \'0.0\':\\n         commits = do([\'hg\', \'log\', \'-r\', revset, \'--template\', \'{node|short}\'],\\n                      root)\\n", "test_patch": "diff --git a/testing/test_mercurial.py b/testing/test_mercurial.py\\nindex 1fe6841..1d91444 100644\\n--- a/testing/test_mercurial.py\\n+++ b/testing/test_mercurial.py\\n@@ -108,3 +108,47 @@ def test_version_in_merge(wd):\\n def test_parse_no_worktree(tmpdir):\\n     ret = parse(str(tmpdir))\\n     assert ret is None\\n+\\n+\\n+@pytest.fixture\\n+def version_1_0(wd):\\n+    wd(\'hg branch default\')\\n+    wd.commit_testfile()\\n+    wd(\'hg tag 1.0 -u test -d \\"0 0\\"\')\\n+    return wd\\n+\\n+\\n+@pytest.fixture\\n+def pre_merge_commit_after_tag(wd, version_1_0):\\n+    wd(\'hg branch testbranch\')\\n+    wd.write(\'branchfile\', \'branchtext\')\\n+    wd(wd.add_command)\\n+    wd.commit()\\n+    wd(\'hg update default\')\\n+    wd(\'hg merge testbranch\')\\n+    return wd\\n+\\n+\\n+@pytest.mark.usefixtures(\\"pre_merge_commit_after_tag\\")\\n+def test_version_bump_before_merge_commit(wd):\\n+    assert wd.version.startswith(\'1.1.dev1+\')\\n+\\n+\\n+@pytest.mark.issue(219)\\n+@pytest.mark.usefixtures(\\"pre_merge_commit_after_tag\\")\\n+def test_version_bump_from_merge_commit(wd):\\n+    wd.commit()\\n+    assert wd.version.startswith(\'1.1.dev3+\')  # issue 219\\n+\\n+\\n+@pytest.mark.usefixtures(\\"version_1_0\\")\\n+def test_version_bump_from_commit_including_hgtag_mods(wd):\\n+    \\"\\"\\" Test the case where a commit includes changes to .hgtags and other files\\n+    \\"\\"\\"\\n+    with wd.cwd.join(\'.hgtags\').open(\'a\') as tagfile:\\n+        tagfile.write(\'0  0\\\\n\')\\n+    wd.write(\'branchfile\', \'branchtext\')\\n+    wd(wd.add_command)\\n+    assert wd.version.startswith(\'1.1.dev1+\')  # bump from dirty version\\n+    wd.commit()  # commits both the testfile _and_ .hgtags\\n+    assert wd.version.startswith(\'1.1.dev2+\')\\n", "problem_statement": "Regression: hg repos with setuptools_scm>=1.15 are missing dev version bumps\\n`setuptools_scm` 1.15.* is reporting incorrect version numbers for my hg repositories.\\r\\n\\r\\nSpecifically, a merge commit that follows a tag commit does not seem to trigger the version bump like I would expect. In the screenshot below, for instance, I would expect the tip to be version `1.1.dev[N]+[sha]`. However, versions 1.15.* report the tip\'s version as `1.0`.\\r\\n\\r\\nTo reproduce:\\r\\n```bash\\r\\nmkdir hgmergetest && cd hgmergetest && hg init\\r\\n\\r\\n# create initial commit and tag it\\r\\ntouch a && hg add a && hg commit -m \\"initial commit\\" \\r\\nhg tag 1.0\\r\\n\\r\\n# create a branch\\r\\nhg branch branch1\\r\\ntouch b && hg add b && hg commit -m \\"create branch1\\"\\r\\n\\r\\n# merge the branch into default\\r\\nhg update default && hg merge branch1 && hg commit -m \\"merge branch1 into default\\"\\r\\n```\\r\\n\\r\\nExpected behavior (setuptools_scm 1.13 and 1.14):\\r\\n```bash\\r\\n> pip install \\"setuptools_scm<1.15\\"\\r\\n> python -m setuptools_scm\\r\\nGuessed Version 1.1.dev3+n8dce1535e70a\\r\\n```\\r\\n\\r\\nBehavior with `1.15.*`:\\r\\n```bash\\r\\n> pip install setuptools_scm==1.15.7\\r\\n> python -m setuptools_scm\\r\\nGuessed Version 1.0\\r\\n```\\r\\n\\r\\n![image](https://user-images.githubusercontent.com/9388007/37140764-b80a0c36-2267-11e8-87f8-bd52d199140c.png)", "hints_text": "avirshup: Here\'s my stab at a fix for #219. Please let me know if this seems reasonable and/or if there are any changes that should be made.", "created_at": "2018-03-08 19:18:48", "FAIL_TO_PASS": ["testing/test_mercurial.py::test_version_bump_from_merge_commit"], "PASS_TO_PASS": ["testing/test_mercurial.py::test_archival_to_version[0.0-data0]", "testing/test_mercurial.py::test_archival_to_version[1.0-data1]", "testing/test_mercurial.py::test_archival_to_version[1.1.dev3+h000000000000-data2]", "testing/test_mercurial.py::test_archival_to_version[1.2.2-data3]", "testing/test_mercurial.py::test_find_files_stop_at_root_hg", "testing/test_mercurial.py::test_version_from_hg_id", "testing/test_mercurial.py::test_version_from_archival", "testing/test_mercurial.py::test_version_in_merge", "testing/test_mercurial.py::test_parse_no_worktree", "testing/test_mercurial.py::test_version_bump_before_merge_commit", "testing/test_mercurial.py::test_version_bump_from_commit_including_hgtag_mods"], "environment_setup_commit": "0373c11d2c8968a857ff06c94f101abebf825507", "version": "1.15", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .[dev]", "pre_install": null, "pip_packages": ["pytest"], "reqs_path": null, "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 1}}, {"instance_id": "feder-observatory__stellarphot-459", "repo": "feder-observatory/stellarphot", "base_commit": "7943077b29dedf1204b19774632f8de0a2bed0b0", "patch": "diff --git a/stellarphot/core.py b/stellarphot/core.py\\nindex af29601..8ffd81f 100644\\n--- a/stellarphot/core.py\\n+++ b/stellarphot/core.py\\n@@ -613,6 +613,10 @@ class CatalogData(BaseEnhancedTable):\\n         the documentation for `stellarphot.settings.PassbandMap` for more\\n         information. The object behaves like a dictionary when accessing it.\\n \\n+    no_catalog_error: bool, optional (Default: False)\\n+        If True, the catalog data does not contain error information, so a\\n+        column of NaNs will be added for the error values.\\n+\\n     Attributes\\n     ----------\\n     catalog_name: str\\n@@ -643,8 +647,13 @@ class CatalogData(BaseEnhancedTable):\\n     ra                    u.deg\\n     dec                   u.deg\\n     mag                   None\\n+    mag_error             None   (see note below)\\n     passband              None\\n     =================     =======\\n+\\n+    If the catalog data does not contain error information, then set the option\\n+    ``no_catalog_error=True`` when creating the object.  This will add a column\\n+    of NaNs for the error values.\\n     \\"\\"\\"\\n \\n     # Define columns that must be in table and provide information about their type, and\\n@@ -654,6 +663,7 @@ class CatalogData(BaseEnhancedTable):\\n         \\"ra\\": u.deg,\\n         \\"dec\\": u.deg,\\n         \\"mag\\": None,\\n+        \\"mag_error\\": None,\\n         \\"passband\\": None,\\n     }\\n \\n@@ -665,6 +675,7 @@ class CatalogData(BaseEnhancedTable):\\n         catalog_source=None,\\n         colname_map=None,\\n         passband_map=None,\\n+        no_catalog_error=False,\\n         **kwargs,\\n     ):\\n         if (input_data is None) and (catalog_name is None) and (catalog_source is None):\\n@@ -673,11 +684,35 @@ class CatalogData(BaseEnhancedTable):\\n             self._passband_map = passband_map\\n \\n             if input_data is not None:\\n+                # Check whether the colname_map has an entry for mag_error\\n+                # that is None. If it does, and there is no mag_error column\\n+                # in the input data, then add a column of NaNs to the input\\n+                # data for mag_error.\\n+\\n+                if (\\n+                    # colname_map must be provided\\n+                    colname_map is not None\\n+                    and\\n+                    # mag_error must not be in the colname map as a value\\n+                    \\"mag_error\\" not in colname_map.values()\\n+                    and\\n+                    # mag_error must not be in the input data\\n+                    \\"mag_error\\" not in input_data.colnames\\n+                    and\\n+                    # user has opted in to making a column of NaNs\\n+                    no_catalog_error\\n+                ):\\n+                    input_data_copy = input_data.copy()\\n+                    # Make the error column of NaNs\\n+                    input_data_copy[\\"mag_error\\"] = np.full(len(input_data), np.nan)\\n+                else:\\n+                    input_data_copy = input_data\\n+\\n                 # Convert input data to QTable (while also checking for required\\n                 # columns)\\n                 super().__init__(\\n                     table_description=self.catalog_descript,\\n-                    input_data=input_data,\\n+                    input_data=input_data_copy,\\n                     colname_map=colname_map,\\n                     **kwargs,\\n                 )\\n@@ -828,6 +863,7 @@ class CatalogData(BaseEnhancedTable):\\n         mag_column_regex=r\\"^([a-zA-Z]+|[a-zA-Z]+-[a-zA-Z]+)_?mag$\\",\\n         color_column_regex=r\\"^([a-zA-Z]+-[a-zA-Z]+)$\\",\\n         prepare_catalog=None,\\n+        no_catalog_error=False,\\n     ):\\n         \\"\\"\\"\\n         Return the items from catalog that are within the search radius and\\n@@ -872,6 +908,11 @@ class CatalogData(BaseEnhancedTable):\\n         prepare_catalog : callable, optional\\n             Function to call on the catalog after it is retrieved from Vizier.\\n \\n+\\n+        no_catalog_error: bool, optional (Default: False)\\n+            If True, the catalog data does not contain error information, so a\\n+            column of NaNs will be added for the error values.\\n+\\n         Returns\\n         -------\\n \\n@@ -948,6 +989,7 @@ class CatalogData(BaseEnhancedTable):\\n             colname_map=colname_map,\\n             catalog_name=desired_catalog,\\n             catalog_source=\\"Vizier\\",\\n+            no_catalog_error=no_catalog_error,\\n         )\\n \\n         # ...and now that the column names are standardized, clip by frame if\\n@@ -1160,6 +1202,7 @@ def vsx_vizier(field_center, radius=1 * u.degree, clip_by_frame=False, padding=1\\n         padding=padding,\\n         colname_map=vsx_map,\\n         prepare_catalog=prepare_cat,\\n+        no_catalog_error=True,\\n     )\\n \\n \\n", "test_patch": "diff --git a/stellarphot/tests/test_core.py b/stellarphot/tests/test_core.py\\nindex 80964e5..660d237 100644\\n--- a/stellarphot/tests/test_core.py\\n+++ b/stellarphot/tests/test_core.py\\n@@ -723,6 +723,7 @@ def test_catalog_colname_map():\\n         catalog_name=\\"VSX\\",\\n         catalog_source=\\"Vizier\\",\\n         colname_map=vsx_colname_map,\\n+        no_catalog_error=True,\\n     )\\n \\n     assert catalog_dat[\\"id\\"][0] == \\"ASASSN-V J000052.03+002216.6\\"\\n@@ -754,6 +755,7 @@ def test_catalog_bandpassmap():\\n         catalog_source=\\"Vizier\\",\\n         colname_map=vsx_colname_map,\\n         passband_map=passband_map,\\n+        no_catalog_error=True,\\n     )\\n \\n     assert catalog_dat[\\"passband\\"][0] == \\"SG\\"\\n@@ -778,6 +780,7 @@ def test_catalog_recursive():\\n         catalog_name=\\"VSX\\",\\n         catalog_source=\\"Vizier\\",\\n         colname_map=vsx_colname_map,\\n+        no_catalog_error=True,\\n     )\\n \\n     # Attempt recursive call\\n@@ -958,12 +961,16 @@ def test_catalog_from_vizier_search_vsx(location_method):\\n         clip_by_frame=False,\\n         colname_map=vsx_map,\\n         prepare_catalog=prepare_cat,\\n+        no_catalog_error=True,\\n     )\\n \\n     assert my_cat[\\"id\\"][0] == \\"DQ Psc\\"\\n     assert my_cat[\\"passband\\"][0] == \\"Hp\\"\\n     assert my_cat[\\"Type\\"][0] == \\"SRB\\"\\n \\n+    # Make sure has been set to NaN\\n+    assert np.isnan(my_cat[\\"mag_error\\"][0])\\n+\\n \\n def test_from_vizier_with_coord_and_frame_clip_fails():\\n     # Check that calling from_vizier with a coordinate instead\\n@@ -1025,7 +1032,11 @@ def test_vsx_results(clip, data_file):\\n     # Turn this into an HDU to get the standard FITS image keywords\\n     ccd_im = ccd.to_hdu()\\n \\n-    actual = vsx_vizier(ccd_im[0].header, radius=0.5 * u.degree, clip_by_frame=clip)\\n+    actual = vsx_vizier(\\n+        ccd_im[0].header,\\n+        radius=0.5 * u.degree,\\n+        clip_by_frame=clip,\\n+    )\\n     assert set(actual[\\"OID\\"]) == set(expected[\\"OID\\"])\\n \\n \\n", "problem_statement": "`CatalogData` should explicitly allow for an error column \\nI\'ve been working a bit on magnitude transforms and realized that the current definition of CatalogData does not allo for a magnitude error column.\\r\\n\\r\\nWhile an error column probably should not be required, it should exist where possible.", "hints_text": "", "created_at": "2024-10-06 16:53:56", "FAIL_TO_PASS": ["stellarphot/tests/test_core.py::test_catalog_colname_map", "stellarphot/tests/test_core.py::test_catalog_bandpassmap", "stellarphot/tests/test_core.py::test_catalog_recursive"], "PASS_TO_PASS": ["stellarphot/tests/test_core.py::test_base_enhanced_table_blank", "stellarphot/tests/test_core.py::test_base_enhanced_table_from_existing_table", "stellarphot/tests/test_core.py::test_base_enhanced_table_clean", "stellarphot/tests/test_core.py::test_bet_clean_criteria_none_removed", "stellarphot/tests/test_core.py::test_bet_clean_criteria_some_removed[>0-0]", "stellarphot/tests/test_core.py::test_bet_clean_criteria_some_removed[=1-0]", "stellarphot/tests/test_core.py::test_bet_clean_criteria_some_removed[>=1-0]", "stellarphot/tests/test_core.py::test_bet_clean_criteria_some_removed[<-1-2]", "stellarphot/tests/test_core.py::test_bet_clean_criteria_some_removed[=-1-1]", "stellarphot/tests/test_core.py::test_clean_bad_criteria[criteria0-not", "stellarphot/tests/test_core.py::test_clean_bad_criteria[criteria1-could", "stellarphot/tests/test_core.py::test_clean_masked_handled_correctly[False]", "stellarphot/tests/test_core.py::test_clean_masked_handled_correctly[True]", "stellarphot/tests/test_core.py::test_clean_masked_and_criteria", "stellarphot/tests/test_core.py::test_clean_criteria_none_removed", "stellarphot/tests/test_core.py::test_base_enhanced_table_missing_column", "stellarphot/tests/test_core.py::test_base_enhanced_table_missing_badunits", "stellarphot/tests/test_core.py::test_base_enhanced_table_recursive", "stellarphot/tests/test_core.py::test_base_enhanced_table_no_desription", "stellarphot/tests/test_core.py::test_base_enhanced_table_no_data", "stellarphot/tests/test_core.py::test_photometry_blank", "stellarphot/tests/test_core.py::test_photometry_data", "stellarphot/tests/test_core.py::test_photometry_data_short_filter_name", "stellarphot/tests/test_core.py::test_photometry_data_filter_name_map_preserves_original_names", "stellarphot/tests/test_core.py::test_photometry_roundtrip_ecsv", "stellarphot/tests/test_core.py::test_photometry_slicing", "stellarphot/tests/test_core.py::test_photometry_recursive", "stellarphot/tests/test_core.py::test_photometry_badtime", "stellarphot/tests/test_core.py::test_photometry_inconsistent_count_units", "stellarphot/tests/test_core.py::test_photometry_inconsistent_computed_col_exists", "stellarphot/tests/test_core.py::test_catalog_missing_col", "stellarphot/tests/test_core.py::test_tidy_vizier_catalog", "stellarphot/tests/test_core.py::test_tidy_vizier_catalog_several_mags", "stellarphot/tests/test_core.py::test_catalog_to_passband_columns", "stellarphot/tests/test_core.py::test_from_vizier_with_coord_and_frame_clip_fails", "stellarphot/tests/test_core.py::test_from_vizier_with_header_no_wcs_raise_error", "stellarphot/tests/test_core.py::test_sourcelist", "stellarphot/tests/test_core.py::test_sourcelist_no_skypos", "stellarphot/tests/test_core.py::test_sourcelist_no_imgpos", "stellarphot/tests/test_core.py::test_sourcelist_missing_cols", "stellarphot/tests/test_core.py::test_sourcelist_recursive", "stellarphot/tests/test_core.py::test_sourcelist_dropping_skycoords", "stellarphot/tests/test_core.py::test_sourcelist_dropping_imagecoords", "stellarphot/tests/test_core.py::test_sourcelist_slicing"], "environment_setup_commit": "adc4858fef2a9b3f828903752b8f846da16e81b0", "version": "2.0", "license_name": "BSD 3-Clause \\"New\\" or \\"Revised\\" License", "install_config": {"python": "3.10", "install": "pip install -e .[test]", "pre_install": [], "pip_packages": ["pytest-cov", "pytest"], "reqs_path": ["requirements/base.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 1}}, {"instance_id": "SpikeInterface__spikeinterface-2716", "repo": "SpikeInterface/spikeinterface", "base_commit": "482bd7406b790528800d0c6eed99617ee6efff9d", "patch": "diff --git a/src/spikeinterface/curation/remove_excess_spikes.py b/src/spikeinterface/curation/remove_excess_spikes.py\\nindex b9f00212e..52653d684 100644\\n--- a/src/spikeinterface/curation/remove_excess_spikes.py\\n+++ b/src/spikeinterface/curation/remove_excess_spikes.py\\n@@ -79,8 +79,9 @@ class RemoveExcessSpikesSortingSegment(BaseSortingSegment):\\n     ) -> np.ndarray:\\n         spike_train = self._parent_segment.get_unit_spike_train(unit_id, start_frame=start_frame, end_frame=end_frame)\\n         max_spike = np.searchsorted(spike_train, self._num_samples, side=\\"left\\")\\n+        min_spike = np.searchsorted(spike_train, 0, side=\\"left\\")\\n \\n-        return spike_train[:max_spike]\\n+        return spike_train[min_spike:max_spike]\\n \\n \\n def remove_excess_spikes(sorting, recording):\\n", "test_patch": "diff --git a/src/spikeinterface/curation/tests/test_remove_excess_spikes.py b/src/spikeinterface/curation/tests/test_remove_excess_spikes.py\\nindex f99c408c2..7175e0a61 100644\\n--- a/src/spikeinterface/curation/tests/test_remove_excess_spikes.py\\n+++ b/src/spikeinterface/curation/tests/test_remove_excess_spikes.py\\n@@ -14,6 +14,7 @@ def test_remove_excess_spikes():\\n     num_spikes = 100\\n     num_num_samples_spikes_per_segment = 5\\n     num_excess_spikes_per_segment = 5\\n+    num_neg_spike_times_per_segment = 2\\n     times = []\\n     labels = []\\n     for segment_index in range(recording.get_num_segments()):\\n@@ -21,12 +22,15 @@ def test_remove_excess_spikes():\\n         times_segment = np.array([], dtype=int)\\n         labels_segment = np.array([], dtype=int)\\n         for unit in range(num_units):\\n+            neg_spike_times = np.random.randint(-50, 0, num_neg_spike_times_per_segment)\\n             spike_times = np.random.randint(0, num_samples, num_spikes)\\n             last_samples_spikes = (num_samples - 1) * np.ones(num_num_samples_spikes_per_segment, dtype=int)\\n             num_samples_spike_times = num_samples * np.ones(num_num_samples_spikes_per_segment, dtype=int)\\n             excess_spikes = np.random.randint(num_samples, num_samples + 100, num_excess_spikes_per_segment)\\n             spike_times = np.sort(\\n-                np.concatenate((spike_times, last_samples_spikes, num_samples_spike_times, excess_spikes))\\n+                np.concatenate(\\n+                    (neg_spike_times, spike_times, last_samples_spikes, num_samples_spike_times, excess_spikes)\\n+                )\\n             )\\n             spike_labels = unit * np.ones_like(spike_times)\\n             times_segment = np.concatenate((times_segment, spike_times))\\n@@ -47,7 +51,10 @@ def test_remove_excess_spikes():\\n \\n             assert (\\n                 len(spike_train_corrected)\\n-                == len(spike_train_excess) - num_num_samples_spikes_per_segment - num_excess_spikes_per_segment\\n+                == len(spike_train_excess)\\n+                - num_num_samples_spikes_per_segment\\n+                - num_excess_spikes_per_segment\\n+                - num_neg_spike_times_per_segment\\n             )\\n \\n \\n", "problem_statement": "Unable to open kilosort4 results with phy\\nDear,\\r\\n\\r\\nI\'m using `spikeinterface==0.100.5` and `kilosort4== .4.0.3`, on 64 channel 6 shanks probe. The phy output is works perfectly with the  `ironclust ` generated results.\\r\\n\\r\\nHowever, I have encountered some issues when open the kilosort4 phy output folder. And I saw there are some silimer reports, and I tried the following things:\\r\\n1. Before waveform extraction `sorting.remove_empty_units()` and `sc.remove_excess_spikes(sorting_no_empty,rec_corrected1)`\\r\\n2. Set `sparse=True` or `sparse=False` or `sparse=True, method=\\"by_property\\",by_property=\\"group\\"` in the `si.extract_waveforms()`\\r\\n3. `sparsity=None` or `sparsity=True` in the `sexp.export_to_phy()`\\r\\nNone of above was works, and the phy is giving me the error message as below. I\'m wondering if there any solution or any parameter I can adjust?\\r\\n\\r\\nError message:\\r\\n```bash\\r\\n19:51:55.272 [E] __init__:62          An error has occurred (AssertionError):\\r\\nTraceback (most recent call last):\\r\\n  File \\"<frozen runpy>\\", line 198, in _run_module_as_main\\r\\n  File \\"<frozen runpy>\\", line 88, in _run_code\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Scripts\\\\phy.exe\\\\__main__.py\\", line 7, in <module>\\r\\n    sys.exit(phycli())\\r\\n             ^^^^^^^^\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\click\\\\core.py\\", line 1157, in __call__\\r\\n    return self.main(*args, **kwargs)\\r\\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\click\\\\core.py\\", line 1078, in main\\r\\n    rv = self.invoke(ctx)\\r\\n         ^^^^^^^^^^^^^^^^\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\click\\\\core.py\\", line 1688, in invoke\\r\\n    return _process_result(sub_ctx.command.invoke(sub_ctx))\\r\\n                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\click\\\\core.py\\", line 1434, in invoke\\r\\n    return ctx.invoke(self.callback, **ctx.params)\\r\\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\click\\\\core.py\\", line 783, in invoke\\r\\n    return __callback(*args, **kwargs)\\r\\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\click\\\\decorators.py\\", line 33, in new_func\\r\\n    return f(get_current_context(), *args, **kwargs)\\r\\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\phy\\\\apps\\\\__init__.py\\", line 159, in cli_template_gui\\r\\n    template_gui(params_path, **kwargs)\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\phy\\\\apps\\\\template\\\\gui.py\\", line 209, in template_gui\\r\\n    model = load_model(params_path)\\r\\n            ^^^^^^^^^^^^^^^^^^^^^^^\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\phylib\\\\io\\\\model.py\\", line 1433, in load_model\\r\\n    return TemplateModel(**get_template_params(params_path))\\r\\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\phylib\\\\io\\\\model.py\\", line 339, in __init__\\r\\n    self._load_data()\\r\\n  File \\"C:\\\\Users\\\\sachur\\\\AppData\\\\Local\\\\anaconda3\\\\envs\\\\phy2\\\\Lib\\\\site-packages\\\\phylib\\\\io\\\\model.py\\", line 358, in _load_data\\r\\n    assert self.amplitudes.shape == (ns,)\\r\\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\\r\\nAssertionError\\r\\n\\r\\nQWidget: Must construct a QApplication before a QWidget\\r\\n```", "hints_text": "", "created_at": "2024-04-12 14:58:47", "FAIL_TO_PASS": ["src/spikeinterface/curation/tests/test_remove_excess_spikes.py::test_remove_excess_spikes"], "PASS_TO_PASS": [], "environment_setup_commit": "b3383575feade4bf70d623937d6ea2db9cd05b37", "version": "0.100", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .[full,widgets]", "pre_install": null, "pip_packages": ["pytest", "pytest-cov"], "reqs_path": null, "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 0, "files": 1}}, {"instance_id": "agronholm__sqlacodegen-375", "repo": "agronholm/sqlacodegen", "base_commit": "70e1a5c079a9d1568f57b0a242e155d43df9510c", "patch": "diff --git a/CHANGES.rst b/CHANGES.rst\\nindex 431359f..a1159ca 100644\\n--- a/CHANGES.rst\\n+++ b/CHANGES.rst\\n@@ -9,6 +9,8 @@ Version history\\n - Fixed two rendering issues in ``ENUM`` columns when a non-default schema is used: an\\n   unwarranted positional argument and missing the ``schema`` argument\\n - Fixed ``AttributeError`` when metadata contains user defined column types\\n+- Fixed ``AssertionError`` when metadata contains a column type that is a type decorator\\n+  with an all-uppercase name\\n \\n **3.0.0rc5**\\n \\ndiff --git a/src/sqlacodegen/generators.py b/src/sqlacodegen/generators.py\\nindex b3fc777..f992624 100644\\n--- a/src/sqlacodegen/generators.py\\n+++ b/src/sqlacodegen/generators.py\\n@@ -36,6 +36,7 @@ from sqlalchemy import (\\n     String,\\n     Table,\\n     Text,\\n+    TypeDecorator,\\n     UniqueConstraint,\\n )\\n from sqlalchemy.dialects.postgresql import JSONB\\n@@ -684,7 +685,7 @@ class TablesGenerator(CodeGenerator):\\n                 supercls, \\"__visit_name__\\"\\n             ):\\n                 # Don\'t try to adapt UserDefinedType as it\'s not a proper column type\\n-                if supercls is UserDefinedType:\\n+                if supercls is UserDefinedType or issubclass(supercls, TypeDecorator):\\n                     return coltype\\n \\n                 # Hack to fix adaptation of the Enum class which is broken since\\n", "test_patch": "diff --git a/tests/test_generator_tables.py b/tests/test_generator_tables.py\\nindex fc99b7a..0bbd322 100644\\n--- a/tests/test_generator_tables.py\\n+++ b/tests/test_generator_tables.py\\n@@ -4,6 +4,7 @@ from textwrap import dedent\\n \\n import pytest\\n from _pytest.fixtures import FixtureRequest\\n+from sqlalchemy import TypeDecorator\\n from sqlalchemy.dialects import mysql, postgresql\\n from sqlalchemy.engine import Engine\\n from sqlalchemy.schema import (\\n@@ -18,7 +19,7 @@ from sqlalchemy.schema import (\\n     UniqueConstraint,\\n )\\n from sqlalchemy.sql.expression import text\\n-from sqlalchemy.sql.sqltypes import NullType\\n+from sqlalchemy.sql.sqltypes import DateTime, NullType\\n from sqlalchemy.types import INTEGER, NUMERIC, SMALLINT, VARCHAR, Text\\n \\n from sqlacodegen.generators import CodeGenerator, TablesGenerator\\n@@ -26,6 +27,11 @@ from sqlacodegen.generators import CodeGenerator, TablesGenerator\\n from .conftest import validate_code\\n \\n \\n+# This needs to be uppercased to trigger #315\\n+class TIMESTAMP_DECORATOR(TypeDecorator[DateTime]):\\n+    impl = DateTime\\n+\\n+\\n @pytest.fixture\\n def generator(\\n     request: FixtureRequest, metadata: MetaData, engine: Engine\\n@@ -45,12 +51,15 @@ def test_fancy_coltypes(generator: CodeGenerator) -> None:\\n         Column(\\"bool\\", postgresql.BOOLEAN),\\n         Column(\\"vector\\", VECTOR(3)),\\n         Column(\\"number\\", NUMERIC(10, asdecimal=False)),\\n+        Column(\\"timestamp\\", TIMESTAMP_DECORATOR()),\\n         schema=\\"someschema\\",\\n     )\\n \\n     validate_code(\\n         generator.generate(),\\n         \\"\\"\\"\\\\\\n+        from tests.test_generator_tables import TIMESTAMP_DECORATOR\\n+\\n         from pgvector.sqlalchemy.vector import VECTOR\\n         from sqlalchemy import Boolean, Column, Enum, MetaData, Numeric, Table\\n \\n@@ -63,6 +72,7 @@ def test_fancy_coltypes(generator: CodeGenerator) -> None:\\n             Column(\'bool\', Boolean),\\n             Column(\'vector\', VECTOR(3)),\\n             Column(\'number\', Numeric(10, asdecimal=False)),\\n+            Column(\'timestamp\', TIMESTAMP_DECORATOR),\\n             schema=\'someschema\'\\n         )\\n         \\"\\"\\",\\n", "problem_statement": "AssertionError: TypeDecorator implementations require a class-level variable \'impl\'\\n### Things to check first\\n\\n- [X] I have searched the existing issues and didn\'t find my bug already reported there\\n\\n- [X] I have checked that my bug is still present in the latest release\\n\\n\\n### Sqlacodegen version\\n\\n3.0.0rc3\\n\\n### SQLAlchemy version\\n\\n2.0.25\\n\\n### RDBMS vendor\\n\\nOther\\n\\n### What happened?\\n\\nI\'m seeing the below error when trying to generate code for a `databricks` \\"database\\"\\r\\n```\\r\\n\\u276f sqlacodegen \\"${db_uri}\\"\\r\\nTraceback (most recent call last):\\r\\n  File \\"/opt/python/envs/dev310/bin/sqlacodegen\\", line 11, in <module>\\r\\n    sys.exit(main())\\r\\n  File \\"/opt/python/envs/dev310/lib/python3.10/site-packages/sqlacodegen/cli.py\\", line 92, in main\\r\\n    outfile.write(generator.generate())\\r\\n  File \\"/opt/python/envs/dev310/lib/python3.10/site-packages/sqlacodegen/generators.py\\", line 172, in generate\\r\\n    self.fix_column_types(table)\\r\\n  File \\"/opt/python/envs/dev310/lib/python3.10/site-packages/sqlacodegen/generators.py\\", line 649, in fix_column_types\\r\\n    column.type = self.get_adapted_type(column.type)\\r\\n  File \\"/opt/python/envs/dev310/lib/python3.10/site-packages/sqlacodegen/generators.py\\", line 681, in get_adapted_type\\r\\n    new_coltype = coltype.adapt(supercls)\\r\\n  File \\"/opt/python/envs/dev310/lib/python3.10/site-packages/sqlalchemy/sql/type_api.py\\", line 1033, in adapt\\r\\n    return util.constructor_copy(\\r\\n  File \\"/opt/python/envs/dev310/lib/python3.10/site-packages/sqlalchemy/util/langhelpers.py\\", line 1422, in constructor_copy\\r\\n    return cls(*args, **kw)\\r\\n  File \\"/opt/python/envs/dev310/lib/python3.10/site-packages/sqlalchemy/sql/type_api.py\\", line 1692, in __init__\\r\\n    raise AssertionError(\\r\\nAssertionError: TypeDecorator implementations require a class-level variable \'impl\' which refers to the class of type being decorated\\r\\n```\\r\\n\\r\\n\\r\\nJust posting to remind me to investigate further and in case others are also seeing this...\\n\\n### Database schema for reproducing the bug\\n\\n_No response_", "hints_text": "coveralls: \\n[![Coverage Status](https://coveralls.io/builds/72427695/badge)](https://coveralls.io/builds/72427695)\\n\\ncoverage: 97.057% (+0.007%) from 97.05%\\nwhen pulling **72751276c22bff12bcc185b8c5d1b7240cfea3cf on fix-typedecorator-adapt**\\ninto **70e1a5c079a9d1568f57b0a242e155d43df9510c on master**.\\n", "created_at": "2025-02-25 20:54:06", "FAIL_TO_PASS": ["tests/test_generator_tables.py::test_fancy_coltypes[postgresql]"], "PASS_TO_PASS": ["tests/test_generator_tables.py::test_boolean_detection", "tests/test_generator_tables.py::test_arrays[postgresql]", "tests/test_generator_tables.py::test_jsonb[postgresql]", "tests/test_generator_tables.py::test_jsonb_default[postgresql]", "tests/test_generator_tables.py::test_enum_detection", "tests/test_generator_tables.py::test_column_adaptation[postgresql]", "tests/test_generator_tables.py::test_mysql_column_types[mysql]", "tests/test_generator_tables.py::test_constraints", "tests/test_generator_tables.py::test_indexes", "tests/test_generator_tables.py::test_table_comment", "tests/test_generator_tables.py::test_table_name_identifiers", "tests/test_generator_tables.py::test_option_noindexes[generator0]", "tests/test_generator_tables.py::test_option_noconstraints[generator0]", "tests/test_generator_tables.py::test_option_nocomments[generator0]", "tests/test_generator_tables.py::test_computed_column[None-]", "tests/test_generator_tables.py::test_computed_column[False-,", "tests/test_generator_tables.py::test_computed_column[True-,", "tests/test_generator_tables.py::test_schema", "tests/test_generator_tables.py::test_foreign_key_options", "tests/test_generator_tables.py::test_pk_default", "tests/test_generator_tables.py::test_mysql_timestamp[mysql]", "tests/test_generator_tables.py::test_mysql_integer_display_width[mysql]", "tests/test_generator_tables.py::test_mysql_tinytext[mysql]", "tests/test_generator_tables.py::test_mysql_mediumtext[mysql]", "tests/test_generator_tables.py::test_mysql_longtext[mysql]", "tests/test_generator_tables.py::test_schema_boolean", "tests/test_generator_tables.py::test_server_default_multiline", "tests/test_generator_tables.py::test_server_default_colon", "tests/test_generator_tables.py::test_null_type", "tests/test_generator_tables.py::test_identity_column", "tests/test_generator_tables.py::test_multiline_column_comment", "tests/test_generator_tables.py::test_multiline_table_comment", "tests/test_generator_tables.py::test_postgresql_sequence_standard_name[postgresql]", "tests/test_generator_tables.py::test_postgresql_sequence_nonstandard_name[postgresql]", "tests/test_generator_tables.py::test_postgresql_sequence_with_schema[postgresql-myschema-test_seq]", "tests/test_generator_tables.py::test_postgresql_sequence_with_schema[postgresql-myschema-\\"test_seq\\"]", "tests/test_generator_tables.py::test_postgresql_sequence_with_schema[postgresql-\\"my.schema\\"-test_seq]", "tests/test_generator_tables.py::test_postgresql_sequence_with_schema[postgresql-\\"my.schema\\"-\\"test_seq\\"]"], "environment_setup_commit": "636680d7948ee40710f13e6a451ccd771f2c7f1f", "version": "3.0", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .[test]", "pre_install": null, "pip_packages": ["pytest"], "reqs_path": null, "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 2}}, {"instance_id": "sunpy__ndcube-448", "repo": "sunpy/ndcube", "base_commit": "196daeb407ff7cdb918313fa729d44510e6d9172", "patch": "diff --git a/changelog/448.feature.rst b/changelog/448.feature.rst\\nnew file mode 100644\\nindex 0000000..0de881d\\n--- /dev/null\\n+++ b/changelog/448.feature.rst\\n@@ -0,0 +1,1 @@\\n+Extends `~.NDCube.reproject_to` functionality by supporting ``adaptive`` and ``exact`` algorithms for an `~NDCube` with 2D celestial WCS.\\ndiff --git a/ndcube/ndcube.py b/ndcube/ndcube.py\\nindex 3dc1f97..189fb93 100644\\n--- a/ndcube/ndcube.py\\n+++ b/ndcube/ndcube.py\\n@@ -705,12 +705,33 @@ class NDCubeBase(NDCubeSlicingMixin, NDCubeABC):\\n         # Creating a new NDCubeSequence with the result_cubes and common axis as axis\\n         return NDCubeSequence(result_cubes, meta=self.meta)\\n \\n-    def reproject_to(self, target_wcs, shape_out=None, order=\'bilinear\', output_array=None, return_footprint=False):\\n+    def _validate_algorithm_and_order(self, algorithm, order):\\n+        order_compatibility = {\\n+            \'interpolation\': [\'nearest-neighbor\', \'bilinear\', \'biquadratic\', \'bicubic\'],\\n+            \'adaptive\': [\'nearest-neighbor\', \'bilinear\'],\\n+            \'exact\': []\\n+        }\\n+\\n+        if algorithm in order_compatibility:\\n+            if order_compatibility[algorithm] and order not in order_compatibility[algorithm]:\\n+                raise ValueError(f\\"For \'{algorithm}\' algorithm, the \'order\' argument must be \\"\\n+                                 f\\"one of {\', \'.join(order_compatibility[algorithm])}.\\")\\n+\\n+        else:\\n+            raise ValueError(f\\"The \'algorithm\' argument must be one of \\"\\n+                             f\\"{\', \'.join(order_compatibility.keys())}.\\")\\n+\\n+    def reproject_to(self, target_wcs, algorithm=\'interpolation\', shape_out=None, order=\'bilinear\',\\n+                     output_array=None, parallel=False, return_footprint=False):\\n         \\"\\"\\"\\n         Reprojects this NDCube to the coordinates described by another WCS object.\\n \\n         Parameters\\n         ----------\\n+        algorithm: `str`\\n+            The algorithm to use for reprojecting. This can be any of: \'interpolation\', \'adaptive\',\\n+            and \'exact\'.\\n+\\n         target_wcs : `astropy.wcs.wcsapi.BaseHighLevelWCS`, `astropy.wcs.wcsapi.BaseLowLevelWCS`,\\n             or `astropy.io.fits.Header`\\n             The WCS object to which the ``NDCube`` is to be reprojected.\\n@@ -722,13 +743,24 @@ class NDCubeBase(NDCubeSlicingMixin, NDCubeABC):\\n             (if available) from the low level API of the ``target_wcs`` is used.\\n \\n         order: `int` or `str`\\n-            The order of the interpolation. This can be any of: \'nearest-neighbor\', \'bilinear\',\\n-            \'biquadratic\', or \'bicubic\'.\\n+            The order of the interpolation (used only when the \'interpolation\' or \'adaptive\'\\n+            algorithm is selected).\\n+            For \'interpolation\' algorithm, this can be any of: \'nearest-neighbor\', \'bilinear\',\\n+            \'biquadratic\', and \'bicubic\'.\\n+            For \'adaptive\' algorithm, this can be either \'nearest-neighbor\' or \'bilinear\'.\\n \\n         output_array: `numpy.ndarray`, optional\\n             An array in which to store the reprojected data. This can be any numpy array\\n             including a memory map, which may be helpful when dealing with extremely large files.\\n \\n+        parallel: `bool` or `int`\\n+            Flag for parallel implementation (used only when the \'exact\' algorithm is selected).\\n+            If ``True``, a parallel implementation is chosen and the number of processes is\\n+            selected automatically as the number of logical CPUs detected on the machine.\\n+            If ``False``, a serial implementation is chosen.\\n+            If the flag is a positive integer n greater than one, a parallel implementation\\n+            using n processes is chosen.\\n+\\n         return_footprint: `bool`\\n             Whether to return the footprint in addition to the output NDCube.\\n \\n@@ -747,18 +779,28 @@ class NDCubeBase(NDCubeSlicingMixin, NDCubeABC):\\n         However, ``meta`` and ``global_coords`` are copied to the output ``NDCube``.\\n         \\"\\"\\"\\n         try:\\n-            from reproject import reproject_interp\\n+            from reproject import reproject_adaptive, reproject_exact, reproject_interp\\n+            from reproject.wcs_utils import has_celestial\\n         except ModuleNotFoundError:\\n             raise ImportError(\\"The NDCube.reproject_to method requires the optional package `reproject`.\\")\\n \\n         if isinstance(target_wcs, Mapping):\\n             target_wcs = WCS(header=target_wcs)\\n \\n+        low_level_target_wcs = utils.wcs.get_low_level_wcs(target_wcs, \'target_wcs\')\\n+\\n+        # \'adaptive\' and \'exact\' algorithms work only on 2D celestial WCS.\\n+        if algorithm == \'adaptive\' or algorithm == \'exact\':\\n+            if low_level_target_wcs.pixel_n_dim != 2 or low_level_target_wcs.world_n_dim != 2:\\n+                raise ValueError(\'For adaptive and exact algorithms, target_wcs must be 2D.\')\\n+\\n+            if not has_celestial(target_wcs):\\n+                raise ValueError(\'For adaptive and exact algorithms, \'\\n+                                 \'target_wcs must contain celestial axes only.\')\\n+\\n         if not utils.wcs.compare_wcs_physical_types(self.wcs, target_wcs):\\n             raise ValueError(\'Given target_wcs is not compatible with this NDCube, the physical types do not match.\')\\n \\n-        low_level_target_wcs = utils.wcs.get_low_level_wcs(target_wcs, \'target_wcs\')\\n-\\n         # If shape_out is not specified explicity, try to extract it from the low level WCS\\n         if not shape_out:\\n             if hasattr(low_level_target_wcs, \'array_shape\') and low_level_target_wcs.array_shape is not None:\\n@@ -766,10 +808,20 @@ class NDCubeBase(NDCubeSlicingMixin, NDCubeABC):\\n             else:\\n                 raise ValueError(\\"shape_out must be specified if target_wcs does not have the array_shape attribute.\\")\\n \\n-        data = reproject_interp(self, output_projection=target_wcs,\\n-                                shape_out=shape_out, order=order,\\n-                                output_array=output_array,\\n-                                return_footprint=return_footprint)\\n+        self._validate_algorithm_and_order(algorithm, order)\\n+\\n+        if algorithm == \'interpolation\':\\n+            data = reproject_interp(self, output_projection=target_wcs, shape_out=shape_out,\\n+                                    order=order, output_array=output_array,\\n+                                    return_footprint=return_footprint)\\n+\\n+        elif algorithm == \'adaptive\':\\n+            data = reproject_adaptive(self, output_projection=target_wcs, shape_out=shape_out,\\n+                                      order=order, return_footprint=return_footprint)\\n+\\n+        elif algorithm == \'exact\':\\n+            data = reproject_exact(self, output_projection=target_wcs, shape_out=shape_out,\\n+                                   parallel=parallel, return_footprint=return_footprint)\\n \\n         if return_footprint:\\n             data, footprint = data\\n", "test_patch": "diff --git a/ndcube/tests/test_ndcube.py b/ndcube/tests/test_ndcube.py\\nindex 708fbc8..1666b91 100644\\n--- a/ndcube/tests/test_ndcube.py\\n+++ b/ndcube/tests/test_ndcube.py\\n@@ -566,13 +566,13 @@ def test_initialize_from_ndcube(ndcube_3d_l_ln_lt_ectime):\\n     assert ec is not ec3\\n \\n \\n-def test_reproject(ndcube_4d_ln_l_t_lt, wcs_4d_lt_t_l_ln):\\n+def test_reproject_interpolation(ndcube_4d_ln_l_t_lt, wcs_4d_lt_t_l_ln):\\n     target_wcs_header = wcs_4d_lt_t_l_ln.low_level_wcs.to_header()\\n     target_wcs_header[\'CDELT3\'] = 0.1   # original value = 0.2\\n     target_wcs = astropy.wcs.WCS(header=target_wcs_header)\\n     shape_out = (5, 20, 12, 8)\\n \\n-    resampled_cube = ndcube_4d_ln_l_t_lt.reproject_to(target_wcs, shape_out)\\n+    resampled_cube = ndcube_4d_ln_l_t_lt.reproject_to(target_wcs, shape_out=shape_out)\\n \\n     assert ndcube_4d_ln_l_t_lt.data.shape == (5, 10, 12, 8)\\n     assert resampled_cube.data.shape == (5, 20, 12, 8)\\n@@ -582,14 +582,14 @@ def test_reproject_invalid_wcs(ndcube_4d_ln_l_t_lt, wcs_3d_lt_ln_l):\\n     shape_out = (5, 20, 12, 8)\\n \\n     with pytest.raises(Exception):\\n-        _ = ndcube_4d_ln_l_t_lt.reproject_to(wcs_3d_lt_ln_l, shape_out)\\n+        _ = ndcube_4d_ln_l_t_lt.reproject_to(wcs_3d_lt_ln_l, shape_out=shape_out)\\n \\n \\n def test_reproject_with_header(ndcube_4d_ln_l_t_lt, wcs_4d_lt_t_l_ln):\\n     target_wcs_header = wcs_4d_lt_t_l_ln.low_level_wcs.to_header()\\n     shape_out = (5, 20, 12, 8)\\n \\n-    _ = ndcube_4d_ln_l_t_lt.reproject_to(target_wcs_header, shape_out)\\n+    _ = ndcube_4d_ln_l_t_lt.reproject_to(target_wcs_header, shape_out=shape_out)\\n \\n \\n def test_reproject_return_footprint(ndcube_4d_ln_l_t_lt, wcs_4d_lt_t_l_ln):\\n@@ -598,7 +598,7 @@ def test_reproject_return_footprint(ndcube_4d_ln_l_t_lt, wcs_4d_lt_t_l_ln):\\n     target_wcs = astropy.wcs.WCS(header=target_wcs_header)\\n     shape_out = (5, 20, 12, 8)\\n \\n-    resampled_cube, footprint = ndcube_4d_ln_l_t_lt.reproject_to(target_wcs, shape_out,\\n+    resampled_cube, footprint = ndcube_4d_ln_l_t_lt.reproject_to(target_wcs, shape_out=shape_out,\\n                                                                  return_footprint=True)\\n \\n     assert ndcube_4d_ln_l_t_lt.data.shape == (5, 10, 12, 8)\\n@@ -630,3 +630,72 @@ def test_wcs_type_after_init(ndcube_3d_ln_lt_l, wcs_3d_l_lt_ln):\\n     cube = NDCube(ndcube_3d_ln_lt_l.data[slices], low_level_wcs)\\n     # Check the WCS has been converted to high level but NDCube init.\\n     assert isinstance(cube.wcs, BaseHighLevelWCS)\\n+\\n+\\n+def test_reproject_adaptive(ndcube_2d_ln_lt, wcs_2d_lt_ln):\\n+    shape_out = (10, 12)\\n+    resampled_cube = ndcube_2d_ln_lt.reproject_to(wcs_2d_lt_ln, algorithm=\'adaptive\',\\n+                                                  shape_out=shape_out)\\n+\\n+    assert ndcube_2d_ln_lt.data.shape == (10, 12)\\n+    assert resampled_cube.data.shape == (10, 12)\\n+\\n+\\n+def test_reproject_exact(ndcube_2d_ln_lt, wcs_2d_lt_ln):\\n+    shape_out = (10, 12)\\n+    resampled_cube = ndcube_2d_ln_lt.reproject_to(wcs_2d_lt_ln, algorithm=\'exact\',\\n+                                                  shape_out=shape_out)\\n+\\n+    assert ndcube_2d_ln_lt.data.shape == (10, 12)\\n+    assert resampled_cube.data.shape == (10, 12)\\n+\\n+\\n+def test_reproject_invalid_algorithm(ndcube_4d_ln_l_t_lt, wcs_4d_lt_t_l_ln):\\n+    with pytest.raises(ValueError):\\n+        _ = ndcube_4d_ln_l_t_lt.reproject_to(wcs_4d_lt_t_l_ln, algorithm=\'my_algorithm\',\\n+                                             shape_out=(5, 10, 12, 8))\\n+\\n+\\n+def test_reproject_invalid_order(ndcube_2d_ln_lt, wcs_2d_lt_ln):\\n+    shape_out = (10, 12)\\n+\\n+    # Supported for interpolation\\n+    for order in [\'nearest-neighbor\', \'bilinear\', \'biquadratic\', \'bicubic\']:\\n+        _ = ndcube_2d_ln_lt.reproject_to(wcs_2d_lt_ln, algorithm=\'interpolation\',\\n+                                         shape_out=shape_out, order=order)\\n+\\n+    # Supported for adaptive\\n+    for order in [\'nearest-neighbor\', \'bilinear\']:\\n+        _ = ndcube_2d_ln_lt.reproject_to(wcs_2d_lt_ln, algorithm=\'adaptive\',\\n+                                         shape_out=shape_out, order=order)\\n+\\n+    # Not supported for adaptive\\n+    for order in [\'biquadratic\', \'bicubic\', \'my_order\']:\\n+        with pytest.raises(ValueError):\\n+            _ = ndcube_2d_ln_lt.reproject_to(wcs_2d_lt_ln, algorithm=\'adaptive\',\\n+                                             shape_out=shape_out, order=order)\\n+\\n+    # \'exact\' does not need \'order\'\\n+    _ = ndcube_2d_ln_lt.reproject_to(wcs_2d_lt_ln, algorithm=\'exact\', shape_out=shape_out)\\n+\\n+\\n+def test_reproject_adaptive_incompatible_wcs(ndcube_4d_ln_l_t_lt, wcs_4d_lt_t_l_ln,\\n+                                             wcs_1d_l, ndcube_1d_l):\\n+    with pytest.raises(ValueError):\\n+        _ = ndcube_1d_l.reproject_to(wcs_1d_l, algorithm=\'adaptive\',\\n+                                     shape_out=(10,))\\n+\\n+    with pytest.raises(ValueError):\\n+        _ = ndcube_4d_ln_l_t_lt.reproject_to(wcs_4d_lt_t_l_ln, algorithm=\'adaptive\',\\n+                                             shape_out=(5, 10, 12, 8))\\n+\\n+\\n+def test_reproject_exact_incompatible_wcs(ndcube_4d_ln_l_t_lt, wcs_4d_lt_t_l_ln,\\n+                                          wcs_1d_l, ndcube_1d_l):\\n+    with pytest.raises(ValueError):\\n+        _ = ndcube_1d_l.reproject_to(wcs_1d_l, algorithm=\'exact\',\\n+                                     shape_out=(10,))\\n+\\n+    with pytest.raises(ValueError):\\n+        _ = ndcube_4d_ln_l_t_lt.reproject_to(wcs_4d_lt_t_l_ln, algorithm=\'exact\',\\n+                                             shape_out=(5, 10, 12, 8))\\n", "problem_statement": "Extend reproject method to support other algorithms\\n## Outline\\r\\n\\r\\nThe `adaptive` and `exact` reproject algorithms are applicable to image axes only.  This is why PR #434 does not support them.  This issue summarizes the way forward for extending `NDCube.reproject_to` to those other algorithms.\\r\\n* Initially, we will only support these algorithms in the case of a 2-D celestial `NDCube`.  (Supporting more general cases are discussed further below.)\\r\\n* If user requests `adaptive` or `exact` to be used then check the original WCS.\\r\\n  * If not 2-D, raise error\\r\\n  * If 2-D but not celestial, raise error.\\r\\n  * Otherwise proceed.\\r\\n\\r\\nThe question now is how to check for celestial axes.\\r\\n\\r\\n## Checking For Celestial Axes\\r\\nIn order to fast-track this feature, we will use a shortcut for identifying celestial axes which we hope will work for most cases. (See below for the future more generalized approach.)\\r\\n\\r\\nCheck the physical types of the axes against a finite list options.  These will include that the physical type will include:\\r\\n* `pos.` and `.lat` or `.lon`\\r\\n* `.ra`\\r\\n* `.dec`\\r\\n* Add other cases here.\\r\\n\\r\\nBecause we hope to deal with this more generally in the future, we should check against the UCD1+ vocab list for other valid possibilities.  But beyond that we will encourage users to inform us of other cases which we can then add to our list.  Before this list gets too long we hope to have a more generalized implementation which will remove the need for this checklist.\\r\\n\\r\\n## Future Generalized Implementation\\r\\nIn general, these algorithms should be valid for any >=2-D cube which contains two dependent spatial axes.  The reprojection could then be done in these two dimensions while keeping the rest unchanged.  To do this we need a way of identifying invariant axes.  This is discussed in #432.  Once that is implemented, we can leverage it remove the above limitations.\\r\\n\\r\\n## Justification for Immediate Implementation\\r\\nSupporting other algorithms for 2-D celestial cubes is needed by SunPy Map.  By using the above quick-and-dirty implementation, reproject can be released as part of 2.0 and enable Map to use reproject\'s full power.  The more generalized subsequent implementation will have no impact on Map\'s use case.\\r\\n\\r\\n", "hints_text": "pep8speaks: Hello @adwaitbhope! Thanks for opening this PR. \\n\\n* In the file [`ndcube/ndcube.py`](https://github.com/sunpy/ndcube/blob/bcc423a94958406abe3f32d78e2a78742f810e0a/ndcube/ndcube.py):\\n\\n> [Line 705:30](https://github.com/sunpy/ndcube/blob/bcc423a94958406abe3f32d78e2a78742f810e0a/ndcube/ndcube.py#L705): [E222](https://duckduckgo.com/?q=pep8%20E222) multiple spaces after operator\\n> [Line 657:38](https://github.com/sunpy/ndcube/blob/bcc423a94958406abe3f32d78e2a78742f810e0a/ndcube/ndcube.py#L657): [E741](https://duckduckgo.com/?q=pep8%20E741) ambiguous variable name \'l\'\\n\\n\\n\\n\\n\\n\\n\\nDo see the [Hitchhiker\'s guide to code style](https://goo.gl/hqbW4r)", "created_at": "2021-07-29 07:23:15", "FAIL_TO_PASS": ["ndcube/tests/test_ndcube.py::test_reproject_exact", "ndcube/tests/test_ndcube.py::test_reproject_invalid_algorithm", "ndcube/tests/test_ndcube.py::test_reproject_adaptive_incompatible_wcs", "ndcube/tests/test_ndcube.py::test_reproject_exact_incompatible_wcs"], "PASS_TO_PASS": ["ndcube/tests/test_ndcube.py::test_wcs_object[ndcube_4d_ln_lt_l_t]", "ndcube/tests/test_ndcube.py::test_wcs_object[ndcube_4d_uncertainty]", "ndcube/tests/test_ndcube.py::test_wcs_object[ndcube_4d_mask]", "ndcube/tests/test_ndcube.py::test_wcs_object[ndcube_4d_extra_coords]", "ndcube/tests/test_ndcube.py::test_wcs_object[ndcube_4d_unit_uncertainty]", "ndcube/tests/test_ndcube.py::test_wcs_object[ndcube_3d_ln_lt_l]", "ndcube/tests/test_ndcube.py::test_wcs_object[ndcube_3d_rotated]", "ndcube/tests/test_ndcube.py::test_wcs_object[ndcube_2d_ln_lt]", "ndcube/tests/test_ndcube.py::test_wcs_object[ndcube_1d_l]", "ndcube/tests/test_ndcube.py::test_slicing_ln_lt[ndcube_3d_ln_lt_l-item0]", "ndcube/tests/test_ndcube.py::test_slicing_ln_lt[ndcube_3d_ln_lt_l-item1]", "ndcube/tests/test_ndcube.py::test_slicing_ln_lt[ndcube_3d_ln_lt_l-item2]", "ndcube/tests/test_ndcube.py::test_slicing_ln_lt[ndcube_3d_ln_lt_l-item3]", "ndcube/tests/test_ndcube.py::test_slicing_ln_lt[ndcube_3d_ln_lt_l-item4]", "ndcube/tests/test_ndcube.py::test_slicing_ln_lt[ndcube_3d_ln_lt_l-item5]", "ndcube/tests/test_ndcube.py::test_slicing_ln_lt[ndcube_4d_ln_lt_l_t-item6]", "ndcube/tests/test_ndcube.py::test_slicing_ln_lt[ndcube_4d_ln_lt_l_t-item7]", "ndcube/tests/test_ndcube.py::test_slicing_ln_lt[ndcube_4d_ln_lt_l_t-item8]", "ndcube/tests/test_ndcube.py::test_slicing_wave[ndcube_3d_ln_lt_l-item0]", "ndcube/tests/test_ndcube.py::test_slicing_wave[ndcube_3d_ln_lt_l-item1]", "ndcube/tests/test_ndcube.py::test_slicing_wave[ndcube_3d_ln_lt_l-item2]", "ndcube/tests/test_ndcube.py::test_slicing_wave[ndcube_3d_ln_lt_l-item3]", "ndcube/tests/test_ndcube.py::test_slicing_wave[ndcube_3d_ln_lt_l-item4]", "ndcube/tests/test_ndcube.py::test_slicing_wave[ndcube_3d_ln_lt_l-item5]", "ndcube/tests/test_ndcube.py::test_slicing_wave[ndcube_4d_ln_lt_l_t-item6]", "ndcube/tests/test_ndcube.py::test_slicing_wave[ndcube_4d_ln_lt_l_t-item7]", "ndcube/tests/test_ndcube.py::test_slicing_wave[ndcube_4d_ln_lt_l_t-item8]", "ndcube/tests/test_ndcube.py::test_slicing_split_celestial[ndcube_3d_ln_lt_l-item0]", "ndcube/tests/test_ndcube.py::test_slicing_split_celestial[ndcube_3d_ln_lt_l-item1]", "ndcube/tests/test_ndcube.py::test_slicing_split_celestial[ndcube_3d_ln_lt_l-item2]", "ndcube/tests/test_ndcube.py::test_slicing_split_celestial[ndcube_3d_ln_lt_l-item3]", "ndcube/tests/test_ndcube.py::test_slicing_split_celestial[ndcube_3d_ln_lt_l-item4]", "ndcube/tests/test_ndcube.py::test_slicing_split_celestial[ndcube_3d_ln_lt_l-item5]", "ndcube/tests/test_ndcube.py::test_slicing_split_celestial[ndcube_4d_ln_lt_l_t-item6]", "ndcube/tests/test_ndcube.py::test_slicing_split_celestial[ndcube_4d_ln_lt_l_t-item7]", "ndcube/tests/test_ndcube.py::test_slicing_split_celestial[ndcube_4d_ln_lt_l_t-item8]", "ndcube/tests/test_ndcube.py::test_slicing_preserves_global_coords", "ndcube/tests/test_ndcube.py::test_slicing_removed_world_coords", "ndcube/tests/test_ndcube.py::test_axis_world_coords_wave_ec", "ndcube/tests/test_ndcube.py::test_axis_world_coords_sliced_all_3d[ndcube_3d_ln_lt_l-item0]", "ndcube/tests/test_ndcube.py::test_axis_world_coords_sliced_all_3d[ndcube_3d_ln_lt_l-item1]", "ndcube/tests/test_ndcube.py::test_axis_world_coords_sliced_all_4d[ndcube_4d_ln_lt_l_t-item0]", "ndcube/tests/test_ndcube.py::test_axis_world_coords_sliced_all_4d[ndcube_4d_ln_lt_l_t-item1]", "ndcube/tests/test_ndcube.py::test_axis_world_coords_all_4d_split", "ndcube/tests/test_ndcube.py::test_axis_world_coords_all_4d_split_sub[wapt0]", "ndcube/tests/test_ndcube.py::test_axis_world_coords_all_4d_split_sub[wapt1]", "ndcube/tests/test_ndcube.py::test_axis_world_coords_all_4d_split_sub[wapt2]", "ndcube/tests/test_ndcube.py::test_axis_world_coords_all_4d_split_sub[wapt3]", "ndcube/tests/test_ndcube.py::test_axis_world_coords_all", "ndcube/tests/test_ndcube.py::test_axis_world_coords_wave", "ndcube/tests/test_ndcube.py::test_axis_world_coords_sky[custom:pos.helioprojective.lon]", "ndcube/tests/test_ndcube.py::test_axis_world_coords_sky[custom:pos.helioprojective.lat]", "ndcube/tests/test_ndcube.py::test_axes_world_coords_sky_only", "ndcube/tests/test_ndcube.py::test_axis_world_coords_values_all", "ndcube/tests/test_ndcube.py::test_array_axis_physical_types", "ndcube/tests/test_ndcube.py::test_crop", "ndcube/tests/test_ndcube.py::test_crop_tuple_non_tuple_input", "ndcube/tests/test_ndcube.py::test_crop_with_nones", "ndcube/tests/test_ndcube.py::test_crop_1d_independent", "ndcube/tests/test_ndcube.py::test_crop_1d_dependent", "ndcube/tests/test_ndcube.py::test_crop_by_values", "ndcube/tests/test_ndcube.py::test_crop_by_coords_with_units", "ndcube/tests/test_ndcube.py::test_crop_by_values_with_nones", "ndcube/tests/test_ndcube.py::test_crop_by_values_all_nones", "ndcube/tests/test_ndcube.py::test_crop_by_values_valueerror1", "ndcube/tests/test_ndcube.py::test_crop_by_values_valueerror2", "ndcube/tests/test_ndcube.py::test_crop_by_values_1d_dependent", "ndcube/tests/test_ndcube.py::test_crop_rotated_celestial", "ndcube/tests/test_ndcube.py::test_initialize_from_ndcube", "ndcube/tests/test_ndcube.py::test_reproject_interpolation", "ndcube/tests/test_ndcube.py::test_reproject_invalid_wcs", "ndcube/tests/test_ndcube.py::test_reproject_with_header", "ndcube/tests/test_ndcube.py::test_reproject_return_footprint", "ndcube/tests/test_ndcube.py::test_reproject_shape_out", "ndcube/tests/test_ndcube.py::test_wcs_type_after_init"], "environment_setup_commit": "dbffe4fad1e1aa04d50d9d53d9b2a12ee4d48d7e", "version": "2.0", "license_name": "BSD 2-Clause \\"Simplified\\" License", "install_config": {"python": "3.9", "install": "pip install -e .[dev]", "pre_install": null, "pip_packages": ["pytest", "pytest-rerunfailures", "pytest-timeout"], "reqs_path": null, "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 1}}, {"instance_id": "tox-dev__tox-2000", "repo": "tox-dev/tox", "base_commit": "2b7f522d356cafb24e29d255b0d44bc5c561df29", "patch": "diff --git a/docs/changelog/1985.bugfix.rst b/docs/changelog/1985.bugfix.rst\\nnew file mode 100644\\nindex 00000000..62d49a53\\n--- /dev/null\\n+++ b/docs/changelog/1985.bugfix.rst\\n@@ -0,0 +1,2 @@\\n+Enable replacements (a.k.a section substitions) for section names containing a dash in sections\\n+without the ``testenv:`` prefix - by :user:`jugmac00`, :user:`obestwalter`, :user:`eumiro`.\\ndiff --git a/src/tox/config/loader/ini/replace.py b/src/tox/config/loader/ini/replace.py\\nindex 6b16ef71..898d7dbe 100644\\n--- a/src/tox/config/loader/ini/replace.py\\n+++ b/src/tox/config/loader/ini/replace.py\\n@@ -89,7 +89,7 @@ def _replace_match(\\n \\n _REPLACE_REF = re.compile(\\n     rf\\"\\"\\"\\n-    (\\\\[(?P<full_env>{BASE_TEST_ENV}(:(?P<env>[^]]+))?|(?P<section>\\\\w+))\\\\])? # env/section\\n+    (\\\\[(?P<full_env>{BASE_TEST_ENV}(:(?P<env>[^]]+))?|(?P<section>[-\\\\w]+))\\\\])? # env/section\\n     (?P<key>[a-zA-Z0-9_]+) # key\\n     (:(?P<default>.*))? # default value\\n \\"\\"\\",\\n", "test_patch": "diff --git a/tests/config/loader/ini/replace/test_replace_tox_env.py b/tests/config/loader/ini/replace/test_replace_tox_env.py\\nindex a111495a..1116214b 100644\\n--- a/tests/config/loader/ini/replace/test_replace_tox_env.py\\n+++ b/tests/config/loader/ini/replace/test_replace_tox_env.py\\n@@ -151,3 +151,18 @@ def test_replace_from_tox_other_tox_section_same_name(tox_ini_conf: ToxIniCreato\\n     conf_a = tox_ini_conf(\\"[testenv:a]\\\\nx={[testenv:b]c}\\\\nc=d\\\\n[testenv:b]}\\").get_env(\\"a\\")\\n     conf_a.add_config(keys=\\"x\\", of_type=str, default=\\"\\", desc=\\"d\\")\\n     assert conf_a[\\"x\\"] == \\"{[testenv:b]c}\\"\\n+\\n+\\n+@pytest.mark.parametrize(\\n+    (\\"env_name\\", \\"exp\\"),\\n+    [\\n+        (\\"testenv:foobar\\", \\"1\\"),\\n+        (\\"testenv:foo-bar\\", \\"1\\"),\\n+        (\\"foo-bar\\", \\"1\\"),\\n+        (\\"foobar\\", \\"1\\"),\\n+    ],\\n+)\\n+def test_replace_valid_section_names(tox_ini_conf: ToxIniCreator, env_name: str, exp: str) -> None:\\n+    conf_a = tox_ini_conf(f\\"[{env_name}]\\\\na={exp}\\\\n[testenv:a]\\\\nx = {{[{env_name}]a}}\\").get_env(\\"a\\")\\n+    conf_a.add_config(keys=\\"x\\", of_type=str, default=\\"o\\", desc=\\"o\\")\\n+    assert conf_a[\\"x\\"] == exp\\n", "problem_statement": "tox4: substituting values from other sections seems to not work\\nHi, I gave a try new tox alpha release (tox-4.0.0a6) and I found one issue with tox not respecting values substitution from other sections. I do not see any references to this section in the documentation anymore so maybe it is something you want to get rid of... \\r\\n\\r\\nAnyway - this is how it looks like:\\r\\n```\\r\\n$ tox -rvv\\r\\nROOT: 217 D setup logging to DEBUG on pid 86537 [tox/report.py:211]\\r\\nTraceback (most recent call last):\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/config/main.py\\", line 51, in get_env\\r\\n    return self._env_to_set[item]\\r\\nKeyError: \'py\'\\r\\n\\r\\nDuring handling of the above exception, another exception occurred:\\r\\n\\r\\nTraceback (most recent call last):\\r\\n  File \\"/Users/mrusi/venv/bin/tox4\\", line 8, in <module>\\r\\n    sys.exit(run())\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/run.py\\", line 22, in run\\r\\n    result = main(sys.argv[1:] if args is None else args)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/run.py\\", line 47, in main\\r\\n    result = handler(state)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/session/cmd/legacy.py\\", line 104, in legacy\\r\\n    return run_sequential(state)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/session/cmd/run/sequential.py\\", line 20, in run_sequential\\r\\n    return execute(state, max_workers=1, has_spinner=False, live=True)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/session/cmd/run/common.py\\", line 172, in execute\\r\\n    if state.tox_env(env).active:\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/session/state.py\\", line 64, in tox_env\\r\\n    self.conf.get_env(name)  # the lookup here will trigger register_config_set, which will build it\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/config/main.py\\", line 61, in get_env\\r\\n    self.register_config_set(item, env)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/session/state.py\\", line 76, in register_config_set\\r\\n    self._build_run_env(config_set)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/session/state.py\\", line 91, in _build_run_env\\r\\n    env: RunToxEnv = builder(env_conf, self.conf.core, self.options, journal, self.log_handler)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/python/virtual_env/runner.py\\", line 26, in __init__\\r\\n    super().__init__(conf, core, options, journal, log_handler)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/python/virtual_env/api.py\\", line 35, in __init__\\r\\n    super().__init__(conf, core, options, journal, log_handler)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/python/runner.py\\", line 25, in __init__\\r\\n    super().__init__(conf, core, options, journal, log_handler)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/python/api.py\\", line 79, in __init__\\r\\n    super().__init__(conf, core, options, journal, log_handler)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/runner.py\\", line 27, in __init__\\r\\n    super().__init__(conf, core, options, journal, log_handler)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/api.py\\", line 40, in __init__\\r\\n    self.register_config()\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/python/virtual_env/api.py\\", line 38, in register_config\\r\\n    super().register_config()\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/python/runner.py\\", line 29, in register_config\\r\\n    super().register_config()\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/python/api.py\\", line 82, in register_config\\r\\n    super().register_config()\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/runner.py\\", line 77, in register_config\\r\\n    self.has_package = self.add_package_conf()\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/tox_env/python/virtual_env/runner.py\\", line 38, in add_package_conf\\r\\n    develop_mode = self.conf[\\"usedevelop\\"] or getattr(self.options, \\"develop\\", False)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/config/sets.py\\", line 86, in __getitem__\\r\\n    return self.load(item)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/config/sets.py\\", line 97, in load\\r\\n    return config_definition(self._conf, item, self.loaders, chain)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/config/of_type.py\\", line 99, in __call__\\r\\n    value = loader.load(key, self.of_type, self.kwargs, conf, self.env_name, chain)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/config/loader/api.py\\", line 98, in load\\r\\n    converted = self.to(prepared, of_type, kwargs)\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/config/loader/convert.py\\", line 31, in to\\r\\n    return self.to_bool(raw)  # type: ignore[return-value]\\r\\n  File \\"/Users/mrusi/venv/lib/python3.8/site-packages/tox/config/loader/str_convert.py\\", line 90, in to_bool\\r\\n    raise TypeError(f\\"value {value} cannot be transformed to bool, valid: {\', \'.join(StrConvert.VALID_BOOL)}\\")\\r\\nTypeError: value {[project-info]develop} cannot be transformed to bool, valid: , 0, 1, false, no, off, on, true, yes\\r\\n\\r\\n$ cat tox.ini\\r\\n[project-info]\\r\\ndevelop = false\\r\\n[testenv]\\r\\ndeps=-rrequirements.txt\\r\\nusedevelop = {[project-info]develop}\\r\\ncommands=echo \\"commands\\"\\r\\n```", "hints_text": "codecov[bot]: # [Codecov](https://codecov.io/gh/tox-dev/tox/pull/2000?src=pr&el=h1) Report\\n> Merging [#2000](https://codecov.io/gh/tox-dev/tox/pull/2000?src=pr&el=desc) (3f2eec8) into [rewrite](https://codecov.io/gh/tox-dev/tox/commit/00723c507f37e99bd78edc82aa389c5af396b762?el=desc) (00723c5) will **increase** coverage by `0.16%`.\\n> The diff coverage is `100.00%`.\\n\\n[![Impacted file tree graph](https://codecov.io/gh/tox-dev/tox/pull/2000/graphs/tree.svg?width=650&height=150&src=pr&token=DYodAwDCZ5)](https://codecov.io/gh/tox-dev/tox/pull/2000?src=pr&el=tree)\\n\\n```diff\\n@@             Coverage Diff              @@\\n##           rewrite     #2000      +/-   ##\\n============================================\\n+ Coverage    99.83%   100.00%   +0.16%     \\n============================================\\n  Files          147       146       -1     \\n  Lines         8276      8181      -95     \\n  Branches       823       798      -25     \\n============================================\\n- Hits          8262      8181      -81     \\n+ Misses           3         0       -3     \\n+ Partials        11         0      -11     \\n```\\n\\n| Flag | Coverage \\u0394 | |\\n|---|---|---|\\n| tests | `100.00% <100.00%> (+0.16%)` | :arrow_up: |\\n\\nFlags with carried forward coverage won\'t be shown. [Click here](https://docs.codecov.io/docs/carryforward-flags#carryforward-flags-in-the-pull-request-comment) to find out more.\\n\\n| [Impacted Files](https://codecov.io/gh/tox-dev/tox/pull/2000?src=pr&el=tree) | Coverage \\u0394 | |\\n|---|---|---|\\n| [src/tox/config/loader/ini/replace.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-c3JjL3RveC9jb25maWcvbG9hZGVyL2luaS9yZXBsYWNlLnB5) | `100.00% <\\u00f8> (\\u00f8)` | |\\n| [.../config/loader/ini/replace/test\\\\_replace\\\\_tox\\\\_env.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-dGVzdHMvY29uZmlnL2xvYWRlci9pbmkvcmVwbGFjZS90ZXN0X3JlcGxhY2VfdG94X2Vudi5weQ==) | `100.00% <100.00%> (\\u00f8)` | |\\n| [src/tox/tox\\\\_env/api.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-c3JjL3RveC90b3hfZW52L2FwaS5weQ==) | `100.00% <0.00%> (\\u00f8)` | |\\n| [src/tox/execute/request.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-c3JjL3RveC9leGVjdXRlL3JlcXVlc3QucHk=) | `100.00% <0.00%> (\\u00f8)` | |\\n| [src/tox/util/pep517/backend.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-c3JjL3RveC91dGlsL3BlcDUxNy9iYWNrZW5kLnB5) | `100.00% <0.00%> (\\u00f8)` | |\\n| [...ecute/local\\\\_sub\\\\_process/read\\\\_via\\\\_thread\\\\_windows.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-c3JjL3RveC9leGVjdXRlL2xvY2FsX3N1Yl9wcm9jZXNzL3JlYWRfdmlhX3RocmVhZF93aW5kb3dzLnB5) | | |\\n| [src/tox/tox\\\\_env/python/api.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-c3JjL3RveC90b3hfZW52L3B5dGhvbi9hcGkucHk=) | `100.00% <0.00%> (+0.77%)` | :arrow_up: |\\n| [src/tox/config/loader/str\\\\_convert.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-c3JjL3RveC9jb25maWcvbG9hZGVyL3N0cl9jb252ZXJ0LnB5) | `100.00% <0.00%> (+1.47%)` | :arrow_up: |\\n| [src/tox/util/spinner.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-c3JjL3RveC91dGlsL3NwaW5uZXIucHk=) | `100.00% <0.00%> (+2.27%)` | :arrow_up: |\\n| [src/tox/execute/local\\\\_sub\\\\_process/\\\\_\\\\_init\\\\_\\\\_.py](https://codecov.io/gh/tox-dev/tox/pull/2000/diff?src=pr&el=tree#diff-c3JjL3RveC9leGVjdXRlL2xvY2FsX3N1Yl9wcm9jZXNzL19faW5pdF9fLnB5) | `100.00% <0.00%> (+4.32%)` | :arrow_up: |\\n\\n------\\n\\n[Continue to review full report at Codecov](https://codecov.io/gh/tox-dev/tox/pull/2000?src=pr&el=continue).\\n> **Legend** - [Click here to learn more](https://docs.codecov.io/docs/codecov-delta)\\n> `\\u0394 = absolute <relative> (impact)`, `\\u00f8 = not affected`, `? = missing data`\\n> Powered by [Codecov](https://codecov.io/gh/tox-dev/tox/pull/2000?src=pr&el=footer). Last update [00723c5...3f2eec8](https://codecov.io/gh/tox-dev/tox/pull/2000?src=pr&el=lastupdated). Read the [comment docs](https://docs.codecov.io/docs/pull-request-comments).\\n", "created_at": "2021-04-06 16:08:32", "FAIL_TO_PASS": ["tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_valid_section_names[foo-bar-1]"], "PASS_TO_PASS": ["tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_within_tox_env", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_within_tox_env_missing_raises", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_within_tox_env_missing_default", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_within_tox_env_missing_default_env_only", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_within_tox_env_missing_no_default", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_within_tox_env_from_base", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_ref_bad_type", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[0-0]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[0}-0}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[{0-{0]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[{0}-{0}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[{}{0}-{}{0}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[{0}{}-{0}{}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[\\\\\\\\{0}-{0}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[{0\\\\\\\\}-{0}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[\\\\\\\\{0\\\\\\\\}-{0}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[f\\\\\\\\{0\\\\\\\\}-f{0}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[\\\\\\\\{0\\\\\\\\}f-{0}f]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[\\\\\\\\{\\\\\\\\{0-{{0]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[0\\\\\\\\}\\\\\\\\}-0}}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_do_not_replace[\\\\\\\\{\\\\\\\\{0\\\\\\\\}\\\\\\\\}-{{0}}]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_from_tox_section_non_registered", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_from_tox_section_missing_section", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_circular", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_from_tox_section_missing_value", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_from_section_bad_type", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_from_tox_section_registered", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_from_tox_other_tox_section_same_name", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_valid_section_names[testenv:foobar-1]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_valid_section_names[testenv:foo-bar-1]", "tests/config/loader/ini/replace/test_replace_tox_env.py::test_replace_valid_section_names[foobar-1]"], "environment_setup_commit": "72245f2a84beafa04f82c031137b484b2575a940", "version": "3.23", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .[testing]", "pre_install": null, "pip_packages": ["pytest", "pytest-cov", "pytest-xdist", "pytest-mock", "pytest-asyncio"], "reqs_path": null, "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 1}}, {"instance_id": "nilearn__nilearn-4414", "repo": "nilearn/nilearn", "base_commit": "2c0452e5267cedabf035569c27753473372e16ea", "patch": "diff --git a/nilearn/glm/first_level/first_level.py b/nilearn/glm/first_level/first_level.py\\nindex b70d5f8e0..1b27ce2b3 100644\\n--- a/nilearn/glm/first_level/first_level.py\\n+++ b/nilearn/glm/first_level/first_level.py\\n@@ -1292,7 +1292,15 @@ def first_level_from_bids(\\n \\n     dataset_path = Path(dataset_path).absolute()\\n \\n-    kwargs_load_confounds = _check_kwargs_load_confounds(**kwargs)\\n+    kwargs_load_confounds, remaining_kwargs = _check_kwargs_load_confounds(\\n+        **kwargs\\n+    )\\n+\\n+    if len(remaining_kwargs) > 0:\\n+        raise RuntimeError(\\n+            \\"Unknown keyword arguments. Keyword arguments should start with \\"\\n+            f\\"`confounds_` prefix: {remaining_kwargs}\\"\\n+        )\\n \\n     if drift_model is not None and kwargs_load_confounds is not None:\\n         if \\"high_pass\\" in kwargs_load_confounds.get(\\"strategy\\"):\\n@@ -1903,18 +1911,18 @@ def _check_kwargs_load_confounds(**kwargs):\\n     }\\n \\n     if kwargs.get(\\"confounds_strategy\\") is None:\\n-        return None\\n-\\n-    kwargs_load_confounds = {\\n-        key: (\\n-            defaults[key]\\n-            if f\\"confounds_{key}\\" not in kwargs\\n-            else kwargs[f\\"confounds_{key}\\"]\\n-        )\\n-        for key in defaults\\n-    }\\n+        return None, kwargs\\n+\\n+    remaining_kwargs = kwargs.copy()\\n+    kwargs_load_confounds = {}\\n+    for key in defaults:\\n+        confounds_key = f\\"confounds_{key}\\"\\n+        if confounds_key in kwargs:\\n+            kwargs_load_confounds[key] = remaining_kwargs.pop(confounds_key)\\n+        else:\\n+            kwargs_load_confounds[key] = defaults[key]\\n \\n-    return kwargs_load_confounds\\n+    return kwargs_load_confounds, remaining_kwargs\\n \\n \\n def _make_bids_files_filter(\\n", "test_patch": "diff --git a/doc/changes/latest.rst b/doc/changes/latest.rst\\nindex 308946c1b..4f65a8636 100644\\n--- a/doc/changes/latest.rst\\n+++ b/doc/changes/latest.rst\\n@@ -28,3 +28,5 @@ Changes\\n - :bdg-dark:`Code` Implement argument ``sample_mask`` for :meth:`nilearn.maskers.MultiNiftiMasker.transform_imgs` (:gh:`4273` by `R\\u00e9mi Gau`_).\\n \\n - :bdg-dark:`Code` Remove the unused arguments ``upper_cutoff`` and ``exclude_zeros`` for :func:`nilearn.masking.compute_multi_background_mask` (:gh:`4273` by `R\\u00e9mi Gau`_).\\n+\\n+- :bdg-dark:`Code` Throw error in :func:`nilearn.glm.first_level.first_level_from_bids` if unknown ``kwargs`` are passed (:gh:`4414` by `Michelle Wang`_).\\ndiff --git a/nilearn/glm/tests/test_first_level.py b/nilearn/glm/tests/test_first_level.py\\nindex 1b4702499..6310e39be 100644\\n--- a/nilearn/glm/tests/test_first_level.py\\n+++ b/nilearn/glm/tests/test_first_level.py\\n@@ -1958,6 +1958,23 @@ def test_first_level_from_bids_no_subject(tmp_path):\\n         )\\n \\n \\n+def test_first_level_from_bids_unused_kwargs(tmp_path):\\n+    \\"\\"\\"Check that unused kwargs are properly handled.\\"\\"\\"\\n+    bids_path = create_fake_bids_dataset(\\n+        base_dir=tmp_path, n_sub=1, n_ses=1, tasks=[\\"main\\"], n_runs=[2]\\n+    )\\n+    with pytest.raises(RuntimeError, match=\\"Unknown keyword arguments\\"):\\n+        # wrong kwarg name `confound_strategy` (wrong)\\n+        # instead of `confounds_strategy` (correct)\\n+        first_level_from_bids(\\n+            dataset_path=bids_path,\\n+            task_label=\\"main\\",\\n+            space_label=\\"MNI\\",\\n+            slice_time_ref=None,\\n+            confound_strategy=\\"motion\\",\\n+        )\\n+\\n+\\n def test_check_run_tables_errors():\\n     # check high level wrapper keeps behavior\\n     with pytest.raises(ValueError, match=\\"len.* does not match len.*\\"):\\n", "problem_statement": "[ENH] Throw error when extra kwargs are given to `glm.first_level.first_level_from_bids`\\n### Is there an existing issue for this?\\n\\n- [X] I have searched the existing issues\\n\\n### Describe your proposed enhancement in detail.\\n\\n`nilearn.glm.first_level.first_level_from_bids` takes kwargs, and passes kwargs that have the `confounds_` prefix to another function. However, the remaining kwargs are not checked and nothing happens if for example the confound kwargs are not given correctly (e.g. with typo `confound_` instead of `confounds_`).\\r\\n\\r\\nI think it would be nice/useful to throw a runtime error or something when that happens (first mentioned in PR #4352). Would be happy to work on this too.\\n\\n### Benefits to the change\\n\\nBetter debugging when invalid/unknown kwargs are given.\\n\\n### Pseudocode for the new behavior, if applicable\\n\\n_No response_", "hints_text": "github-actions[bot]: \\ud83d\\udc4b @michellewang Thanks for creating a PR!\\n\\nUntil this PR is ready for review, you can include the [WIP] tag in its title, or leave it as a github draft.\\n\\nPlease make sure it is compliant with our [contributing guidelines](https://nilearn.github.io/stable/development.html#contribution-guidelines). In particular, be sure it checks the boxes listed below.\\n- [ ] PR has an interpretable title.\\n- [ ] PR links to Github issue with mention `Closes #XXXX` (see our documentation on [PR structure](https://nilearn.github.io/stable/development.html#pr-structure))\\n- [ ] Code is PEP8-compliant (see our documentation on [coding style](https://nilearn.github.io/stable/development.html#coding-style))\\n- [ ] Changelog or what\'s new entry in `doc/changes/latest.rst` (see our documentation on [PR structure](https://nilearn.github.io/stable/development.html#pr-structure))\\n\\nFor new features:\\n- [ ] There is at least one unit test per new function / class (see our documentation on [testing](https://nilearn.github.io/stable/development.html#tests))\\n- [ ] The new feature is demoed in at least one relevant example.\\n\\nFor bug fixes:\\n- [ ] There is at least one test that would fail under the original bug conditions.\\n\\nWe will review it as quick as possible, feel free to ping us with questions if needed.\\ncodecov[bot]: ## [Codecov](https://app.codecov.io/gh/nilearn/nilearn/pull/4414?dropdown=coverage&src=pr&el=h1&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn) Report\\nAll modified and coverable lines are covered by tests :white_check_mark:\\n> Project coverage is 68.73%. Comparing base [(`abb80ff`)](https://app.codecov.io/gh/nilearn/nilearn/commit/abb80ffb7aab132d096c2a8bc987e75040bacbfc?dropdown=coverage&el=desc&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn) to head [(`612a901`)](https://app.codecov.io/gh/nilearn/nilearn/pull/4414?dropdown=coverage&src=pr&el=desc&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn).\\n> Report is 70 commits behind head on main.\\n\\n\\n<details><summary>Additional details and impacted files</summary>\\n\\n\\n```diff\\n@@             Coverage Diff             @@\\n##             main    #4414       +/-   ##\\n===========================================\\n- Coverage   91.85%   68.73%   -23.12%     \\n===========================================\\n  Files         144      143        -1     \\n  Lines       16419    16641      +222     \\n  Branches     3434     3222      -212     \\n===========================================\\n- Hits        15082    11439     -3643     \\n- Misses        792     4753     +3961     \\n+ Partials      545      449       -96     \\n```\\n\\n| [Flag](https://app.codecov.io/gh/nilearn/nilearn/pull/4414/flags?src=pr&el=flags&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn) | Coverage \\u0394 | |\\n|---|---|---|\\n| [macos-latest_3.11_test_plotting](https://app.codecov.io/gh/nilearn/nilearn/pull/4414/flags?src=pr&el=flag&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn) | `?` | |\\n| [ubuntu-latest_3.10_test_plotting](https://app.codecov.io/gh/nilearn/nilearn/pull/4414/flags?src=pr&el=flag&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn) | `?` | |\\n| [ubuntu-latest_3.8_test_min](https://app.codecov.io/gh/nilearn/nilearn/pull/4414/flags?src=pr&el=flag&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn) | `68.73% <100.00%> (?)` | |\\n\\nFlags with carried forward coverage won\'t be shown. [Click here](https://docs.codecov.io/docs/carryforward-flags?utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn#carryforward-flags-in-the-pull-request-comment) to find out more.\\n\\n\\n</details>\\n\\n[:umbrella: View full report in Codecov by Sentry](https://app.codecov.io/gh/nilearn/nilearn/pull/4414?dropdown=coverage&src=pr&el=continue&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn).   \\n:loudspeaker: Have feedback on the report? [Share it here](https://about.codecov.io/codecov-pr-comment-feedback/?utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=nilearn).\\n\\nmichellewang: @man-shu it is now!", "created_at": "2024-05-16 21:54:11", "FAIL_TO_PASS": ["nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_unused_kwargs"], "PASS_TO_PASS": ["nilearn/glm/tests/test_first_level.py::test_high_level_glm_one_run", "nilearn/glm/tests/test_first_level.py::test_explicit_fixed_effects", "nilearn/glm/tests/test_first_level.py::test_explicit_fixed_effects_without_mask", "nilearn/glm/tests/test_first_level.py::test_high_level_glm_with_data", "nilearn/glm/tests/test_first_level.py::test_high_level_glm_with_data_with_mask", "nilearn/glm/tests/test_first_level.py::test_high_level_glm_with_paths", "nilearn/glm/tests/test_first_level.py::test_high_level_glm_null_contrasts", "nilearn/glm/tests/test_first_level.py::test_high_level_glm_different_design_matrices", "nilearn/glm/tests/test_first_level.py::test_high_level_glm_different_design_matrices_formulas", "nilearn/glm/tests/test_first_level.py::test_compute_contrast_num_contrasts", "nilearn/glm/tests/test_first_level.py::test_run_glm_ols", "nilearn/glm/tests/test_first_level.py::test_run_glm_ar1", "nilearn/glm/tests/test_first_level.py::test_run_glm_ar3", "nilearn/glm/tests/test_first_level.py::test_run_glm_errors", "nilearn/glm/tests/test_first_level.py::test_glm_AR_estimates[ar_vals0]", "nilearn/glm/tests/test_first_level.py::test_glm_AR_estimates[ar_vals1]", "nilearn/glm/tests/test_first_level.py::test_glm_AR_estimates[ar_vals2]", "nilearn/glm/tests/test_first_level.py::test_glm_AR_estimates_errors", "nilearn/glm/tests/test_first_level.py::test_glm_random_state[3]", "nilearn/glm/tests/test_first_level.py::test_glm_random_state[random_state1]", "nilearn/glm/tests/test_first_level.py::test_scaling", "nilearn/glm/tests/test_first_level.py::test_fmri_inputs", "nilearn/glm/tests/test_first_level.py::test_fmri_inputs_errors", "nilearn/glm/tests/test_first_level.py::test_first_level_design_creation", "nilearn/glm/tests/test_first_level.py::test_first_level_glm_computation", "nilearn/glm/tests/test_first_level.py::test_first_level_glm_computation_with_memory_caching", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_set_repetition_time_warnings", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_set_repetition_time_errors[not", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_set_repetition_time_errors[-1-ValueError-positive]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_set_slice_timing_ref_warnings", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_set_slice_timing_ref_errors[not", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_set_slice_timing_ref_errors[2-ValueError-between", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_get_metadata_from_derivatives", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_get_RepetitionTime_from_derivatives", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_get_StartTime_from_derivatives", "nilearn/glm/tests/test_first_level.py::test_first_level_contrast_computation", "nilearn/glm/tests/test_first_level.py::test_first_level_contrast_computation_errors", "nilearn/glm/tests/test_first_level.py::test_first_level_with_scaling", "nilearn/glm/tests/test_first_level.py::test_first_level_with_no_signal_scaling", "nilearn/glm/tests/test_first_level.py::test_first_level_residuals", "nilearn/glm/tests/test_first_level.py::test_first_level_residuals_errors", "nilearn/glm/tests/test_first_level.py::test_get_voxelwise_attributes_should_return_as_many_as_design_matrices[shapes0]", "nilearn/glm/tests/test_first_level.py::test_get_voxelwise_attributes_should_return_as_many_as_design_matrices[shapes1]", "nilearn/glm/tests/test_first_level.py::test_first_level_predictions_r_square", "nilearn/glm/tests/test_first_level.py::test_first_level_hrf_model[False-spm]", "nilearn/glm/tests/test_first_level.py::test_first_level_hrf_model[False-spm", "nilearn/glm/tests/test_first_level.py::test_first_level_hrf_model[False-glover]", "nilearn/glm/tests/test_first_level.py::test_first_level_hrf_model[False-<lambda>]", "nilearn/glm/tests/test_first_level.py::test_first_level_hrf_model[True-spm]", "nilearn/glm/tests/test_first_level.py::test_first_level_hrf_model[True-spm", "nilearn/glm/tests/test_first_level.py::test_first_level_hrf_model[True-glover]", "nilearn/glm/tests/test_first_level.py::test_first_level_hrf_model[True-<lambda>]", "nilearn/glm/tests/test_first_level.py::test_glm_sample_mask", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-0-0-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-0-0-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-0-0-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-0-1-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-0-1-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-0-1-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-0-2-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-0-2-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-0-2-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-1-0-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-1-0-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-1-0-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-1-1-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-1-1-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-1-1-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-1-2-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-1-2-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[MNI-1-2-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-0-0-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-0-0-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-0-0-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-0-1-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-0-1-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-0-1-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-0-2-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-0-2-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-0-2-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-1-0-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-1-0-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-1-0-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-1-1-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-1-1-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-1-1-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-1-2-n_runs0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-1-2-n_runs1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids[T1w-1-2-n_runs2]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_select_one_run_per_session", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_select_all_runs_of_one_session", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_smoke_test_for_verbose_argument[0]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_smoke_test_for_verbose_argument[1]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_several_labels_per_entity[acq]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_several_labels_per_entity[ce]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_several_labels_per_entity[dir]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_several_labels_per_entity[rec]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_several_labels_per_entity[echo]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_several_labels_per_entity[res]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_several_labels_per_entity[den]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_with_subject_labels", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_no_duplicate_sub_labels", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_input_dataset_path", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_task_label[42-TypeError]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_task_label[$$$-ValueError]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_sub_labels[42-TypeError-must", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_sub_labels[sub_labels1-TypeError-must", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_sub_labels[sub_labels2-TypeError-must", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_space_label[42-TypeError]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_space_label[$$$-ValueError]", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_img_filter[foo-TypeError-\'img_filters\'", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_img_filter[img_filters1-TypeError-Filters", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_img_filter[img_filters2-ValueError-bids", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_validation_img_filter[img_filters3-ValueError-is", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_too_many_bold_files", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_with_missing_events", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_no_tr", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_no_bold_file", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_with_one_events_missing", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_one_confound_missing", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_all_confounds_missing", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_no_derivatives", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_no_session", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_mismatch_run_index", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_deprecated_slice_time_default", "nilearn/glm/tests/test_first_level.py::test_slice_time_ref_warning_only_when_not_provided", "nilearn/glm/tests/test_first_level.py::test_check_trial_type_warning", "nilearn/glm/tests/test_first_level.py::test_missing_trial_type_column_warning", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_load_confounds", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_load_confounds_warnings", "nilearn/glm/tests/test_first_level.py::test_first_level_from_bids_no_subject", "nilearn/glm/tests/test_first_level.py::test_check_run_tables_errors", "nilearn/glm/tests/test_first_level.py::test_img_table_checks"], "environment_setup_commit": "c39ca5ec29463a262b635a47aadfb5dfc19cafb6", "version": "0.10", "license_name": "New BSD License", "install_config": {"python": "3.9", "install": "pip install -e .[dev]", "pre_install": null, "pip_packages": ["pytest", "pytest-cov", "pytest-xdist", "pytest-mock", "pytest-asyncio"], "reqs_path": ["requirements/base.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 1}}, {"instance_id": "christiansandberg__canopen-506", "repo": "christiansandberg/canopen", "base_commit": "fa74236b3f3312c03d09e72de3fa6d3cf077fbec", "patch": "diff --git a/canopen/nmt.py b/canopen/nmt.py\\nindex 401ad15..8ce737e 100644\\n--- a/canopen/nmt.py\\n+++ b/canopen/nmt.py\\n@@ -86,10 +86,10 @@ class NmtBase:\\n         - \'RESET\'\\n         - \'RESET COMMUNICATION\'\\n         \\"\\"\\"\\n-        if self._state in NMT_STATES:\\n+        try:\\n             return NMT_STATES[self._state]\\n-        else:\\n-            return self._state\\n+        except KeyError:\\n+            return f\\"UNKNOWN STATE \'{self._state}\'\\"\\n \\n     @state.setter\\n     def state(self, new_state: str):\\n", "test_patch": "diff --git a/test/test_nmt.py b/test/test_nmt.py\\nindex c24e8d9..b6892cf 100644\\n--- a/test/test_nmt.py\\n+++ b/test/test_nmt.py\\n@@ -88,12 +88,11 @@ class TestNmtMaster(unittest.TestCase):\\n         state = self.node.nmt.wait_for_heartbeat(self.TIMEOUT)\\n         self.assertEqual(state, \\"PRE-OPERATIONAL\\")\\n \\n-    @unittest.expectedFailure\\n     def test_nmt_master_on_heartbeat_unknown_state(self):\\n         task = self.net.send_periodic(self.COB_ID, [0xcb], self.PERIOD)\\n         self.addCleanup(task.stop)\\n         state = self.node.nmt.wait_for_heartbeat(self.TIMEOUT)\\n-        # Expect the high bit to be masked out, and and unknown state string to\\n+        # Expect the high bit to be masked out, and a formatted string to\\n         # be returned.\\n         self.assertEqual(state, \\"UNKNOWN STATE \'75\'\\")\\n \\n", "problem_statement": "`NmtBase.state` should always return a `str`\\n## Problem\\r\\n\\r\\n`NmtBase.state` is documented and annotated as always returning a string, however there is a code path that may return an `int`:\\r\\n\\r\\nhttps://github.com/christiansandberg/canopen/blob/62e9c1f851cb8721355d22c8571f2b41599c1b36/canopen/nmt.py#L74-L92\\r\\n\\r\\nThe code path in question is highly unlikely; the `NmtBase._state` member is set in the following places:\\r\\n\\r\\n- \\u2705 `NmtBase.__init__`: explicitly set to 0\\r\\n- \\u2705 `NmtBase.on_command`: set _iff_ the new state is a valid/known state\\r\\n- \\u2705 `NmtBase.send_command`: set _iff_ the new state is a valid/known state\\r\\n- \\u2753 `NmtMaster.on_heartbeat`: set to whatever was decoded from the heartbeat message, which _should_ be fine for any compliant device\\r\\n\\r\\n## Possible solutions\\r\\n\\r\\nIn `NmtMaster.on_heartbeat`, only set `_state` if it is a valid state; if it\'s not, `log.error(...)` or raise an exception.\\r\\n\\r\\nIn `NmtBase.state`, either raise an exception if `._state` is not a valid state, or return something a la `f\\"UNKNOWN STATE {self._state}\\"`.\\r\\n\\r\\n<details>\\r\\n<summary>Draft diff</summary>\\r\\n\\r\\n```diff\\r\\ndiff --git a/canopen/nmt.py b/canopen/nmt.py\\r\\nindex 401ad15..1316a5c 100644\\r\\n--- a/canopen/nmt.py\\r\\n+++ b/canopen/nmt.py\\r\\n@@ -86,10 +86,8 @@ class NmtBase:\\r\\n         - \'RESET\'\\r\\n         - \'RESET COMMUNICATION\'\\r\\n         \\"\\"\\"\\r\\n-        if self._state in NMT_STATES:\\r\\n-            return NMT_STATES[self._state]\\r\\n-        else:\\r\\n-            return self._state\\r\\n+        assert self._state in NMT_STATES:\\r\\n+        return NMT_STATES[self._state]\\r\\n \\r\\n     @state.setter\\r\\n     def state(self, new_state: str):\\r\\n@@ -122,6 +120,12 @@ class NmtMaster(NmtBase):\\r\\n             logger.debug(\\"Received heartbeat can-id %d, state is %d\\", can_id, new_state)\\r\\n             for callback in self._callbacks:\\r\\n                 callback(new_state)\\r\\n+            if new_state not in NMT_STATES:\\r\\n+                log.error(\\r\\n+                    \\"Received heartbeat can-id %d with invalid state %d\\",\\r\\n+                    can_id, new_state\\r\\n+                )\\r\\n+                return\\r\\n             if new_state == 0:\\r\\n                 # Boot-up, will go to PRE-OPERATIONAL automatically\\r\\n                 self._state = 127\\r\\n```\\r\\n\\r\\n</details>", "hints_text": "", "created_at": "2024-07-09 10:51:31", "FAIL_TO_PASS": ["test/test_nmt.py::TestNmtMaster::test_nmt_master_on_heartbeat_unknown_state"], "PASS_TO_PASS": ["test/test_nmt.py::TestNmtBase::test_send_command", "test/test_nmt.py::TestNmtBase::test_state_getset", "test/test_nmt.py::TestNmtBase::test_state_set_invalid", "test/test_nmt.py::TestNmtMaster::test_nmt_master_add_heartbeat_callback", "test/test_nmt.py::TestNmtMaster::test_nmt_master_no_heartbeat", "test/test_nmt.py::TestNmtMaster::test_nmt_master_node_guarding", "test/test_nmt.py::TestNmtMaster::test_nmt_master_on_heartbeat", "test/test_nmt.py::TestNmtMaster::test_nmt_master_on_heartbeat_initialising", "test/test_nmt.py::TestNmtSlave::test_heartbeat", "test/test_nmt.py::TestNmtSlave::test_start_two_remote_nodes", "test/test_nmt.py::TestNmtSlave::test_stop_two_remote_nodes_using_broadcast"], "environment_setup_commit": "932c6212bf077596dbfec91fd6ca66308142b972", "version": "2.3", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .[dev]", "pre_install": null, "pip_packages": ["pytest", "pytest-cov"], "reqs_path": ["requirements/base.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 0, "files": 1}}, {"instance_id": "tobymao__sqlglot-4523", "repo": "tobymao/sqlglot", "base_commit": "98906d4520a0c582a0534384ee3d0c1449846ee6", "patch": "diff --git a/sqlglot/dialects/tsql.py b/sqlglot/dialects/tsql.py\\nindex 856142ed..1acc5ca8 100644\\n--- a/sqlglot/dialects/tsql.py\\n+++ b/sqlglot/dialects/tsql.py\\n@@ -232,7 +232,7 @@ def _build_date_delta(\\n         if start_date and start_date.is_number:\\n             # Numeric types are valid DATETIME values\\n             if start_date.is_int:\\n-                adds = DEFAULT_START_DATE + datetime.timedelta(days=int(start_date.this))\\n+                adds = DEFAULT_START_DATE + datetime.timedelta(days=start_date.to_py())\\n                 start_date = exp.Literal.string(adds.strftime(\\"%F\\"))\\n             else:\\n                 # We currently don\'t handle float values, i.e. they\'re not converted to equivalent DATETIMEs.\\n", "test_patch": "diff --git a/tests/dialects/test_tsql.py b/tests/dialects/test_tsql.py\\nindex 61365994..e8cd6964 100644\\n--- a/tests/dialects/test_tsql.py\\n+++ b/tests/dialects/test_tsql.py\\n@@ -1579,6 +1579,11 @@ WHERE\\n             },\\n         )\\n \\n+        self.validate_identity(\\n+            \\"SELECT DATEADD(DAY, DATEDIFF(DAY, -3, GETDATE()), \'08:00:00\')\\",\\n+            \\"SELECT DATEADD(DAY, DATEDIFF(DAY, CAST(\'1899-12-29\' AS DATETIME2), CAST(GETDATE() AS DATETIME2)), \'08:00:00\')\\",\\n+        )\\n+\\n     def test_lateral_subquery(self):\\n         self.validate_all(\\n             \\"SELECT x.a, x.b, t.v, t.y FROM x CROSS APPLY (SELECT v, y FROM t) t(v, y)\\",\\n", "problem_statement": "TSQL fails on parsing DATEDIFF function with literals\\nHello,\\r\\nI just discovered that this code generates an error while parsing DATEDIFF second argument:\\r\\n```\\r\\nimport sqlglot\\r\\nstatement = \\"\\"\\"\\r\\nSELECT DATEADD(DAY, DATEDIFF(DAY, -3, CREATION_TIME_NEW), \'08:00:00\')\\r\\nFROM y\\r\\n\\"\\"\\"\\r\\n\\r\\nast = sqlglot.parse_one(statement, \'tsql\')\\r\\nprint(ast.sql(\'tsql\'))\\r\\n```\\r\\n\\r\\nError:\\r\\n```\\r\\nPython312\\\\Lib\\\\site-packages\\\\sqlglot\\\\dialects\\\\tsql.py:235, in _build_date_delta.<locals>._builder(args)\\r\\n\\r\\n    232 if start_date and start_date.is_number:\\r\\n    233     # Numeric types are valid DATETIME values\\r\\n    234     if start_date.is_int:\\r\\n--> 235         adds = DEFAULT_START_DATE + datetime.timedelta(days=int(start_date.this))\\r\\n    236         start_date = exp.Literal.string(adds.strftime(\\"%F\\"))\\r\\n    237     else:\\r\\n    238         # We currently don\'t handle float values, i.e. they\'re not converted to equivalent DATETIMEs.\\r\\n    239         # This is not a problem when generating T-SQL code, it is when transpiling to other dialects.\\r\\n\\r\\nTypeError: int() argument must be a string, a bytes-like object or a real number, not \'Literal\'\\r\\n```\\r\\nstart_date at this point is:\\r\\n```\\r\\nNeg(\\r\\n  this=Literal(this=3, is_string=False))\\r\\n```\\r\\n\\r\\nSo `int(start_date.this)` fails.", "hints_text": "", "created_at": "2024-12-16 13:12:42", "FAIL_TO_PASS": ["tests/dialects/test_tsql.py::TestTSQL::test_date_diff"], "PASS_TO_PASS": ["tests/dialects/test_tsql.py::TestTSQL::test_add_date", "tests/dialects/test_tsql.py::TestTSQL::test_charindex", "tests/dialects/test_tsql.py::TestTSQL::test_commit", "tests/dialects/test_tsql.py::TestTSQL::test_convert", "tests/dialects/test_tsql.py::TestTSQL::test_count", "tests/dialects/test_tsql.py::TestTSQL::test_current_user", "tests/dialects/test_tsql.py::TestTSQL::test_datefromparts", "tests/dialects/test_tsql.py::TestTSQL::test_datename", "tests/dialects/test_tsql.py::TestTSQL::test_datepart", "tests/dialects/test_tsql.py::TestTSQL::test_ddl", "tests/dialects/test_tsql.py::TestTSQL::test_declare", "tests/dialects/test_tsql.py::TestTSQL::test_eomonth", "tests/dialects/test_tsql.py::TestTSQL::test_format", "tests/dialects/test_tsql.py::TestTSQL::test_fullproc", "tests/dialects/test_tsql.py::TestTSQL::test_grant", "tests/dialects/test_tsql.py::TestTSQL::test_hints", "tests/dialects/test_tsql.py::TestTSQL::test_identifier_prefixes", "tests/dialects/test_tsql.py::TestTSQL::test_insert_cte", "tests/dialects/test_tsql.py::TestTSQL::test_isnull", "tests/dialects/test_tsql.py::TestTSQL::test_json", "tests/dialects/test_tsql.py::TestTSQL::test_lateral_subquery", "tests/dialects/test_tsql.py::TestTSQL::test_lateral_table_valued_function", "tests/dialects/test_tsql.py::TestTSQL::test_len", "tests/dialects/test_tsql.py::TestTSQL::test_next_value_for", "tests/dialects/test_tsql.py::TestTSQL::test_openjson", "tests/dialects/test_tsql.py::TestTSQL::test_option", "tests/dialects/test_tsql.py::TestTSQL::test_parsename", "tests/dialects/test_tsql.py::TestTSQL::test_procedure_keywords", "tests/dialects/test_tsql.py::TestTSQL::test_qualify_derived_table_outputs", "tests/dialects/test_tsql.py::TestTSQL::test_replicate", "tests/dialects/test_tsql.py::TestTSQL::test_rollback", "tests/dialects/test_tsql.py::TestTSQL::test_scope_resolution_op", "tests/dialects/test_tsql.py::TestTSQL::test_set", "tests/dialects/test_tsql.py::TestTSQL::test_string", "tests/dialects/test_tsql.py::TestTSQL::test_system_time", "tests/dialects/test_tsql.py::TestTSQL::test_temporal_table", "tests/dialects/test_tsql.py::TestTSQL::test_top", "tests/dialects/test_tsql.py::TestTSQL::test_transaction", "tests/dialects/test_tsql.py::TestTSQL::test_tsql", "tests/dialects/test_tsql.py::TestTSQL::test_types", "tests/dialects/test_tsql.py::TestTSQL::test_types_bin", "tests/dialects/test_tsql.py::TestTSQL::test_types_date", "tests/dialects/test_tsql.py::TestTSQL::test_types_decimals", "tests/dialects/test_tsql.py::TestTSQL::test_types_ints", "tests/dialects/test_tsql.py::TestTSQL::test_types_string", "tests/dialects/test_tsql.py::TestTSQL::test_udf"], "environment_setup_commit": "0d61fa0e28650a79d608e140bc725c46d6de5706", "version": "26.0", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .[dev]", "pre_install": null, "pip_packages": ["pytest pytest-cov pytest-xdist pytest-mock pytest-asyncio", "pytest"], "reqs_path": ["requirements/base.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 0, "files": 1}}, {"instance_id": "IAMconsortium__pyam-792", "repo": "IAMconsortium/pyam", "base_commit": "f023ec3444e0653408dbf95616b31fff745a2ca0", "patch": "diff --git a/RELEASE_NOTES.md b/RELEASE_NOTES.md\\nindex e7d85bc..d373650 100644\\n--- a/RELEASE_NOTES.md\\n+++ b/RELEASE_NOTES.md\\n@@ -1,5 +1,7 @@\\n # Release v2.0.0\\n \\n+- [#792](https://github.com/IAMconsortium/pyam/pull/792) Support region-aggregation with weights-index >> data-index\\n+\\n ## Highlights\\n \\n - Use **ixmp4** as dependency for better integration with the IIASA Scenario Explorer database infrastructure \\ndiff --git a/pyam/aggregation.py b/pyam/aggregation.py\\nindex ee781b6..a02c028 100644\\n--- a/pyam/aggregation.py\\n+++ b/pyam/aggregation.py\\n@@ -4,7 +4,7 @@ import logging\\n from itertools import compress\\n \\n from pyam.index import replace_index_values\\n-from pyam.logging import adjust_log_level\\n+from pyam.logging import adjust_log_level, format_log_message\\n from pyam.str import find_depth, is_str, reduce_hierarchy\\n from pyam.utils import KNOWN_FUNCS, is_list_like, to_list\\n from pyam._compare import _compare\\n@@ -116,7 +116,10 @@ def _aggregate_region(\\n         raise ValueError(\\"Using weights and components in one operation not supported.\\")\\n \\n     # default subregions to all regions other than `region`\\n-    subregions = subregions or df._all_other_regions(region, variable)\\n+    if weight is None:\\n+        subregions = subregions or df._all_other_regions(region, variable)\\n+    else:\\n+        subregions = subregions or df._all_other_regions(region, [variable, weight])\\n \\n     if not len(subregions):\\n         logger.info(\\n@@ -214,10 +217,28 @@ def _agg_weight(data, weight, method, drop_negative_weights):\\n         raise ValueError(\\"Only method \'np.sum\' allowed for weighted average.\\")\\n \\n     weight = weight.droplevel([\\"variable\\", \\"unit\\"])\\n+    data_index = data.droplevel([\\"variable\\", \\"unit\\"]).index\\n+\\n+    # check that weights exist for all data rows\\n+    missing_weights = data_index.difference(weight.index)\\n+    if not missing_weights.empty:\\n+        raise ValueError(\\n+            format_log_message(\\n+                \\"Missing weights for the following data rows\\", missing_weights\\n+            )\\n+        )\\n \\n-    if not data.droplevel([\\"variable\\", \\"unit\\"]).index.equals(weight.index):\\n-        raise ValueError(\\"Inconsistent index between variable and weight!\\")\\n+    # warn if no data exists for available weights\\n+    missing_data = weight.index.difference(data_index)\\n+    if not missing_data.empty:\\n+        logger.warning(\\n+            format_log_message(\\n+                \\"Ignoring weights for the following missing data rows\\", missing_data\\n+            )\\n+        )\\n+        weight[missing_data] = np.nan\\n \\n+    # remove (and warn) negative values from weights due to strange behavior\\n     if drop_negative_weights is True:\\n         if any(weight < 0):\\n             logger.warning(\\n", "test_patch": "diff --git a/tests/test_feature_aggregate.py b/tests/test_feature_aggregate.py\\nindex 10c27e6..54f75ba 100644\\n--- a/tests/test_feature_aggregate.py\\n+++ b/tests/test_feature_aggregate.py\\n@@ -1,5 +1,5 @@\\n import pytest\\n-import logging\\n+import re\\n \\n import numpy as np\\n import pandas as pd\\n@@ -238,7 +238,7 @@ def test_check_aggregate_region_log(simple_df, caplog):\\n @pytest.mark.parametrize(\\n     \\"variable\\",\\n     (\\n-        (\\"Primary Energy\\"),\\n+        \\"Primary Energy\\",\\n         ([\\"Primary Energy\\", \\"Primary Energy|Coal\\", \\"Primary Energy|Wind\\"]),\\n     ),\\n )\\n@@ -252,7 +252,7 @@ def test_aggregate_region_append(simple_df, variable):\\n @pytest.mark.parametrize(\\n     \\"variable\\",\\n     (\\n-        (\\"Primary Energy\\"),\\n+        \\"Primary Energy\\",\\n         ([\\"Primary Energy\\", \\"Primary Energy|Coal\\", \\"Primary Energy|Wind\\"]),\\n     ),\\n )\\n@@ -315,7 +315,13 @@ def test_aggregate_region_with_weights(simple_df, caplog):\\n     exp = simple_df.filter(variable=v, region=\\"World\\")\\n     assert_iamframe_equal(simple_df.aggregate_region(v, weight=w), exp)\\n \\n-    # test that dropping negative weights works as expected\\n+\\n+def test_aggregate_region_with_negative_weights(simple_df, caplog):\\n+    # carbon price shouldn\'t be summed but be weighted by emissions\\n+    v = \\"Price|Carbon\\"\\n+    w = \\"Emissions|CO2\\"\\n+\\n+    # dropping negative weights works as expected\\n     neg_weights_df = simple_df.copy()\\n     neg_weights_df._data[18] = -6\\n     exp = simple_df.filter(variable=v, region=\\"World\\", year=2010)\\n@@ -329,7 +335,7 @@ def test_aggregate_region_with_weights(simple_df, caplog):\\n     idx = caplog.messages.index(msg)\\n     assert caplog.records[idx].levelname == \\"WARNING\\"\\n \\n-    # test that not dropping negative weights works as expected\\n+    # *not* dropping negative weights works as expected\\n     exp = simple_df.filter(variable=v, region=\\"World\\")\\n     exp._data[0] = -8\\n     assert_iamframe_equal(\\n@@ -337,16 +343,57 @@ def test_aggregate_region_with_weights(simple_df, caplog):\\n     )\\n \\n \\n-def test_aggregate_region_with_weights_raises(simple_df):\\n+@pytest.mark.parametrize(\\n+    \\"filter_arg,log_message\\",\\n+    (\\n+        (dict(year=2010), \\"\\"),\\n+        (dict(), \\"model_a   scen_a  reg_b  2005\\\\n1  \\"),\\n+    ),\\n+)\\n+def test_aggregate_region_with_weights_inconsistent_index(\\n+    simple_df, caplog, filter_arg, log_message\\n+):\\n     # carbon price shouldn\'t be summed but be weighted by emissions\\n     v = \\"Price|Carbon\\"\\n     w = \\"Emissions|CO2\\"\\n \\n-    # inconsistent index of variable and weight raises an error\\n-    _df = simple_df.filter(variable=w, region=\\"reg_b\\", keep=False)\\n-    with pytest.raises(ValueError, match=\\"Inconsistent index between variable and wei\\"):\\n+    log_message = \\"\\\\n0  \\" + log_message + \\"model_a   scen_a  reg_b  2010\\"\\n+    if simple_df.time_domain == \\"datetime\\":\\n+        time_col = \\"     time\\"\\n+        log_message = log_message.replace(\\" 2005\\", \\"2005-06-17\\").replace(\\n+            \\" 2010\\", \\"2010-07-21\\"\\n+        )\\n+    else:\\n+        time_col = \\"year\\"\\n+\\n+    # missing weight row raises an error\\n+    _df = simple_df.filter(variable=w, region=\\"reg_b\\", keep=False, **filter_arg)\\n+    match = r\\"Missing weights for the following data.*\\\\n.*\\" + re.escape(log_message)\\n+    with pytest.raises(ValueError, match=match):\\n         _df.aggregate_region(v, weight=w)\\n \\n+    # missing data row prints a warning (data-index is a subset of weight-index)\\n+    exp = simple_df.filter(variable=v, region=\\"World\\")\\n+    if not filter_arg:\\n+        exp._data[0] = 1.0\\n+    exp._data[1] = 30.0\\n+    _df = simple_df.filter(variable=v, region=\\"reg_b\\", keep=False, **filter_arg)\\n+    assert_iamframe_equal(_df.aggregate_region(v, weight=w), exp)\\n+\\n+    msg = (\\n+        \\"Ignoring weights for the following missing data rows:\\\\n\\"\\n+        f\\"     model scenario region  {time_col}\\" + log_message\\n+    )\\n+\\n+    idx = caplog.messages.index(msg)\\n+    assert caplog.records[idx].levelname == \\"WARNING\\"\\n+\\n+\\n+def test_aggregate_region_with_weights_raises(simple_df):\\n+    # carbon price shouldn\'t be summed but be weighted by emissions\\n+    v = \\"Price|Carbon\\"\\n+    w = \\"Emissions|CO2\\"\\n+\\n     # using weight and method other than \'sum\' raises an error\\n     pytest.raises(ValueError, simple_df.aggregate_region, v, method=\\"max\\", weight=\\"bar\\")\\n \\ndiff --git a/tests/test_feature_growth_rate.py b/tests/test_feature_growth_rate.py\\nindex fa9ea3b..708e98b 100644\\n--- a/tests/test_feature_growth_rate.py\\n+++ b/tests/test_feature_growth_rate.py\\n@@ -66,5 +66,5 @@ def test_growth_rate_timeseries(x2010, rates):\\n def test_growth_rate_timeseries_fails(value):\\n     \\"\\"\\"Check that a timeseries reaching/crossing 0 raises\\"\\"\\"\\n \\n-    with pytest.raises(ValueError, match=\\"Cannot compute growth rate when*.\\"):\\n+    with pytest.raises(ValueError, match=\\"Cannot compute growth rate when\\"):\\n         growth_rate(pd.Series([1.0, value]))\\ndiff --git a/tests/test_io.py b/tests/test_io.py\\nindex c358e95..fddf907 100644\\n--- a/tests/test_io.py\\n+++ b/tests/test_io.py\\n@@ -43,7 +43,7 @@ def test_not_a_file():\\n \\n def test_io_list():\\n     # initializing with a list raises an error\\n-    match = r\\"Initializing from list is not supported,*.\\"\\n+    match = \\"Initializing from list is not supported,\\"\\n     with pytest.raises(ValueError, match=match):\\n         IamDataFrame([1, 2])\\n \\n", "problem_statement": "df.aggregate_region() when inconsistent index (e.g. years)\\nThere are instances when doing weighted aggregation of  regions, when years between the variable and the weight variable may not be aligned or one variable has more years than the other.  One gets error message:\\r\\n```\\r\\n  File \\"c:\\\\github\\\\pyam\\\\pyam\\\\aggregation.py\\", line 216, in _agg_weight\\r\\n    raise ValueError(\\"Inconsistent index between variable and weight!\\")\\r\\n\\r\\nValueError: Inconsistent index between variable and weight!\\r\\n```\\r\\nwhich comes from [here](https://github.com/IAMconsortium/pyam/blob/6de47ccec5d477a1a35425cea557849df699238c/pyam/aggregation.py#L215):\\r\\n```\\r\\n    if not data.droplevel([\\"variable\\", \\"unit\\"]).index.equals(weight.index):\\r\\n        raise ValueError(\\"Inconsistent index between variable and weight!\\")\\r\\n```\\r\\n\\r\\nIt would be nice if there\'s a way to improve the logic here - e.g. for example, the minimum requirement should be not that the indexes are equal, but that the index of the variable A being weighted, is 100% present within the weight variable B. Im rusty on set notation, but basically that index A is a subset of index B?\\r\\n\\r\\nAny quick  suggestion  how this should be done and/or arguments against?\\r\\n\\r\\n", "hints_text": "codecov[bot]: ## [Codecov](https://app.codecov.io/gh/IAMconsortium/pyam/pull/792?src=pr&el=h1&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=IAMconsortium) Report\\n> Merging [#792](https://app.codecov.io/gh/IAMconsortium/pyam/pull/792?src=pr&el=desc&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=IAMconsortium) (6d11700) into [main](https://app.codecov.io/gh/IAMconsortium/pyam/commit/f023ec3444e0653408dbf95616b31fff745a2ca0?el=desc&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=IAMconsortium) (f023ec3) will **increase** coverage by `0.1%`.\\n> The diff coverage is `100.0%`.\\n\\n```diff\\n@@           Coverage Diff           @@\\n##            main    #792     +/-   ##\\n=======================================\\n+ Coverage   94.5%   94.7%   +0.1%     \\n=======================================\\n  Files         62      62             \\n  Lines       5966    5996     +30     \\n=======================================\\n+ Hits        5643    5681     +38     \\n+ Misses       323     315      -8     \\n```\\n\\n\\n| [Files](https://app.codecov.io/gh/IAMconsortium/pyam/pull/792?src=pr&el=tree&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=IAMconsortium) | Coverage \\u0394 | |\\n|---|---|---|\\n| [pyam/aggregation.py](https://app.codecov.io/gh/IAMconsortium/pyam/pull/792?src=pr&el=tree&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=IAMconsortium#diff-cHlhbS9hZ2dyZWdhdGlvbi5weQ==) | `99.2% <100.0%> (+<0.1%)` | :arrow_up: |\\n| [tests/test\\\\_feature\\\\_aggregate.py](https://app.codecov.io/gh/IAMconsortium/pyam/pull/792?src=pr&el=tree&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=IAMconsortium#diff-dGVzdHMvdGVzdF9mZWF0dXJlX2FnZ3JlZ2F0ZS5weQ==) | `99.0% <100.0%> (+0.1%)` | :arrow_up: |\\n| [tests/test\\\\_feature\\\\_growth\\\\_rate.py](https://app.codecov.io/gh/IAMconsortium/pyam/pull/792?src=pr&el=tree&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=IAMconsortium#diff-dGVzdHMvdGVzdF9mZWF0dXJlX2dyb3d0aF9yYXRlLnB5) | `100.0% <100.0%> (\\u00f8)` | |\\n| [tests/test\\\\_io.py](https://app.codecov.io/gh/IAMconsortium/pyam/pull/792?src=pr&el=tree&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=IAMconsortium#diff-dGVzdHMvdGVzdF9pby5weQ==) | `100.0% <100.0%> (\\u00f8)` | |\\n\\n... and [2 files with indirect coverage changes](https://app.codecov.io/gh/IAMconsortium/pyam/pull/792/indirect-changes?src=pr&el=tree-more&utm_medium=referral&utm_source=github&utm_content=comment&utm_campaign=pr+comments&utm_term=IAMconsortium)\\n\\n\\ndanielhuppmann: I vaguely recall that @orichters expressed interest in a more flexible handling of the region-aggregation-with-weights...?\\norichters: > I vaguely recall that @orichters expressed interest in a more flexible handling of the region-aggregation-with-weights...?\\r\\n\\r\\nI do not \\ud83d\\ude03. Maybe @Renato-Rodrigues, if it was someone from the REMIND team?", "created_at": "2023-10-17 20:44:04", "FAIL_TO_PASS": ["tests/test_feature_aggregate.py::test_aggregate_region_with_weights_inconsistent_index[None-filter_arg0-]", "tests/test_feature_aggregate.py::test_aggregate_region_with_weights_inconsistent_index[None-filter_arg1-model_a", "tests/test_feature_aggregate.py::test_aggregate_region_with_weights_inconsistent_index[datetime-filter_arg0-]", "tests/test_feature_aggregate.py::test_aggregate_region_with_weights_inconsistent_index[datetime-filter_arg1-model_a"], "PASS_TO_PASS": ["tests/test_feature_aggregate.py::test_aggregate[None-Primary", "tests/test_feature_aggregate.py::test_aggregate[None-variable1-data1]", "tests/test_feature_aggregate.py::test_aggregate[datetime-Primary", "tests/test_feature_aggregate.py::test_aggregate[datetime-variable1-data1]", "tests/test_feature_aggregate.py::test_check_aggregate[None]", "tests/test_feature_aggregate.py::test_check_aggregate[datetime]", "tests/test_feature_aggregate.py::test_check_aggregate_top_level[None]", "tests/test_feature_aggregate.py::test_check_aggregate_top_level[datetime]", "tests/test_feature_aggregate.py::test_aggregate_append[None-Primary", "tests/test_feature_aggregate.py::test_aggregate_append[None-variable1]", "tests/test_feature_aggregate.py::test_aggregate_append[datetime-Primary", "tests/test_feature_aggregate.py::test_aggregate_append[datetime-variable1]", "tests/test_feature_aggregate.py::test_aggregate_with_components[None]", "tests/test_feature_aggregate.py::test_aggregate_with_components[datetime]", "tests/test_feature_aggregate.py::test_aggregate_by_list_with_components_raises[None]", "tests/test_feature_aggregate.py::test_aggregate_by_list_with_components_raises[datetime]", "tests/test_feature_aggregate.py::test_aggregate_recursive[year]", "tests/test_feature_aggregate.py::test_aggregate_recursive[datetime]", "tests/test_feature_aggregate.py::test_aggregate_skip_intermediate[year]", "tests/test_feature_aggregate.py::test_aggregate_skip_intermediate[datetime]", "tests/test_feature_aggregate.py::test_aggregate_unknown_method[None]", "tests/test_feature_aggregate.py::test_aggregate_unknown_method[datetime]", "tests/test_feature_aggregate.py::test_aggregate_components_as_dict[None]", "tests/test_feature_aggregate.py::test_aggregate_components_as_dict[datetime]", "tests/test_feature_aggregate.py::test_aggregate_region[None-Primary", "tests/test_feature_aggregate.py::test_aggregate_region[None-variable1]", "tests/test_feature_aggregate.py::test_aggregate_region[datetime-Primary", "tests/test_feature_aggregate.py::test_aggregate_region[datetime-variable1]", "tests/test_feature_aggregate.py::test_check_aggregate_region[None]", "tests/test_feature_aggregate.py::test_check_aggregate_region[datetime]", "tests/test_feature_aggregate.py::test_aggregate_region_append[None-Primary", "tests/test_feature_aggregate.py::test_aggregate_region_append[None-variable1]", "tests/test_feature_aggregate.py::test_aggregate_region_append[datetime-Primary", "tests/test_feature_aggregate.py::test_aggregate_region_append[datetime-variable1]", "tests/test_feature_aggregate.py::test_aggregate_region_with_subregions[None-Primary", "tests/test_feature_aggregate.py::test_aggregate_region_with_subregions[None-variable1]", "tests/test_feature_aggregate.py::test_aggregate_region_with_subregions[datetime-Primary", "tests/test_feature_aggregate.py::test_aggregate_region_with_subregions[datetime-variable1]", "tests/test_feature_aggregate.py::test_aggregate_region_with_other_method[None-Price|Carbon-data0]", "tests/test_feature_aggregate.py::test_aggregate_region_with_other_method[None-variable1-data1]", "tests/test_feature_aggregate.py::test_aggregate_region_with_other_method[datetime-Price|Carbon-data0]", "tests/test_feature_aggregate.py::test_aggregate_region_with_other_method[datetime-variable1-data1]", "tests/test_feature_aggregate.py::test_aggregate_region_with_components[None]", "tests/test_feature_aggregate.py::test_aggregate_region_with_components[datetime]", "tests/test_feature_aggregate.py::test_aggregate_region_with_weights[None]", "tests/test_feature_aggregate.py::test_aggregate_region_with_weights[datetime]", "tests/test_feature_aggregate.py::test_aggregate_region_with_negative_weights[None]", "tests/test_feature_aggregate.py::test_aggregate_region_with_negative_weights[datetime]", "tests/test_feature_aggregate.py::test_aggregate_region_with_weights_raises[None]", "tests/test_feature_aggregate.py::test_aggregate_region_with_weights_raises[datetime]", "tests/test_feature_aggregate.py::test_aggregate_region_unknown_method[None]", "tests/test_feature_aggregate.py::test_aggregate_region_unknown_method[datetime]", "tests/test_feature_aggregate.py::test_aggregate_time[Primary", "tests/test_feature_aggregate.py::test_aggregate_time[variable1]", "tests/test_feature_aggregate.py::test_check_internal_consistency[None]", "tests/test_feature_aggregate.py::test_check_internal_consistency[datetime]", "tests/test_feature_growth_rate.py::test_growth_rate[False]", "tests/test_feature_growth_rate.py::test_growth_rate[True]", "tests/test_feature_growth_rate.py::test_growth_rate_empty[False]", "tests/test_feature_growth_rate.py::test_growth_rate_empty[True]", "tests/test_feature_growth_rate.py::test_growth_rate_timeseries[rates0-1]", "tests/test_feature_growth_rate.py::test_growth_rate_timeseries[rates0-27]", "tests/test_feature_growth_rate.py::test_growth_rate_timeseries[rates0--3]", "tests/test_feature_growth_rate.py::test_growth_rate_timeseries[rates1-1]", "tests/test_feature_growth_rate.py::test_growth_rate_timeseries[rates1-27]", "tests/test_feature_growth_rate.py::test_growth_rate_timeseries[rates1--3]", "tests/test_feature_growth_rate.py::test_growth_rate_timeseries_fails[0]", "tests/test_feature_growth_rate.py::test_growth_rate_timeseries_fails[-1]", "tests/test_io.py::test_data_none", "tests/test_io.py::test_unknown_type", "tests/test_io.py::test_not_a_file", "tests/test_io.py::test_io_list", "tests/test_io.py::test_io_csv_to_file[test_df0]", "tests/test_io.py::test_io_csv_to_file[test_df1]", "tests/test_io.py::test_io_csv_to_file[test_df2]", "tests/test_io.py::test_io_csv_to_file[test_df3]", "tests/test_io.py::test_io_csv_none", "tests/test_io.py::test_io_xlsx[test_df0-meta_args0]", "tests/test_io.py::test_io_xlsx[test_df0-meta_args1]", "tests/test_io.py::test_io_xlsx[test_df1-meta_args0]", "tests/test_io.py::test_io_xlsx[test_df1-meta_args1]", "tests/test_io.py::test_io_xlsx[test_df2-meta_args0]", "tests/test_io.py::test_io_xlsx[test_df2-meta_args1]", "tests/test_io.py::test_io_xlsx[test_df3-meta_args0]", "tests/test_io.py::test_io_xlsx[test_df3-meta_args1]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df0-sheets0-sheetname0]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df0-sheets1-sheetname1]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df0-sheets2-sheetname2]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df1-sheets0-sheetname0]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df1-sheets1-sheetname1]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df1-sheets2-sheetname2]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df2-sheets0-sheetname0]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df2-sheets1-sheetname1]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df2-sheets2-sheetname2]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df3-sheets0-sheetname0]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df3-sheets1-sheetname1]", "tests/test_io.py::test_io_xlsx_multiple_data_sheets[test_df3-sheets2-sheetname2]", "tests/test_io.py::test_read_xls", "tests/test_io.py::test_init_df_with_na_unit", "tests/test_io.py::test_init_df_with_na_column_raises", "tests/test_io.py::test_load_meta_xlsx[meta-init_args0-False]", "tests/test_io.py::test_load_meta_xlsx[meta-init_args1-False]", "tests/test_io.py::test_load_meta_xlsx[foo-init_args2-False]", "tests/test_io.py::test_load_meta_xlsx[foo-init_args3-True]", "tests/test_io.py::test_load_meta_csv[True]", "tests/test_io.py::test_load_meta_csv[False]", "tests/test_io.py::test_load_meta_wrong_index", "tests/test_io.py::test_load_meta_empty_rows", "tests/test_io.py::test_load_meta_exclude", "tests/test_io.py::test_load_meta_empty", "tests/test_io.py::test_load_ssp_database_downloaded_file", "tests/test_io.py::test_load_rcp_database_downloaded_file", "tests/test_io.py::test_io_datapackage[test_df0]", "tests/test_io.py::test_io_datapackage[test_df1]", "tests/test_io.py::test_io_datapackage[test_df2]", "tests/test_io.py::test_io_datapackage[test_df3]"], "environment_setup_commit": "f023ec3444e0653408dbf95616b31fff745a2ca0", "version": "2.0", "license_name": "Apache License 2.0", "install_config": {"python": "3.10", "install": "pip install --editable .[tests,optional_io_formats,tutorials]", "pre_install": null, "pip_packages": ["pytest"], "reqs_path": null, "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 2}}, {"instance_id": "CrossGL__crosstl-98", "repo": "CrossGL/crosstl", "base_commit": "d4a35e6554bc249fff7764276173ba42c6b5d422", "patch": "diff --git a/crosstl/src/translator/parser.py b/crosstl/src/translator/parser.py\\nindex 82899e6..a46a9d2 100644\\n--- a/crosstl/src/translator/parser.py\\n+++ b/crosstl/src/translator/parser.py\\n@@ -638,6 +638,10 @@ class Parser:\\n             \\"GREATER_THAN\\",\\n             \\"LESS_EQUAL\\",\\n             \\"GREATER_EQUAL\\",\\n+            \\"ASSIGN_AND\\",\\n+            \\"ASSIGN_OR\\",\\n+            \\"ASSIGN_XOR\\",\\n+            \\"ASSIGN_MOD\\",\\n             \\"BITWISE_SHIFT_RIGHT\\",\\n             \\"BITWISE_SHIFT_LEFT\\",\\n         ]:\\n@@ -694,6 +698,10 @@ class Parser:\\n             \\"GREATER_THAN\\",\\n             \\"LESS_EQUAL\\",\\n             \\"GREATER_EQUAL\\",\\n+            \\"ASSIGN_AND\\",\\n+            \\"ASSIGN_OR\\",\\n+            \\"ASSIGN_XOR\\",\\n+            \\"ASSIGN_MOD\\",\\n             \\"BITWISE_SHIFT_RIGHT\\",\\n             \\"BITWISE_SHIFT_LEFT\\",\\n         ]:\\n@@ -727,6 +735,10 @@ class Parser:\\n             \\"BITWISE_SHIFT_RIGHT\\",\\n             \\"BITWISE_SHIFT_LEFT\\",\\n             \\"EQUAL\\",\\n+            \\"ASSIGN_AND\\",\\n+            \\"ASSIGN_OR\\",\\n+            \\"ASSIGN_XOR\\",\\n+            \\"ASSIGN_MOD\\",\\n         ):\\n             op = self.current_token[0]\\n             self.eat(op)\\n@@ -774,6 +786,10 @@ class Parser:\\n             \\"GREATER_THAN\\",\\n             \\"LESS_EQUAL\\",\\n             \\"GREATER_EQUAL\\",\\n+            \\"ASSIGN_AND\\",\\n+            \\"ASSIGN_OR\\",\\n+            \\"ASSIGN_XOR\\",\\n+            \\"ASSIGN_MOD\\",\\n             \\"BITWISE_SHIFT_RIGHT\\",\\n             \\"BITWISE_SHIFT_LEFT\\",\\n         ]:\\n@@ -934,6 +950,10 @@ class Parser:\\n             \\"ASSIGN_SUB\\",\\n             \\"ASSIGN_MUL\\",\\n             \\"ASSIGN_DIV\\",\\n+            \\"ASSIGN_AND\\",\\n+            \\"ASSIGN_OR\\",\\n+            \\"ASSIGN_XOR\\",\\n+            \\"ASSIGN_MOD\\",\\n             \\"BITWISE_SHIFT_RIGHT\\",\\n             \\"BITWISE_SHIFT_LEFT\\",\\n         ]:\\n", "test_patch": "diff --git a/tests/test_translator/test_parser.py b/tests/test_translator/test_parser.py\\nindex 672f075..95c1eae 100644\\n--- a/tests/test_translator/test_parser.py\\n+++ b/tests/test_translator/test_parser.py\\n@@ -335,6 +335,50 @@ def test_var_assignment():\\n         pytest.fail(\\"Variable assignment parsing not implemented.\\")\\n \\n \\n+def test_assign_ops():\\n+\\n+    code = \\"\\"\\"\\n+            shader LightControl {\\n+                vertex {\\n+                    input vec3 position;\\n+                    output int lightStatus;\\n+\\n+                    void main() {\\n+                        int xStatus = int(position.x * 10.0);\\n+                        int yStatus = int(position.y * 10.0);\\n+                        int zStatus = int(position.z * 10.0);\\n+\\n+                        xStatus |= yStatus;\\n+                        yStatus &= zStatus;\\n+                        zStatus %= xStatus;\\n+                        lightStatus = xStatus;\\n+                        lightStatus ^= zStatus;\\n+\\n+                        gl_Position = vec4(position, 1.0);\\n+                    }\\n+                }\\n+\\n+                fragment {\\n+                    input int lightStatus;\\n+                    output vec4 fragColor;\\n+\\n+                    void main() {\\n+                        if (lightStatus > 0) {\\n+                            fragColor = vec4(1.0, 1.0, 0.0, 1.0); \\n+                        } else {\\n+                            fragColor = vec4(0.0, 0.0, 0.0, 1.0);\\n+                        }\\n+                    }\\n+                }\\n+            }\\n+        \\"\\"\\"\\n+    try:\\n+        tokens = tokenize_code(code)\\n+        parse_code(tokens)\\n+    except SyntaxError:\\n+        pytest.fail(\\"Assignment Operator parsing not implemented.\\")\\n+\\n+\\n def test_bitwise_operators():\\n     code = \\"\\"\\"\\n         shader LightControl {\\n", "problem_statement": "Add Parsing for `Assignment AND`  Token\\nUpdate the parser to handle the ASSIGN_AND token, allowing it to correctly parse expressions involving the &= operator.", "hints_text": "ArchitGupta07: Updated PR description\\r\\n\\nsamthakur587: hii @ArchitGupta07  can you please resolve the merge conflict \\nArchitGupta07: Resolved the merge conflict\\nsamthakur587: hii @ArchitGupta07 can you please resolve the merge conflicts \\nArchitGupta07: Resolved the merge conflicts\\ncoderabbitai[bot]: <!-- This is an auto-generated comment: summarize by coderabbit.ai -->\\n<!-- This is an auto-generated comment: summarize review in progress by coderabbit.ai -->\\n\\n> [!NOTE]\\n> Currently processing new changes in this PR. This may take a few minutes, please wait...\\n> \\n> \\t<details>\\n> <summary>Commits</summary>\\n> \\n> Files that changed from the base of the PR and between d4a35e6554bc249fff7764276173ba42c6b5d422 and a635301ecd0825e092f99e4be9def47b584386a4.\\n> \\n> </details>\\n> \\n> \\n> <details>\\n> <summary>Files selected for processing (2)</summary>\\n> \\n> * crosstl/src/translator/parser.py (5 hunks)\\n> * tests/test_translator/test_parser.py (1 hunks)\\n> \\n> </details>\\n> \\n> \\t```ascii\\n>  _________________________________________________\\n> < Code Wars Episode II: Attack of the Git Clones. >\\n>  -------------------------------------------------\\n>   \\\\\\n>    \\\\   (\\\\__/)\\n>        (\\u2022\\u3145\\u2022)\\n>        / \\u3000 \\u3065\\n> ```\\n\\n<!-- end of auto-generated comment: summarize review in progress by coderabbit.ai --><!-- usage_tips_start -->\\n\\n> [!TIP]\\n> <details>\\n> <summary>You can generate walkthrough in a markdown collapsible section to save space.</summary>\\n> \\n> Enable the `reviews.collapse_walkthrough` setting in your project\'s settings in CodeRabbit to generate walkthrough in a markdown collapsible section.\\n> \\n> </details>\\n\\n<!-- usage_tips_end --><!-- tips_start -->\\n\\n---\\n\\nThank you for using CodeRabbit. We offer it for free to the OSS community and would appreciate your support in helping us grow. If you find it useful, would you consider giving us a shout-out on your favorite social media?\\n\\n<details>\\n<summary>Share</summary>\\n\\n- [X](https://twitter.com/intent/tweet?text=I%20just%20used%20%40coderabbitai%20for%20my%20code%20review%2C%20and%20it%27s%20fantastic%21%20It%27s%20free%20for%20OSS%20and%20offers%20a%20free%20trial%20for%20the%20proprietary%20code.%20Check%20it%20out%3A&url=https%3A//coderabbit.ai)\\n- [Mastodon](https://mastodon.social/share?text=I%20just%20used%20%40coderabbitai%20for%20my%20code%20review%2C%20and%20it%27s%20fantastic%21%20It%27s%20free%20for%20OSS%20and%20offers%20a%20free%20trial%20for%20the%20proprietary%20code.%20Check%20it%20out%3A%20https%3A%2F%2Fcoderabbit.ai)\\n- [Reddit](https://www.reddit.com/submit?title=Great%20tool%20for%20code%20review%20-%20CodeRabbit&text=I%20just%20used%20CodeRabbit%20for%20my%20code%20review%2C%20and%20it%27s%20fantastic%21%20It%27s%20free%20for%20OSS%20and%20offers%20a%20free%20trial%20for%20proprietary%20code.%20Check%20it%20out%3A%20https%3A//coderabbit.ai)\\n- [LinkedIn](https://www.linkedin.com/sharing/share-offsite/?url=https%3A%2F%2Fcoderabbit.ai&mini=true&title=Great%20tool%20for%20code%20review%20-%20CodeRabbit&summary=I%20just%20used%20CodeRabbit%20for%20my%20code%20review%2C%20and%20it%27s%20fantastic%21%20It%27s%20free%20for%20OSS%20and%20offers%20a%20free%20trial%20for%20proprietary%20code)\\n\\n</details>\\n\\n<details>\\n<summary>Tips</summary>\\n\\n### Chat\\n\\nThere are 3 ways to chat with [CodeRabbit](https://coderabbit.ai):\\n\\n- Review comments: Directly reply to a review comment made by CodeRabbit. Example:\\n\\t- `I pushed a fix in commit <commit_id>.`\\n\\t- `Generate unit testing code for this file.`\\n\\t- `Open a follow-up GitHub issue for this discussion.`\\n- Files and specific lines of code (under the \\"Files changed\\" tab): Tag `@coderabbitai` in a new review comment at the desired location with your query. Examples:\\n\\t- `@coderabbitai generate unit testing code for this file.`\\n\\t-\\t`@coderabbitai modularize this function.`\\n- PR comments: Tag `@coderabbitai` in a new PR comment to ask questions about the PR branch. For the best results, please provide a very specific query, as very limited context is provided in this mode. Examples:\\n\\t- `@coderabbitai generate interesting stats about this repository and render them as a table.`\\n\\t- `@coderabbitai show all the console.log statements in this repository.`\\n\\t- `@coderabbitai read src/utils.ts and generate unit testing code.`\\n\\t- `@coderabbitai read the files in the src/scheduler package and generate a class diagram using mermaid and a README in the markdown format.`\\n\\t- `@coderabbitai help me debug CodeRabbit configuration file.`\\n\\nNote: Be mindful of the bot\'s finite context window. It\'s strongly recommended to break down tasks such as reading entire modules into smaller chunks. For a focused discussion, use review comments to chat about specific files and their changes, instead of using the PR comments.\\n\\n### CodeRabbit Commands (Invoked using PR comments)\\n\\n- `@coderabbitai pause` to pause the reviews on a PR.\\n- `@coderabbitai resume` to resume the paused reviews.\\n- `@coderabbitai review` to trigger an incremental review. This is useful when automatic reviews are disabled for the repository.\\n- `@coderabbitai full review` to do a full review from scratch and review all the files again.\\n- `@coderabbitai summary` to regenerate the summary of the PR.\\n- `@coderabbitai resolve` resolve all the CodeRabbit review comments.\\n- `@coderabbitai configuration` to show the current CodeRabbit configuration for the repository.\\n- `@coderabbitai help` to get help.\\n\\n### Other keywords and placeholders\\n\\n- Add `@coderabbitai ignore` anywhere in the PR description to prevent this PR from being reviewed.\\n- Add `@coderabbitai summary` to generate the high-level summary at a specific location in the PR description.\\n- Add `@coderabbitai` anywhere in the PR title to generate the title automatically.\\n\\n### CodeRabbit Configuration File (`.coderabbit.yaml`)\\n\\n- You can programmatically configure CodeRabbit by adding a `.coderabbit.yaml` file to the root of your repository.\\n- Please see the [configuration documentation](https://docs.coderabbit.ai/guides/configure-coderabbit) for more information.\\n- If your editor has YAML language server enabled, you can add the path at the top of this file to enable auto-completion and validation: `# yaml-language-server: $schema=https://coderabbit.ai/integrations/schema.v2.json`\\n\\n### Documentation and Community\\n\\n- Visit our [Documentation](https://coderabbit.ai/docs) for detailed information on how to use CodeRabbit.\\n- Join our [Discord Community](https://discord.com/invite/GsXnASn26c) to get help, request features, and share feedback.\\n- Follow us on [X/Twitter](https://twitter.com/coderabbitai) for updates and announcements.\\n\\n</details>\\n\\n\\n\\n<!-- tips_end -->", "created_at": "2024-08-25 10:36:57", "FAIL_TO_PASS": ["tests/test_translator/test_parser.py::test_assign_ops"], "PASS_TO_PASS": ["tests/test_translator/test_parser.py::test_input_output", "tests/test_translator/test_parser.py::test_if_statement", "tests/test_translator/test_parser.py::test_for_statement", "tests/test_translator/test_parser.py::test_else_statement", "tests/test_translator/test_parser.py::test_else_if_statement", "tests/test_translator/test_parser.py::test_function_call", "tests/test_translator/test_parser.py::test_logical_operators", "tests/test_translator/test_parser.py::test_var_assignment", "tests/test_translator/test_parser.py::test_bitwise_operators"], "environment_setup_commit": "36bed5871a8d102f73cfebf82c8d8495aaa89e87", "version": "0.0", "license_name": "Apache License 2.0", "install_config": {"python": "3.9", "install": "pip install -e .", "pre_install": null, "pip_packages": ["pytest"], "reqs_path": ["requirements.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 1}}, {"instance_id": "pypa__pip-11987", "repo": "pypa/pip", "base_commit": "87678eefec0ad4fc6083a5843e8080ad56e64196", "patch": "diff --git a/docs/html/topics/configuration.md b/docs/html/topics/configuration.md\\nindex 521bc9af4..e4aafcd2b 100644\\n--- a/docs/html/topics/configuration.md\\n+++ b/docs/html/topics/configuration.md\\n@@ -19,14 +19,13 @@ and how they are related to pip\'s various command line options.\\n \\n ## Configuration Files\\n \\n-Configuration files can change the default values for command line options.\\n-They are written using standard INI style configuration files.\\n+Configuration files can change the default values for command line option.\\n+They are written using a standard INI style configuration files.\\n \\n-pip has 4 \\"levels\\" of configuration files:\\n+pip has 3 \\"levels\\" of configuration files:\\n \\n-- `global`: system-wide configuration file, shared across all users.\\n-- `user`: per-user configuration file, shared across all environments.\\n-- `base` : per-base environment configuration file, shared across all virtualenvs with the same base. (available since pip 23.0)\\n+- `global`: system-wide configuration file, shared across users.\\n+- `user`: per-user configuration file.\\n - `site`: per-environment configuration file; i.e. per-virtualenv.\\n \\n ### Location\\n@@ -48,9 +47,6 @@ User\\n \\n   The legacy \\"per-user\\" configuration file is also loaded, if it exists: {file}`$HOME/.pip/pip.conf`.\\n \\n-Base\\n-: {file}`\\\\{sys.base_prefix\\\\}/pip.conf`\\n-\\n Site\\n : {file}`$VIRTUAL_ENV/pip.conf`\\n ```\\n@@ -67,9 +63,6 @@ User\\n \\n   The legacy \\"per-user\\" configuration file is also loaded, if it exists: {file}`$HOME/.pip/pip.conf`.\\n \\n-Base\\n-: {file}`\\\\{sys.base_prefix\\\\}/pip.conf`\\n-\\n Site\\n : {file}`$VIRTUAL_ENV/pip.conf`\\n ```\\n@@ -88,9 +81,6 @@ User\\n \\n   The legacy \\"per-user\\" configuration file is also loaded, if it exists: {file}`%HOME%\\\\\\\\pip\\\\\\\\pip.ini`\\n \\n-Base\\n-: {file}`\\\\{sys.base_prefix\\\\}\\\\\\\\pip.ini`\\n-\\n Site\\n : {file}`%VIRTUAL_ENV%\\\\\\\\pip.ini`\\n ```\\n@@ -112,7 +102,6 @@ order:\\n - `PIP_CONFIG_FILE`, if given.\\n - Global\\n - User\\n-- Base\\n - Site\\n \\n Each file read overrides any values read from previous files, so if the\\ndiff --git a/news/11987.bugfix.rst b/news/11987.bugfix.rst\\nnew file mode 100644\\nindex 000000000..de03de640\\n--- /dev/null\\n+++ b/news/11987.bugfix.rst\\n@@ -0,0 +1,1 @@\\n+Revert `#11487 <https://github.com/pypa/pip/pull/11487>`_, as it causes issues with virtualenvs created by the Windows Store distribution of Python.\\ndiff --git a/src/pip/_internal/configuration.py b/src/pip/_internal/configuration.py\\nindex 6cce8bcbc..8fd46c9b8 100644\\n--- a/src/pip/_internal/configuration.py\\n+++ b/src/pip/_internal/configuration.py\\n@@ -36,20 +36,12 @@ ENV_NAMES_IGNORED = \\"version\\", \\"help\\"\\n kinds = enum(\\n     USER=\\"user\\",  # User Specific\\n     GLOBAL=\\"global\\",  # System Wide\\n-    BASE=\\"base\\",  # Base environment specific (e.g. for all venvs with the same base)\\n-    SITE=\\"site\\",  # Environment Specific (e.g. per venv)\\n+    SITE=\\"site\\",  # [Virtual] Environment Specific\\n     ENV=\\"env\\",  # from PIP_CONFIG_FILE\\n     ENV_VAR=\\"env-var\\",  # from Environment Variables\\n )\\n-OVERRIDE_ORDER = (\\n-    kinds.GLOBAL,\\n-    kinds.USER,\\n-    kinds.BASE,\\n-    kinds.SITE,\\n-    kinds.ENV,\\n-    kinds.ENV_VAR,\\n-)\\n-VALID_LOAD_ONLY = kinds.USER, kinds.GLOBAL, kinds.BASE, kinds.SITE\\n+OVERRIDE_ORDER = kinds.GLOBAL, kinds.USER, kinds.SITE, kinds.ENV, kinds.ENV_VAR\\n+VALID_LOAD_ONLY = kinds.USER, kinds.GLOBAL, kinds.SITE\\n \\n logger = getLogger(__name__)\\n \\n@@ -78,7 +70,6 @@ def get_configuration_files() -> Dict[Kind, List[str]]:\\n         os.path.join(path, CONFIG_BASENAME) for path in appdirs.site_config_dirs(\\"pip\\")\\n     ]\\n \\n-    base_config_file = os.path.join(sys.base_prefix, CONFIG_BASENAME)\\n     site_config_file = os.path.join(sys.prefix, CONFIG_BASENAME)\\n     legacy_config_file = os.path.join(\\n         os.path.expanduser(\\"~\\"),\\n@@ -87,7 +78,6 @@ def get_configuration_files() -> Dict[Kind, List[str]]:\\n     )\\n     new_config_file = os.path.join(appdirs.user_config_dir(\\"pip\\"), CONFIG_BASENAME)\\n     return {\\n-        kinds.BASE: [base_config_file],\\n         kinds.GLOBAL: global_config_files,\\n         kinds.SITE: [site_config_file],\\n         kinds.USER: [legacy_config_file, new_config_file],\\n@@ -354,8 +344,6 @@ class Configuration:\\n             # The legacy config file is overridden by the new config file\\n             yield kinds.USER, config_files[kinds.USER]\\n \\n-        yield kinds.BASE, config_files[kinds.BASE]\\n-\\n         # finally virtualenv configuration first trumping others\\n         yield kinds.SITE, config_files[kinds.SITE]\\n \\n", "test_patch": "diff --git a/tests/unit/test_configuration.py b/tests/unit/test_configuration.py\\nindex b0d655d8f..c6b44d45a 100644\\n--- a/tests/unit/test_configuration.py\\n+++ b/tests/unit/test_configuration.py\\n@@ -24,17 +24,11 @@ class TestConfigurationLoading(ConfigurationMixin):\\n         self.configuration.load()\\n         assert self.configuration.get_value(\\"test.hello\\") == \\"2\\"\\n \\n-    def test_base_loading(self) -> None:\\n-        self.patch_configuration(kinds.BASE, {\\"test.hello\\": \\"3\\"})\\n-\\n-        self.configuration.load()\\n-        assert self.configuration.get_value(\\"test.hello\\") == \\"3\\"\\n-\\n     def test_site_loading(self) -> None:\\n-        self.patch_configuration(kinds.SITE, {\\"test.hello\\": \\"4\\"})\\n+        self.patch_configuration(kinds.SITE, {\\"test.hello\\": \\"3\\"})\\n \\n         self.configuration.load()\\n-        assert self.configuration.get_value(\\"test.hello\\") == \\"4\\"\\n+        assert self.configuration.get_value(\\"test.hello\\") == \\"3\\"\\n \\n     def test_environment_config_loading(self, monkeypatch: pytest.MonkeyPatch) -> None:\\n         contents = \\"\\"\\"\\n@@ -113,15 +107,6 @@ class TestConfigurationLoading(ConfigurationMixin):\\n         with pytest.raises(ConfigurationError, match=pat):\\n             self.configuration.get_value(\\"global.index-url\\")\\n \\n-    def test_overrides_normalization(self) -> None:\\n-        # Check that normalized names are used in precedence calculations.\\n-        # Reminder: USER has higher precedence than GLOBAL.\\n-        self.patch_configuration(kinds.USER, {\\"test.hello-world\\": \\"1\\"})\\n-        self.patch_configuration(kinds.GLOBAL, {\\"test.hello_world\\": \\"0\\"})\\n-        self.configuration.load()\\n-\\n-        assert self.configuration.get_value(\\"test.hello_world\\") == \\"1\\"\\n-\\n \\n class TestConfigurationPrecedence(ConfigurationMixin):\\n     # Tests for methods to that determine the order of precedence of\\n@@ -148,13 +133,6 @@ class TestConfigurationPrecedence(ConfigurationMixin):\\n \\n         assert self.configuration.get_value(\\"test.hello\\") == \\"0\\"\\n \\n-    def test_site_overides_base(self) -> None:\\n-        self.patch_configuration(kinds.BASE, {\\"test.hello\\": \\"2\\"})\\n-        self.patch_configuration(kinds.SITE, {\\"test.hello\\": \\"1\\"})\\n-        self.configuration.load()\\n-\\n-        assert self.configuration.get_value(\\"test.hello\\") == \\"1\\"\\n-\\n     def test_site_overides_user(self) -> None:\\n         self.patch_configuration(kinds.USER, {\\"test.hello\\": \\"2\\"})\\n         self.patch_configuration(kinds.SITE, {\\"test.hello\\": \\"1\\"})\\n@@ -169,13 +147,6 @@ class TestConfigurationPrecedence(ConfigurationMixin):\\n \\n         assert self.configuration.get_value(\\"test.hello\\") == \\"1\\"\\n \\n-    def test_base_overides_user(self) -> None:\\n-        self.patch_configuration(kinds.USER, {\\"test.hello\\": \\"2\\"})\\n-        self.patch_configuration(kinds.BASE, {\\"test.hello\\": \\"1\\"})\\n-        self.configuration.load()\\n-\\n-        assert self.configuration.get_value(\\"test.hello\\") == \\"1\\"\\n-\\n     def test_user_overides_global(self) -> None:\\n         self.patch_configuration(kinds.GLOBAL, {\\"test.hello\\": \\"3\\"})\\n         self.patch_configuration(kinds.USER, {\\"test.hello\\": \\"2\\"})\\ndiff --git a/tests/unit/test_options.py b/tests/unit/test_options.py\\nindex 9e3a0a5d6..43d5fdd3d 100644\\n--- a/tests/unit/test_options.py\\n+++ b/tests/unit/test_options.py\\n@@ -587,7 +587,7 @@ class TestOptionsConfigFiles:\\n         for _, val in cp.iter_config_files():\\n             files.extend(val)\\n \\n-        assert len(files) == 5\\n+        assert len(files) == 4\\n \\n     @pytest.mark.parametrize(\\n         \\"args, expect\\",\\n", "problem_statement": "Unexpected --user behavior since version 23.1\\n### Description\\n\\nHey everyone,\\r\\nI\'m using pip typically in virtual environments (managed by PyCharm). Since version 23.1 I get an error message when installing packages.\\r\\n\\r\\n```\\r\\n(venv) C:\\\\Users\\\\florian.wiese\\\\PycharmProjects\\\\GUIManager>pip --version\\r\\npip 23.1 from c:\\\\users\\\\florian.wiese\\\\pycharmprojects\\\\guimanager\\\\venv\\\\lib\\\\site-packages\\\\pip (python 3.9)\\r\\n\\r\\n(venv) C:\\\\Users\\\\florian.wiese\\\\PycharmProjects\\\\GUIManager>pip install numpy\\r\\nERROR: Can not perform a \'--user\' install. User site-packages are not visible in this virtualenv.\\r\\n```\\r\\n\\r\\nA little research told me, that a --user flag within a venv does not really make sense. However, it seems that some kind of default --user is set since 23.1.\\r\\n\\r\\nA workaround I found is to add the --no-user flag.   \\r\\n\\r\\n```\\r\\n(venv) C:\\\\Users\\\\florian.wiese\\\\PycharmProjects\\\\GUIManager>pip install numpy --no-user\\r\\nCollecting numpy\\r\\n   Using cached numpy-1.24.2-cp39-cp39-win_amd64.whl (14.9 MB)\\r\\nInstalling collected packages: numpy\\r\\nSuccessfully installed numpy-1.24.2\\r\\n```  \\r\\n\\r\\nFor completeness, downgrading to 23.0.1 with the --no-user flag gives the me the behavior I\'m used to.\\r\\n \\r\\n```\\r\\n(venv) C:\\\\Users\\\\florian.wiese\\\\PycharmProjects\\\\GUIManager>pip --version\\r\\npip 23.1 from c:\\\\users\\\\florian.wiese\\\\pycharmprojects\\\\guimanager\\\\venv\\\\lib\\\\site-packages\\\\pip (python 3.9)\\r\\n\\r\\n(venv) C:\\\\Users\\\\florian.wiese\\\\PycharmProjects\\\\GUIManager>python.exe -m pip install --upgrade pip==23.0.1 --no-user\\r\\nCollecting pip==23.0.1\\r\\n  Using cached pip-23.0.1-py3-none-any.whl (2.1 MB)\\r\\nInstalling collected packages: pip\\r\\n  Attempting uninstall: pip\\r\\n    Found existing installation: pip 23.1\\r\\n    Uninstalling pip-23.1:\\r\\n      Successfully uninstalled pip-23.1\\r\\nSuccessfully installed pip-23.0.1\\r\\n\\r\\n(venv) C:\\\\Users\\\\florian.wiese\\\\PycharmProjects\\\\GUIManager>pip install numpy\\r\\nCollecting numpy\\r\\n  Using cached numpy-1.24.2-cp39-cp39-win_amd64.whl (14.9 MB)\\r\\nInstalling collected packages: numpy\\r\\nSuccessfully installed numpy-1.24.2\\r\\n\\r\\n[notice] A new release of pip is available: 23.0.1 -> 23.1\\r\\n[notice] To update, run: python.exe -m pip install --upgrade pip\\r\\n```\\r\\n  \\r\\nNot sure if this is a bug or a feature. Maybe I\'m also missing something very obvious. But at least it is documented now for anyone experiencing the same issue :)\\n\\n### Expected behavior\\n\\n_No response_\\n\\n### pip version\\n\\n23.1\\n\\n### Python version\\n\\n3.9\\n\\n### OS\\n\\nWindows 10\\n\\n### How to Reproduce\\n\\ncreate a venv with pip 23.1\\r\\ntry to install a package with `pip install <any-package-name>`\\n\\n### Output\\n\\n`ERROR: Can not perform a \'--user\' install. User site-packages are not visible in this virtualenv.`\\n\\n### Code of Conduct\\n\\n- [X] I agree to follow the [PSF Code of Conduct](https://www.python.org/psf/conduct/).", "hints_text": "", "created_at": "2023-04-21 16:12:36", "FAIL_TO_PASS": ["tests/unit/test_options.py::TestOptionsConfigFiles::test_venv_config_file_found"], "PASS_TO_PASS": ["tests/unit/test_configuration.py::TestConfigurationLoading::test_global_loading", "tests/unit/test_configuration.py::TestConfigurationLoading::test_user_loading", "tests/unit/test_configuration.py::TestConfigurationLoading::test_site_loading", "tests/unit/test_configuration.py::TestConfigurationLoading::test_environment_config_loading", "tests/unit/test_configuration.py::TestConfigurationLoading::test_environment_var_loading", "tests/unit/test_configuration.py::TestConfigurationLoading::test_environment_var_does_not_load_lowercase", "tests/unit/test_configuration.py::TestConfigurationLoading::test_environment_var_does_not_load_version", "tests/unit/test_configuration.py::TestConfigurationLoading::test_environment_config_errors_if_malformed", "tests/unit/test_configuration.py::TestConfigurationLoading::test_no_such_key_error_message_no_command", "tests/unit/test_configuration.py::TestConfigurationLoading::test_no_such_key_error_message_missing_option", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_env_overides_site", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_env_overides_user", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_env_overides_global", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_site_overides_user", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_site_overides_global", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_user_overides_global", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_env_not_overriden_by_environment_var", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_site_not_overriden_by_environment_var", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_user_not_overriden_by_environment_var", "tests/unit/test_configuration.py::TestConfigurationPrecedence::test_global_not_overriden_by_environment_var", "tests/unit/test_configuration.py::TestConfigurationModification::test_no_specific_given_modification", "tests/unit/test_configuration.py::TestConfigurationModification::test_site_modification", "tests/unit/test_configuration.py::TestConfigurationModification::test_user_modification", "tests/unit/test_configuration.py::TestConfigurationModification::test_global_modification", "tests/unit/test_configuration.py::TestConfigurationModification::test_normalization", "tests/unit/test_options.py::TestOptionPrecedence::test_env_override_default_int", "tests/unit/test_options.py::TestOptionPrecedence::test_env_override_default_append[values0]", "tests/unit/test_options.py::TestOptionPrecedence::test_env_override_default_append[values1]", "tests/unit/test_options.py::TestOptionPrecedence::test_env_override_default_choice[choices0]", "tests/unit/test_options.py::TestOptionPrecedence::test_env_override_default_choice[choices1]", "tests/unit/test_options.py::TestOptionPrecedence::test_env_alias_override_default[PIP_LOG_FILE]", "tests/unit/test_options.py::TestOptionPrecedence::test_env_alias_override_default[PIP_LOCAL_LOG]", "tests/unit/test_options.py::TestOptionPrecedence::test_cli_override_environment", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR[1]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR[true]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR[on]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR[yes]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR[0]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR[false]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR[off]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR[no]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR__with_cache_dir[yes]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR__with_cache_dir[no]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR__with_no_cache_dir[yes]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR__with_no_cache_dir[no]", "tests/unit/test_options.py::TestOptionPrecedence::test_cache_dir__PIP_NO_CACHE_DIR_invalid__with_no_cache_dir", "tests/unit/test_options.py::TestUsePEP517Options::test_no_option", "tests/unit/test_options.py::TestUsePEP517Options::test_use_pep517", "tests/unit/test_options.py::TestUsePEP517Options::test_no_use_pep517", "tests/unit/test_options.py::TestUsePEP517Options::test_PIP_USE_PEP517_true", "tests/unit/test_options.py::TestUsePEP517Options::test_PIP_USE_PEP517_false", "tests/unit/test_options.py::TestUsePEP517Options::test_use_pep517_and_PIP_USE_PEP517_false", "tests/unit/test_options.py::TestUsePEP517Options::test_no_use_pep517_and_PIP_USE_PEP517_true", "tests/unit/test_options.py::TestUsePEP517Options::test_PIP_NO_USE_PEP517", "tests/unit/test_options.py::TestOptionsInterspersed::test_general_option_after_subcommand", "tests/unit/test_options.py::TestOptionsInterspersed::test_option_after_subcommand_arg", "tests/unit/test_options.py::TestOptionsInterspersed::test_additive_before_after_subcommand", "tests/unit/test_options.py::TestOptionsInterspersed::test_subcommand_option_before_subcommand_fails", "tests/unit/test_options.py::TestCountOptions::test_cli_long[0-verbose]", "tests/unit/test_options.py::TestCountOptions::test_cli_long[0-quiet]", "tests/unit/test_options.py::TestCountOptions::test_cli_long[1-verbose]", "tests/unit/test_options.py::TestCountOptions::test_cli_long[1-quiet]", "tests/unit/test_options.py::TestCountOptions::test_cli_long[2-verbose]", "tests/unit/test_options.py::TestCountOptions::test_cli_long[2-quiet]", "tests/unit/test_options.py::TestCountOptions::test_cli_long[3-verbose]", "tests/unit/test_options.py::TestCountOptions::test_cli_long[3-quiet]", "tests/unit/test_options.py::TestCountOptions::test_cli_short[1-verbose]", "tests/unit/test_options.py::TestCountOptions::test_cli_short[1-quiet]", "tests/unit/test_options.py::TestCountOptions::test_cli_short[2-verbose]", "tests/unit/test_options.py::TestCountOptions::test_cli_short[2-quiet]", "tests/unit/test_options.py::TestCountOptions::test_cli_short[3-verbose]", "tests/unit/test_options.py::TestCountOptions::test_cli_short[3-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var[0-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var[0-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var[1-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var[1-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var[2-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var[2-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var[3-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var[3-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var_integrate_cli[0-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var_integrate_cli[0-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var_integrate_cli[1-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var_integrate_cli[1-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var_integrate_cli[2-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var_integrate_cli[2-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var_invalid[-1-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var_invalid[-1-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var_invalid[foobar-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var_invalid[foobar-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var_false[no-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var_false[no-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var_false[false-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var_false[false-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var_true[yes-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var_true[yes-quiet]", "tests/unit/test_options.py::TestCountOptions::test_env_var_true[true-verbose]", "tests/unit/test_options.py::TestCountOptions::test_env_var_true[true-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file[0-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file[0-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file[1-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file[1-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file[2-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file[2-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file[3-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file[3-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file_integrate_cli[0-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file_integrate_cli[0-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file_integrate_cli[1-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file_integrate_cli[1-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file_integrate_cli[2-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file_integrate_cli[2-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file_invalid[-1-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file_invalid[-1-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file_invalid[foobar-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file_invalid[foobar-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file_false[no-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file_false[no-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file_false[false-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file_false[false-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file_true[yes-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file_true[yes-quiet]", "tests/unit/test_options.py::TestCountOptions::test_config_file_true[true-verbose]", "tests/unit/test_options.py::TestCountOptions::test_config_file_true[true-quiet]", "tests/unit/test_options.py::TestGeneralOptions::test_cache_dir__default", "tests/unit/test_options.py::TestGeneralOptions::test_cache_dir__provided", "tests/unit/test_options.py::TestGeneralOptions::test_no_cache_dir__provided", "tests/unit/test_options.py::TestGeneralOptions::test_require_virtualenv", "tests/unit/test_options.py::TestGeneralOptions::test_log", "tests/unit/test_options.py::TestGeneralOptions::test_local_log", "tests/unit/test_options.py::TestGeneralOptions::test_no_input", "tests/unit/test_options.py::TestGeneralOptions::test_proxy", "tests/unit/test_options.py::TestGeneralOptions::test_retries", "tests/unit/test_options.py::TestGeneralOptions::test_timeout", "tests/unit/test_options.py::TestGeneralOptions::test_exists_action", "tests/unit/test_options.py::TestGeneralOptions::test_cert", "tests/unit/test_options.py::TestGeneralOptions::test_client_cert", "tests/unit/test_options.py::TestOptionsConfigFiles::test_config_file_options[args0-None]", "tests/unit/test_options.py::TestOptionsConfigFiles::test_config_file_options[args1-global]", "tests/unit/test_options.py::TestOptionsConfigFiles::test_config_file_options[args2-site]", "tests/unit/test_options.py::TestOptionsConfigFiles::test_config_file_options[args3-user]", "tests/unit/test_options.py::TestOptionsConfigFiles::test_config_file_options[args4-PipError]", "tests/unit/test_options.py::TestOptionsConfigFiles::test_config_file_options[args5-PipError]", "tests/unit/test_options.py::TestOptionsConfigFiles::test_config_file_options[args6-PipError]", "tests/unit/test_options.py::TestOptionsExpandUser::test_cache_dir", "tests/unit/test_options.py::TestOptionsExpandUser::test_log", "tests/unit/test_options.py::TestOptionsExpandUser::test_local_log", "tests/unit/test_options.py::TestOptionsExpandUser::test_cert", "tests/unit/test_options.py::TestOptionsExpandUser::test_client_cert"], "environment_setup_commit": "b88addeaf5fde848733d7ef631caa522eb1cfa53", "version": "23.1", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .", "pre_install": null, "pip_packages": ["pytest"], "reqs_path": ["tests/requirements.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 2}}, {"instance_id": "sqlfluff__sqlfluff-5317", "repo": "sqlfluff/sqlfluff", "base_commit": "b36755d2a3ef62261acaed0012b366bffa08cb42", "patch": "diff --git a/src/sqlfluff/core/templaters/jinja.py b/src/sqlfluff/core/templaters/jinja.py\\nindex c291c9a22..eb37dd0d4 100644\\n--- a/src/sqlfluff/core/templaters/jinja.py\\n+++ b/src/sqlfluff/core/templaters/jinja.py\\n@@ -20,7 +20,7 @@ from jinja2.ext import Extension\\n from jinja2.sandbox import SandboxedEnvironment\\n \\n from sqlfluff.core.config import FluffConfig\\n-from sqlfluff.core.errors import SQLBaseError, SQLTemplaterError\\n+from sqlfluff.core.errors import SQLBaseError, SQLFluffUserError, SQLTemplaterError\\n from sqlfluff.core.templaters.base import (\\n     RawFileSlice,\\n     TemplatedFile,\\n@@ -52,12 +52,20 @@ class JinjaTemplater(PythonTemplater):\\n         \\"\\"\\"Take a template string and extract any macros from it.\\n \\n         Lovingly inspired by http://codyaray.com/2015/05/auto-load-jinja2-macros\\n+\\n+        Raises:\\n+            TemplateSyntaxError: If the macro we try to load has invalid\\n+                syntax. We assume that outer functions will catch this\\n+                exception and handle it appropriately.\\n         \\"\\"\\"\\n         from jinja2.runtime import Macro  # noqa\\n \\n         # Iterate through keys exported from the loaded template string\\n         context = {}\\n+        # NOTE: `env.from_string()` will raise TemplateSyntaxError if `template`\\n+        # is invalid.\\n         macro_template = env.from_string(template, globals=ctx)\\n+\\n         # This is kind of low level and hacky but it works\\n         try:\\n             for k in macro_template.module.__dict__:\\n@@ -147,9 +155,14 @@ class JinjaTemplater(PythonTemplater):\\n         # Iterate to load macros\\n         macro_ctx = {}\\n         for value in loaded_context.values():\\n-            macro_ctx.update(\\n-                self._extract_macros_from_template(value, env=env, ctx=ctx)\\n-            )\\n+            try:\\n+                macro_ctx.update(\\n+                    self._extract_macros_from_template(value, env=env, ctx=ctx)\\n+                )\\n+            except TemplateSyntaxError as err:\\n+                raise SQLFluffUserError(\\n+                    f\\"Error loading user provided macro:\\\\n`{value}`\\\\n> {err}.\\"\\n+                )\\n         return macro_ctx\\n \\n     def _extract_libraries_from_config(self, config):\\n", "test_patch": "diff --git a/test/core/templaters/jinja_test.py b/test/core/templaters/jinja_test.py\\nindex ee3f10c89..fff39c0f4 100644\\n--- a/test/core/templaters/jinja_test.py\\n+++ b/test/core/templaters/jinja_test.py\\n@@ -14,7 +14,7 @@ import pytest\\n from jinja2.exceptions import UndefinedError\\n \\n from sqlfluff.core import FluffConfig, Linter\\n-from sqlfluff.core.errors import SQLFluffSkipFile, SQLTemplaterError\\n+from sqlfluff.core.errors import SQLFluffSkipFile, SQLFluffUserError, SQLTemplaterError\\n from sqlfluff.core.templaters import JinjaTemplater\\n from sqlfluff.core.templaters.base import RawFileSlice, TemplatedFile\\n from sqlfluff.core.templaters.jinja import DummyUndefined, JinjaAnalyzer\\n@@ -621,6 +621,23 @@ def test__templater_jinja_error_macro_path_does_not_exist():\\n     assert str(e.value).startswith(\\"Path does not exist\\")\\n \\n \\n+def test__templater_jinja_error_macro_invalid():\\n+    \\"\\"\\"Tests that an error is raised if a macro is invalid.\\"\\"\\"\\n+    invalid_macro_config_string = (\\n+        \\"[sqlfluff]\\\\n\\"\\n+        \\"templater = jinja\\\\n\\"\\n+        \\"dialect = ansi\\\\n\\"\\n+        \\"[sqlfluff:templater:jinja:macros]\\\\n\\"\\n+        \\"a_macro_def = {% macro pkg.my_macro() %}pass{% endmacro %}\\\\n\\"\\n+    )\\n+    config = FluffConfig.from_string(invalid_macro_config_string)\\n+    with pytest.raises(SQLFluffUserError) as e:\\n+        JinjaTemplater().construct_render_func(config=config)\\n+    error_string = str(e.value)\\n+    assert error_string.startswith(\\"Error loading user provided macro\\")\\n+    assert \\"{% macro pkg.my_macro() %}pass{% endmacro %}\\" in error_string\\n+\\n+\\n def test__templater_jinja_lint_empty():\\n     \\"\\"\\"Check that parsing a file which renders to an empty string.\\n \\n", "problem_statement": "Catch uncaught exception jinja2.exceptions.TemplateSyntaxError\\n### Search before asking\\r\\n\\r\\n- [X] I searched the [issues](https://github.com/sqlfluff/sqlfluff/issues) and found no similar issues.\\r\\n\\r\\n\\r\\n### What Happened\\r\\n\\r\\nLargely this is similar to https://github.com/sqlfluff/sqlfluff/issues/3199 where there is an uncaught exception which is crashing sqlfluff. Because sqlfluff crashed - instead of say reporting a templating error instead - the dbt Cloud IDE does not return any errors to the end user (dbt Cloud does not return Python stack traces).\\r\\n\\r\\n### Expected Behaviour\\r\\n\\r\\nHave such errors be caught and reported as templating errors instead so that sqlfluff doesn\'t crash :)\\r\\n\\r\\n### Observed Behaviour\\r\\n\\r\\nsqlfluff runs into an uncaught exception.\\r\\n\\r\\n### How to reproduce\\r\\n\\r\\nAdd the following files to a folder:\\r\\n\\r\\n```yaml\\r\\n# dbt_project.yml\\r\\nname: my_dbt_project\\r\\nprofile: all\\r\\nconfig-version: 2\\r\\nversion: 1.0\\r\\n\\r\\nmodels:\\r\\n  my_dbt_project:\\r\\n    +materialized: table\\r\\n```\\r\\n\\r\\n```sql\\r\\n-- models/foo.sql\\r\\nselect 1 id\\r\\n```\\r\\n\\r\\nThen lint:\\r\\n\\r\\n```sh\\r\\n$ sqlfluff lint models/foo.sql\\r\\n\\r\\nTraceback (most recent call last):\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/bin/sqlfluff\\", line 8, in <module>\\r\\n    sys.exit(cli())\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/click/core.py\\", line 1157, in __call__\\r\\n    return self.main(*args, **kwargs)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/click/core.py\\", line 1078, in main\\r\\n    rv = self.invoke(ctx)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/click/core.py\\", line 1688, in invoke\\r\\n    return _process_result(sub_ctx.command.invoke(sub_ctx))\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/click/core.py\\", line 1434, in invoke\\r\\n    return ctx.invoke(self.callback, **ctx.params)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/click/core.py\\", line 783, in invoke\\r\\n    return __callback(*args, **kwargs)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/cli/commands.py\\", line 635, in lint\\r\\n    result = lnt.lint_paths(\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/linter/linter.py\\", line 1070, in lint_paths\\r\\n    for i, linted_file in enumerate(runner.run(expanded_paths, fix), start=1):\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/linter/runner.py\\", line 107, in run\\r\\n    for fname, partial in self.iter_partials(fnames, fix=fix):\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/linter/runner.py\\", line 60, in iter_partials\\r\\n    for fname, rendered in self.iter_rendered(fnames):\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/linter/runner.py\\", line 47, in iter_rendered\\r\\n    yield fname, self.linter.render_file(fname, self.config)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/linter/linter.py\\", line 739, in render_file\\r\\n    return self.render_string(raw_file, fname, config, encoding)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/linter/linter.py\\", line 710, in render_string\\r\\n    templated_file, templater_violations = self.templater.process(\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/templaters/base.py\\", line 54, in _wrapped\\r\\n    return func(self, in_str=in_str, fname=fname, config=config, **kwargs)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/templaters/jinja.py\\", line 410, in process\\r\\n    env, live_context, render_func = self.construct_render_func(\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/templaters/jinja.py\\", line 352, in construct_render_func\\r\\n    live_context = self.get_context(fname=fname, config=config, env=env)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/templaters/jinja.py\\", line 339, in get_context\\r\\n    self._extract_macros_from_config(\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/templaters/jinja.py\\", line 129, in _extract_macros_from_config\\r\\n    self._extract_macros_from_template(value, env=env, ctx=ctx)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/sqlfluff/core/templaters/jinja.py\\", line 60, in _extract_macros_from_template\\r\\n    macro_template = env.from_string(template, globals=ctx)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/jinja2/environment.py\\", line 1105, in from_string\\r\\n    return cls.from_code(self, self.compile(source), gs, None)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/jinja2/environment.py\\", line 768, in compile\\r\\n    self.handle_exception(source=source_hint)\\r\\n  File \\"/Users/jeremy/src/dbt-basic/venv_dbt_1.6.latest/lib/python3.9/site-packages/jinja2/environment.py\\", line 936, in handle_exception\\r\\n    raise rewrite_traceback_stack(source=source)\\r\\n  File \\"<unknown>\\", line 1, in template\\r\\njinja2.exceptions.TemplateSyntaxError: expected token \'(\', got \'.\'\\r\\n```\\r\\n\\r\\n### Dialect\\r\\n\\r\\nsnowflake\\r\\n\\r\\n### Version\\r\\n\\r\\n```\\r\\nsqlfluff, version 2.3.2\\r\\n```\\r\\n\\r\\n### Configuration\\r\\n\\r\\n```\\r\\n# .sqlfluff\\r\\n[sqlfluff]\\r\\ntemplater = jinja\\r\\ndialect = snowflake\\r\\n\\r\\n[sqlfluff:templater:jinja:macros]\\r\\na_macro_def = {% macro pkg.my_macro() %}pass{% endmacro %}\\r\\n```\\r\\n\\r\\n### Are you willing to work on and submit a PR to address the issue?\\r\\n\\r\\n- [ ] Yes I am willing to submit a PR!\\r\\n\\r\\n### Code of Conduct\\r\\n\\r\\n- [X] I agree to follow this project\'s [Code of Conduct](https://github.com/sqlfluff/sqlfluff/blob/main/CODE_OF_CONDUCT.md)\\r\\n", "hints_text": "github-actions[bot]: # Coverage Results \\u26a0\\ufe0f\\n```\\nName                                    Stmts   Miss  Cover   Missing\\n---------------------------------------------------------------------\\nsrc/sqlfluff/core/templaters/jinja.py     274      2    99%   116, 529\\n---------------------------------------------------------------------\\nTOTAL                                   16733      2    99%\\n\\n218 files skipped due to complete coverage.\\nCoverage failure: total of 99 is less than fail-under=100\\n\\n```\\n\\ncoveralls: \\n[![Coverage Status](https://coveralls.io/builds/63272410/badge)](https://coveralls.io/builds/63272410)\\n\\ncoverage: 99.989% (-0.01%) from 100.0% when pulling **009f32ccb03fc8905e3f819f2346c103d9bce268 on ac/jinja** into **b36755d2a3ef62261acaed0012b366bffa08cb42 on main**.\\n", "created_at": "2023-10-12 21:58:50", "FAIL_TO_PASS": ["test/core/templaters/jinja_test.py::test__templater_jinja_error_macro_invalid"], "PASS_TO_PASS": ["test/core/templaters/jinja_test.py::test__templater_jinja[simple]", "test/core/templaters/jinja_test.py::test__templater_jinja[unboundlocal_bugfix]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[basic_block]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[strip_left_block]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[strip_both_block]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[strip_and_templated_whitespace]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[strip_both_block_hard]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[basic_data]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[strip_right_data]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[strip_both_data]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[strip_both_comment]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[union_all_loop1]", "test/core/templaters/jinja_test.py::test__templater_jinja_slices[set_multiple_variables_and_define_macro]", "test/core/templaters/jinja_test.py::test_templater_set_block_handling", "test/core/templaters/jinja_test.py::test__templater_jinja_error_variable", "test/core/templaters/jinja_test.py::test__templater_jinja_dynamic_variable_no_violations", "test/core/templaters/jinja_test.py::test__templater_jinja_error_syntax", "test/core/templaters/jinja_test.py::test__templater_jinja_error_catastrophic", "test/core/templaters/jinja_test.py::test__templater_jinja_error_macro_path_does_not_exist", "test/core/templaters/jinja_test.py::test__templater_jinja_lint_empty", "test/core/templaters/jinja_test.py::test__templater_full[jinja_a/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_b/jinja-False-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_c_dbt/dbt_builtins_config-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_c_dbt/dbt_builtins_is_incremental-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_c_dbt/dbt_builtins_ref-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_c_dbt/dbt_builtins_source-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_c_dbt/dbt_builtins_this-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_c_dbt/dbt_builtins_var_default-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_c_dbt/dbt_builtins_test-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_e/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_f/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_g_macros/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_h_macros/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_i_raw/raw_tag-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_i_raw/raw_tag_2-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_j_libraries/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_k_config_override_path_macros/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/001-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/002-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/003-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/004-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/005-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/006-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/007-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/008-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/009-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/010-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_l_metas/011-False-True]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_m_libraries_module/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_n_nested_macros/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_o_config_override_dbt_builtins/override_dbt_builtins-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_p_disable_dbt_builtins/disable_dbt_builtins-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_q_multiple_path_macros/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_full[jinja_s_filters_in_library/jinja-True-False]", "test/core/templaters/jinja_test.py::test__templater_jinja_block_matching", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_template[-result0]", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_template[foo-result1]", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_template[foo", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_template[SELECT", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_template[{%", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_file[-None-result0]", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_file[foo-None-result1]", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_file[SELECT", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_file[{{", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_file[SELECT\\\\n", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_file[{%", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_file[{%-", "test/core/templaters/jinja_test.py::test__templater_jinja_slice_file[select\\\\n", "test/core/templaters/jinja_test.py::test__templater_jinja_large_file_check", "test/core/templaters/jinja_test.py::test_jinja_undefined_callable[-expected_violation0]", "test/core/templaters/jinja_test.py::test_jinja_undefined_callable[templating-None]", "test/core/templaters/jinja_test.py::test_dummy_undefined_fail_with_undefined_error", "test/core/templaters/jinja_test.py::test_undefined_magic_methods"], "environment_setup_commit": "24c7ed2cee1372f80d92d52c59d5d1515660981f", "version": "2.3", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .", "pre_install": null, "pip_packages": ["pytest"], "reqs_path": ["requirements.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 0, "files": 1}}, {"instance_id": "tianocore__edk2-pytool-extensions-41", "repo": "tianocore/edk2-pytool-extensions", "base_commit": "6df5a46d4827645afc87967ad9216cf698e70648", "patch": "diff --git a/edk2toolext/environment/extdeptypes/web_dependency.py b/edk2toolext/environment/extdeptypes/web_dependency.py\\nindex fe6d7d7..b3af6cb 100644\\n--- a/edk2toolext/environment/extdeptypes/web_dependency.py\\n+++ b/edk2toolext/environment/extdeptypes/web_dependency.py\\n@@ -142,6 +142,8 @@ def fetch(self):\\n         # If we just downloaded a file, we need to create a directory named self.contents_dir,\\n         # copy the file inside, and name it self.internal_path\\n         else:\\n+            os.makedirs(self.contents_dir, exist_ok=True)\\n+            complete_internal_path = os.path.join(self.contents_dir, self.internal_path)\\n             logging.info(f\\"Copying file to {complete_internal_path}\\")\\n             shutil.move(temp_file_path, complete_internal_path)\\n \\n", "test_patch": "diff --git a/edk2toolext/tests/test_web_dependency.py b/edk2toolext/tests/test_web_dependency.py\\nindex 3798161..1692837 100644\\n--- a/edk2toolext/tests/test_web_dependency.py\\n+++ b/edk2toolext/tests/test_web_dependency.py\\n@@ -1,4 +1,4 @@\\n-## @file test_web_dependency.py\\n+# @file test_web_dependency.py\\n # Unit test suite for the WebDependency class.\\n #\\n ##\\n@@ -14,6 +14,7 @@\\n import tarfile\\n import zipfile\\n import tempfile\\n+import json\\n import urllib.request\\n from edk2toolext.environment import environment_descriptor_files as EDF\\n from edk2toolext.environment.extdeptypes.web_dependency import WebDependency\\n@@ -32,6 +33,17 @@\\n   \\"sha256\\":\\"68f2335344c3f7689f8d69125d182404a3515b8daa53a9c330f115739889f998\\"\\n }\\n \'\'\'\\n+# JSON file that describes a single file to download from the internet\\n+# bing.com was choosen as it\'s probably not going anywhere soon and it\'s small file to download\\n+single_file_extdep = {\\n+    \\"scope\\": \\"global\\",\\n+    \\"type\\": \\"web\\",\\n+    \\"name\\": \\"test\\",\\n+    \\"source\\": \\"https://www.bing.com/\\",\\n+    \\"version\\": \\"20190805\\",\\n+    \\"flags\\": [],\\n+    \\"internal_path\\": \\"test.txt\\"\\n+}\\n \\n \\n def prep_workspace():\\n@@ -81,6 +93,21 @@ def test_fail_with_bad_url(self):\\n             ext_dep.fetch()\\n             self.fail(\\"should have thrown an Exception\\")\\n \\n+    # try to download a single file from the internet\\n+    def test_single_file(self):\\n+        ext_dep_file_path = os.path.join(test_dir, \\"good_ext_dep.json\\")\\n+        with open(ext_dep_file_path, \\"w+\\") as ext_dep_file:\\n+            ext_dep_file.write(json.dumps(single_file_extdep))  # dump to a file\\n+\\n+        ext_dep_descriptor = EDF.ExternDepDescriptor(ext_dep_file_path).descriptor_contents\\n+        ext_dep = WebDependency(ext_dep_descriptor)\\n+        ext_dep.fetch()\\n+\\n+        ext_dep_name = single_file_extdep[\'name\'] + \\"_extdep\\"\\n+        file_path = os.path.join(test_dir, ext_dep_name, single_file_extdep[\'internal_path\'])\\n+        if not os.path.isfile(file_path):\\n+            self.fail(\\"The downloaded file isn\'t there\\")\\n+\\n     # Test that get_internal_path_root works the way we expect with a flat directory structure.\\n     # test_dir\\\\inner_dir - test_dir\\\\inner_dir should be the root.\\n     def test_get_internal_path_root_flat(self):\\n@@ -96,8 +123,8 @@ def test_get_internal_path_root_with_subfolders(self):\\n         first_level_dir_name = \\"first_dir\\"\\n         second_level_dir_name = \\"second_dir\\"\\n         inner_dir_path = os.path.join(outer_dir, first_level_dir_name)\\n-        self.assertEqual(WebDependency.get_internal_path_root(outer_dir,\\n-                         os.path.join(first_level_dir_name, second_level_dir_name)), inner_dir_path)\\n+        inner_second_dir_path = os.path.join(first_level_dir_name, second_level_dir_name)\\n+        self.assertEqual(WebDependency.get_internal_path_root(outer_dir, inner_second_dir_path), inner_dir_path)\\n \\n     # Test that a single file zipped is able to be processed by unpack.\\n     def test_unpack_zip_file(self):\\n", "problem_statement": "web dependency doesn\'t work on single files\\nwhen downloading single files (like a binary from github) it fails to create the ext_dep folder", "hints_text": "matthewfcarlson: Thanks to @out0xb2 for finding the fix for this", "created_at": "2019-08-30 18:16:01", "FAIL_TO_PASS": ["edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_single_file"], "PASS_TO_PASS": ["edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_fail_with_bad_url", "edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_get_internal_path_root_flat", "edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_get_internal_path_root_with_subfolders", "edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_multi_level_directory", "edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_tar_uses_linux_path_sep", "edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_unpack_tar_directory", "edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_unpack_tar_file", "edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_unpack_zip_directory", "edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_unpack_zip_file", "edk2toolext/tests/test_web_dependency.py::TestWebDependency::test_zip_uses_linux_path_sep"], "environment_setup_commit": "6df5a46d4827645afc87967ad9216cf698e70648", "version": "0.9", "license_name": "BSD-2-Clause-Patent", "install_config": {"python": "3.9", "install": "pip install -e .", "pre_install": null, "pip_packages": null, "reqs_path": ["requirements.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 0, "files": 1}}, {"instance_id": "srsudar__eg-88", "repo": "srsudar/eg", "base_commit": "72f507e689816d58b1664aeeea08e394f23f61a9", "patch": "diff --git a/eg/config.py b/eg/config.py\\nindex f448d28..661057a 100644\\n--- a/eg/config.py\\n+++ b/eg/config.py\\n@@ -173,12 +173,16 @@ def get_egrc_config(cli_egrc_path):\\n \\n     if config_path == \'\':\\n       # Try for the xdg config.\\n-      xdg_home_dir = os.getenv(\'XDG_CONFIG_HOME\')\\n-      if xdg_home_dir:\\n-        xdg_config_path = os.path.join(xdg_home_dir, \'eg\', \'egrc\')\\n-        xdg_config_path = get_expanded_path(xdg_config_path)\\n-        if os.path.isfile(xdg_config_path):\\n-          config_path = xdg_config_path\\n+      # The freedesktop.org spec says \'$HOME/.config\' should be used if the\\n+      # environment variable is not set or is empty. os.getenv() will be falsey\\n+      # if it is either not set (None) or an empty string.\\n+      default_xdg_dir = os.path.join(\'~\', \'.config\')\\n+      xdg_home_dir = os.getenv(\'XDG_CONFIG_HOME\') or default_xdg_dir\\n+\\n+      xdg_config_path = os.path.join(xdg_home_dir, \'eg\', \'egrc\')\\n+      xdg_config_path = get_expanded_path(xdg_config_path)\\n+      if os.path.isfile(xdg_config_path):\\n+        config_path = xdg_config_path\\n \\n     if config_path == \'\':\\n       # Fall back to our home directory.\\n", "test_patch": "diff --git a/test/config_test.py b/test/config_test.py\\nindex 4277230..c7ea4cb 100644\\n--- a/test/config_test.py\\n+++ b/test/config_test.py\\n@@ -200,6 +200,50 @@ def test_get_egrc_config_uses_home_dir_default_no_xdg_dir():\\n     )\\n \\n \\n+def test_get_egrc_config_uses_default_xdg_dir_if_env_var_not_set():\\n+    \\"\\"\\"\\n+    If the XDG environment variable isn\'t set, but the file exists at the\\n+    default XDG location, it should be used. Note that here we don\'t have\\n+    \'egrc\', not as a hidden file by default.\\n+\\n+    See: https://github.com/srsudar/eg/issues/87\\n+    \\"\\"\\"\\n+    expected = \'mock config from default position\'\\n+\\n+    _assert_about_get_egrc_config(\\n+        cli_path=None,\\n+        cli_config_exists=False,\\n+        winning_config_path=os.path.join(\\n+          \'~\', \'.config\', \'eg\', \'egrc_expanded\'),\\n+        home_config_exists=True,\\n+        expected_config=expected,\\n+        xdg_dir_variable=None,\\n+        xdg_config_exists=True,\\n+    )\\n+\\n+\\n+def test_get_egrc_config_uses_default_xdg_dir_if_env_var_empty():\\n+    \\"\\"\\"\\n+    If the XDG environment variable is empty, but the file exists at the default\\n+    XDG location, it should be used. Note that here we don\'t have \'egrc\', not as\\n+    a hidden file by default.\\n+\\n+    See: https://github.com/srsudar/eg/issues/87\\n+    \\"\\"\\"\\n+    expected = \'mock config from default position\'\\n+\\n+    _assert_about_get_egrc_config(\\n+        cli_path=None,\\n+        cli_config_exists=False,\\n+        winning_config_path=os.path.join(\\n+          \'~\', \'.config\', \'eg\', \'egrc_expanded\'),\\n+        home_config_exists=True,\\n+        expected_config=expected,\\n+        xdg_dir_variable=\'\',\\n+        xdg_config_exists=True,\\n+    )\\n+\\n+\\n def test_get_egrc_config_uses_home_dir_default_xdg_dir_file_not_present():\\n     \\"\\"\\"\\n     Fallback to the home directory config if XDG directory is set but the XDG\\n@@ -217,7 +261,7 @@ def test_get_egrc_config_uses_home_dir_default_xdg_dir_file_not_present():\\n         xdg_config_exists=False,\\n     )\\n \\n-def test_get_egrc_config_uses_xdg_dir_default_if_present():\\n+def test_get_egrc_config_uses_xdg_dir_default_if_both_present():\\n     \\"\\"\\"\\n     The XDG config should win over the home directory.\\n     \\"\\"\\"\\n@@ -278,8 +322,13 @@ def _assert_about_get_egrc_config(\\n         return cli_config_exists\\n       if file_name == os.path.join(\'~\', \'.egrc_expanded\'):\\n         return home_config_exists\\n-      if file_name == os.path.join(xdg_dir_variable, \'eg\', \'egrc_expanded\'):\\n-        # ${XDG_DIR_HOME}/eg/egrc\\n+      if (xdg_dir_variable and\\n+          file_name == os.path.join(xdg_dir_variable, \'eg\', \'egrc_expanded\')):\\n+        # ${XDG_CONFIG_HOME}/eg/egrc\\n+        return xdg_config_exists\\n+      if file_name == os.path.join(\'~\', \'.config\', \'eg\', \'egrc_expanded\'):\\n+        # This is the default location of XDG_CONFIG_HOME, which we may fall\\n+        # back to under some test cases.\\n         return xdg_config_exists\\n       return False\\n     mock_isfile.side_effect = isfile_side_effect\\n", "problem_statement": "No recognize config path: $HOME/.config/eg/egrc \\nVersion: 1.2.0\\r\\nInstalled from pip\\r\\n\\r\\nI follow the [information](https://github.com/srsudar/eg#configuration-and-extension) on README of the repository. For custom config file I create a file called `egrc` on the next path: `skabit/.config/eg`, but nothing happens. In this file I only add a custom directory with my examples:\\r\\n\\r\\n```\\r\\n[eg-config]\\r\\n# Lines starting with # are treated as comments\\r\\ncustom-dir = ~/docu/cheats\\r\\n```\\r\\n\\r\\n", "hints_text": "", "created_at": "2021-01-08 04:58:14", "FAIL_TO_PASS": ["test/config_test.py::test_get_egrc_config_uses_default_xdg_dir_if_env_var_not_set", "test/config_test.py::test_get_egrc_config_uses_default_xdg_dir_if_env_var_empty"], "PASS_TO_PASS": ["test/config_test.py::test_config_returns_egrc_values_if_present", "test/config_test.py::test_inform_if_paths_invalid_selectively_informs", "test/config_test.py::test_get_resolved_config_uses_custom_egrc_path", "test/config_test.py::test_get_egrc_config_reads_from_command_line", "test/config_test.py::test_get_egrc_config_uses_home_dir_default_no_xdg_dir", "test/config_test.py::test_get_egrc_config_uses_home_dir_default_xdg_dir_file_not_present", "test/config_test.py::test_get_egrc_config_uses_xdg_dir_default_if_both_present", "test/config_test.py::test_get_egrc_returns_empty_if_no_egrc", "test/config_test.py::test_get_resolved_config_calls_expand_paths", "test/config_test.py::test_get_resolved_config_prioritizes_cli", "test/config_test.py::test_get_resolved_config_defaults_to_egrc", "test/config_test.py::test_get_resolved_config_falls_back_to_defaults", "test/config_test.py::test_get_config_tuple_from_egrc_all_none_when_not_present", "test/config_test.py::test_get_config_tuple_from_egrc_when_present", "test/config_test.py::test_merge_color_configs_first_all_none", "test/config_test.py::test_merge_color_configs_take_all_first", "test/config_test.py::test_merge_color_configs_mixed", "test/config_test.py::test_default_color_config", "test/config_test.py::test_parse_bool_true_for_truthy_values", "test/config_test.py::test_parse_bool_false_for_non_truthy_values", "test/config_test.py::test_get_priority_first", "test/config_test.py::test_get_priority_second", "test/config_test.py::test_get_priority_third", "test/config_test.py::test_get_priority_respect_false", "test/config_test.py::test_parse_substitution_from_list_without_is_multiline", "test/config_test.py::test_parse_substitution_from_list_with_is_multiline", "test/config_test.py::test_parse_substitution_error_if_not_list", "test/config_test.py::test_parse_substitution_error_if_wrong_length", "test/config_test.py::test_parse_substitution_error_if_third_element_not_bool", "test/config_test.py::test_get_substitution_from_config_finds_single_substitution", "test/config_test.py::test_get_substitution_from_config_finds_multiple_substitutions"], "environment_setup_commit": "72f507e689816d58b1664aeeea08e394f23f61a9", "version": "1.2", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .[dev]", "pre_install": null, "pip_packages": ["pytest", "mock"], "reqs_path": ["requirements/base.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 0, "files": 1}}, {"instance_id": "GenericMappingTools__pygmt-1258", "repo": "GenericMappingTools/pygmt", "base_commit": "53b4e7416989e76be4ae2b19d352b076c4dd4fa1", "patch": "diff --git a/examples/gallery/3d_plots/scatter3d.py b/examples/gallery/3d_plots/scatter3d.py\\nindex 568c2de7..0c70581f 100644\\n--- a/examples/gallery/3d_plots/scatter3d.py\\n+++ b/examples/gallery/3d_plots/scatter3d.py\\n@@ -47,7 +47,7 @@ fig.plot3d(\\n     y=df.sepal_length,\\n     z=df.petal_length,\\n     # Vary each symbol size according to another feature (sepal width, scaled by 0.1)\\n-    sizes=0.1 * df.sepal_width,\\n+    size=0.1 * df.sepal_width,\\n     # Use 3D cubes (\\"u\\") as symbols, with size in centimeter units (\\"c\\")\\n     style=\\"uc\\",\\n     # Points colored by categorical number code\\ndiff --git a/pygmt/src/plot3d.py b/pygmt/src/plot3d.py\\nindex 85b4826c..e6944c8e 100644\\n--- a/pygmt/src/plot3d.py\\n+++ b/pygmt/src/plot3d.py\\n@@ -6,6 +6,7 @@ from pygmt.exceptions import GMTInvalidInput\\n from pygmt.helpers import (\\n     build_arg_string,\\n     data_kind,\\n+    deprecate_parameter,\\n     fmt_docstring,\\n     is_nonstr_iter,\\n     kwargs_to_strings,\\n@@ -43,8 +44,9 @@ from pygmt.helpers import (\\n     t=\\"transparency\\",\\n )\\n @kwargs_to_strings(R=\\"sequence\\", c=\\"sequence_comma\\", i=\\"sequence_comma\\", p=\\"sequence\\")\\n+@deprecate_parameter(\\"sizes\\", \\"size\\", \\"v0.4.0\\", remove_version=\\"v0.6.0\\")\\n def plot3d(\\n-    self, x=None, y=None, z=None, data=None, sizes=None, direction=None, **kwargs\\n+    self, x=None, y=None, z=None, data=None, size=None, direction=None, **kwargs\\n ):\\n     r\\"\\"\\"\\n     Plot lines, polygons, and symbols in 3-D.\\n@@ -80,8 +82,8 @@ def plot3d(\\n         Either a data file name or a 2d numpy array with the tabular data.\\n         Use parameter ``columns`` to choose which columns are x, y, z,\\n         color, and size, respectively.\\n-    sizes : 1d array\\n-        The sizes of the data points in units specified in ``style``.\\n+    size : 1d array\\n+        The size of the data points in units specified in ``style``.\\n         Only valid if using ``x``/``y``/``z``.\\n     direction : list of two 1d arrays\\n         If plotting vectors (using ``style=\'V\'`` or ``style=\'v\'``), then\\n@@ -179,12 +181,12 @@ def plot3d(\\n             )\\n         extra_arrays.append(kwargs[\\"G\\"])\\n         del kwargs[\\"G\\"]\\n-    if sizes is not None:\\n+    if size is not None:\\n         if kind != \\"vectors\\":\\n             raise GMTInvalidInput(\\n-                \\"Can\'t use arrays for sizes if data is matrix or file.\\"\\n+                \\"Can\'t use arrays for \'size\' if data is a matrix or a file.\\"\\n             )\\n-        extra_arrays.append(sizes)\\n+        extra_arrays.append(size)\\n \\n     for flag in [\\"I\\", \\"t\\"]:\\n         if flag in kwargs and is_nonstr_iter(kwargs[flag]):\\n", "test_patch": "diff --git a/pygmt/tests/test_plot3d.py b/pygmt/tests/test_plot3d.py\\nindex 492e95a9..05708f7f 100644\\n--- a/pygmt/tests/test_plot3d.py\\n+++ b/pygmt/tests/test_plot3d.py\\n@@ -110,7 +110,7 @@ def test_plot3d_fail_no_data(data, region):\\n \\n def test_plot3d_fail_color_size_intensity(data, region):\\n     \\"\\"\\"\\n-    Should raise an exception if array color, sizes and intensity are used with\\n+    Should raise an exception if array color, size and intensity are used with\\n     matrix.\\n     \\"\\"\\"\\n     fig = Figure()\\n@@ -118,7 +118,7 @@ def test_plot3d_fail_color_size_intensity(data, region):\\n     with pytest.raises(GMTInvalidInput):\\n         fig.plot3d(style=\\"c0.2c\\", color=data[:, 2], **kwargs)\\n     with pytest.raises(GMTInvalidInput):\\n-        fig.plot3d(style=\\"cc\\", sizes=data[:, 2], color=\\"red\\", **kwargs)\\n+        fig.plot3d(style=\\"cc\\", size=data[:, 2], color=\\"red\\", **kwargs)\\n     with pytest.raises(GMTInvalidInput):\\n         fig.plot3d(style=\\"cc\\", intensity=data[:, 2], color=\\"red\\", **kwargs)\\n \\n@@ -178,7 +178,7 @@ def test_plot3d_sizes(data, region):\\n         z=data[:, 2],\\n         zscale=5,\\n         perspective=[225, 30],\\n-        sizes=0.5 * data[:, 2],\\n+        size=0.5 * data[:, 2],\\n         region=region,\\n         projection=\\"X10c\\",\\n         # Using inches instead of cm because of upstream bug at\\n@@ -203,7 +203,7 @@ def test_plot3d_colors_sizes(data, region):\\n         zscale=5,\\n         perspective=[225, 30],\\n         color=data[:, 2],\\n-        sizes=0.5 * data[:, 2],\\n+        size=0.5 * data[:, 2],\\n         region=region,\\n         projection=\\"X6c\\",\\n         # Using inches instead of cm because of upstream bug at\\n@@ -231,7 +231,7 @@ def test_plot3d_colors_sizes_proj(data, region):\\n         projection=\\"M20c\\",\\n         frame=[\\"af\\", \\"zaf\\"],\\n         color=data[:, 2],\\n-        sizes=data[:, 2],\\n+        size=data[:, 2],\\n         # Using inches instead of cm because of upstream bug at\\n         # https://github.com/GenericMappingTools/gmt/issues/4386\\n         style=\\"ui\\",\\n@@ -343,7 +343,7 @@ def test_plot3d_sizes_colors_transparencies():\\n         frame=True,\\n         style=\\"uc\\",\\n         color=color,\\n-        sizes=size,\\n+        size=size,\\n         cmap=\\"gray\\",\\n         transparency=transparency,\\n     )\\n@@ -458,3 +458,30 @@ def test_plot3d_scalar_xyz():\\n         x=1.5, y=-1.5, z=1.5, style=\\"s1c\\", color=\\"blue\\", zscale=True, perspective=True\\n     )\\n     return fig\\n+\\n+\\n+@pytest.mark.mpl_image_compare(filename=\\"test_plot3d_sizes.png\\")\\n+def test_plot3d_deprecate_sizes_to_size(data, region):\\n+    \\"\\"\\"\\n+    Make sure that the old parameter \\"sizes\\" is supported and it reports an\\n+    warning.\\n+\\n+    Modified from the test_plot3d_sizes() test.\\n+    \\"\\"\\"\\n+    fig = Figure()\\n+    with pytest.warns(expected_warning=FutureWarning) as record:\\n+        fig.plot3d(\\n+            x=data[:, 0],\\n+            y=data[:, 1],\\n+            z=data[:, 2],\\n+            zscale=5,\\n+            perspective=[225, 30],\\n+            sizes=0.5 * data[:, 2],\\n+            region=region,\\n+            projection=\\"X10c\\",\\n+            style=\\"ui\\",\\n+            color=\\"blue\\",\\n+            frame=[\\"af\\", \\"zaf\\"],\\n+        )\\n+        assert len(record) == 1  # check that only one warning was raised\\n+    return fig\\n", "problem_statement": "Rename the sizes parameter in plot/plot3d to be size\\n**Description of the problem**\\r\\n\\r\\nBoth [plot](https://www.pygmt.org/latest/api/generated/pygmt.Figure.plot.html#pygmt.Figure.plot) and [plot3d](https://www.pygmt.org/latest/api/generated/pygmt.Figure.plot3d.html#pygmt.Figure.plot3d) have the `sizes` parameter that are used to plot symbols with different sizes. However, `size` is used in GMT documentation. Shall we rename this parameter to be `size`? If yes, I think we should do it as early as possible cause this parameter is often used in PyGMT.\\r\\n\\r\\nOriginally posted in https://github.com/GenericMappingTools/pygmt/issues/1076#issuecomment-804986728: \\r\\n> > like the `sizes` parameter (I dont\' understand why we add a **s** at the end) of the `plot`/`plot3d` method\\r\\n> \\r\\n> `plot` is one of the oldest wrappers in PyGMT, and the parameters are not carefully chosen. We had some discussions in the GMT Community meeting, and I think we agree that we will change `sizes` to `size`.\\r\\n\\r\\n----\\r\\n\\r\\nSee the format of the [**-S**](https://docs.generic-mapping-tools.org/latest/plot.html#s) option in GMT `plot`:\\r\\n\\r\\n> x y [ z ] [ size ] [ symbol-parameters ] [ symbol ]\\r\\n\\r\\nSee the format of the [**-S**](https://docs.generic-mapping-tools.org/latest/plot3d.html#s) option in GMT `plot3d`:\\r\\n\\r\\n> x y z [ w ] [ size ] [ symbol-parameters ] [ symbol ]\\r\\n\\r\\n\\r\\nGMT plot | PyGMT plot | GMT plot3d | PyGMT plot3d\\r\\n:---: | :---:|:---: | :---:\\r\\n*z* | `color` | *w* | `color`\\r\\n*size* | `sizes` | *size* | `sizes`\\r\\n*intensity* | `intensity` |*intensity* | `intensity` \\r\\n*symbol* | `symbol` (requested in #1076) | *symbol* | `symbol` (requested in #1076)\\r\\n\\r\\n\\r\\n---\\r\\n\\r\\nNow, `sizes` is used in the following files:\\r\\n- `src/plot.py`\\r\\n- `tests/test_plot.py`\\r\\n- `examples/tutorials/plot.py`\\r\\n- `examples/gallery/symbols/scatter.py`\\r\\n- `examples/gallery/symbols/points_categorical.py`\\r\\n- `src/plot3d.py`\\r\\n- `tests/test_plot3d.py`\\r\\n- `examples/gallery/3d_plots/scatter3d.py`\\r\\n", "hints_text": "", "created_at": "2021-05-04 20:04:14", "FAIL_TO_PASS": ["pygmt/tests/test_plot3d.py::test_plot3d_fail_color_size_intensity"], "PASS_TO_PASS": ["pygmt/tests/test_plot3d.py::test_plot3d_fail_no_data"], "environment_setup_commit": "7f37e1ca5b14a4064447304a5bf3936489cf933f", "version": "0.3", "license_name": "BSD 3-Clause \\"New\\" or \\"Revised\\" License", "install_config": {"python": "3.9", "install": "pip install -e .", "pre_install": null, "pip_packages": ["pytest"], "reqs_path": null, "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 0, "files": 2}}, {"instance_id": "wemake-services__wemake-python-styleguide-1694", "repo": "wemake-services/wemake-python-styleguide", "base_commit": "ece40b94d7649cc30a3e8a896513719d28099aed", "patch": "diff --git a/CHANGELOG.md b/CHANGELOG.md\\nindex ec4ed88a..5cf56332 100644\\n--- a/CHANGELOG.md\\n+++ b/CHANGELOG.md\\n@@ -46,6 +46,7 @@ Semantic versioning in our case means:\\n - Allowed yield statements in call method\\n - Allow to use `^` with `1`\\n - Fixes false positives in WPS513 and WPS323\\n+- Fixes false positive WPS426 if `lambda` in loop uses only its arguments\\n \\n ### Misc\\n \\ndiff --git a/wemake_python_styleguide/visitors/ast/loops.py b/wemake_python_styleguide/visitors/ast/loops.py\\nindex 298f0fd0..92a3c4ce 100644\\n--- a/wemake_python_styleguide/visitors/ast/loops.py\\n+++ b/wemake_python_styleguide/visitors/ast/loops.py\\n@@ -177,15 +177,17 @@ class WrongLoopVisitor(base.BaseNodeVisitor):\\n         self,\\n         node: Union[AnyLoop, AnyComprehension],\\n     ) -> None:\\n-        container_names = self._containers.get(type(node), ())\\n-        for container in container_names:\\n-            body = getattr(node, container, [])\\n-            if not isinstance(body, list):\\n-                body = [body]\\n-\\n-            for subnode in body:\\n-                if walk.is_contained(subnode, ast.Lambda):\\n-                    self.add_violation(LambdaInsideLoopViolation(node))\\n+        for lambda_node in walk.get_subnodes_by_type(node, ast.Lambda):\\n+            arguments = (\\n+                arg.arg for arg\\n+                in walk.get_subnodes_by_type(lambda_node, ast.arg)\\n+            )\\n+            body = (\\n+                subnode.id for subnode\\n+                in walk.get_subnodes_by_type(lambda_node.body, ast.Name)\\n+            )\\n+            if not all(symbol in arguments for symbol in body):\\n+                self.add_violation(LambdaInsideLoopViolation(node))\\n \\n     def _check_useless_continue(self, node: AnyLoop) -> None:\\n         nodes_at_line: DefaultDict[int, List[ast.AST]] = defaultdict(list)\\n", "test_patch": "diff --git a/tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py b/tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py\\nindex f00a135c..722f3325 100644\\n--- a/tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py\\n+++ b/tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py\\n@@ -17,14 +17,7 @@ nested_lambda_inside_for_loop = \\"\\"\\"\\n def wrapper():\\n     for index in range(10):\\n         if some:\\n-            print(lambda: index)\\n-\\"\\"\\"\\n-\\n-nested_argument_lambda_inside_for_loop = \\"\\"\\"\\n-def wrapper():\\n-    for index in range(10):\\n-        if some:\\n-            print(lambda index=inde: index)\\n+            print(lambda x: x + index)\\n \\"\\"\\"\\n \\n lambda_inside_while_loop = \\"\\"\\"\\n@@ -42,17 +35,17 @@ while True:\\n \\n lambda_inside_set = \\"\\"\\"\\n def wrapper():\\n-    return {lambda x=x: x for x in some()}\\n+    return {lambda: x for x in some()}\\n \\"\\"\\"\\n \\n lambda_inside_list = \\"\\"\\"\\n def wrapper():\\n-    return [lambda x: x for x in some()]\\n+    return [lambda: x for x in some()]\\n \\"\\"\\"\\n \\n lambda_inside_gen = \\"\\"\\"\\n def wrapper():\\n-    return (lambda x: x for x in some())\\n+    return (lambda: x for x in some())\\n \\"\\"\\"\\n \\n lambda_inside_dict_key = \\"\\"\\"\\n@@ -68,6 +61,12 @@ def wrapper():\\n # Correct:\\n \\n correct_lambda_inside_for = \\"\\"\\"\\n+def wrapper():\\n+    for index in range(10):\\n+        print(lambda x: x)\\n+\\"\\"\\"\\n+\\n+correct_lambda_inside_map = \\"\\"\\"\\n def wrapper():\\n     for some in map(lambda x: x * 2, items):\\n         ...\\n@@ -80,14 +79,20 @@ def wrapper():\\n \\n correct_lambda_inside_list = \\"\\"\\"\\n def wrapper():\\n-    return [x for x in some() if callback(lambda: x)]\\n+    return [x for x in some() if callback(lambda x: x)]\\n+\\"\\"\\"\\n+\\n+correct_lambda_nested_argument_for = \\"\\"\\"\\n+def wrapper():\\n+    for index in range(10):\\n+        if some:\\n+            print(lambda index=some: index)\\n \\"\\"\\"\\n \\n \\n @pytest.mark.parametrize(\'code\', [\\n     lambda_inside_for_loop,\\n     nested_lambda_inside_for_loop,\\n-    nested_argument_lambda_inside_for_loop,\\n     lambda_inside_while_loop,\\n     nested_lambda_inside_while_loop,\\n     lambda_inside_list,\\n@@ -114,8 +119,10 @@ def test_lambda_body(\\n \\n @pytest.mark.parametrize(\'code\', [\\n     correct_lambda_inside_for,\\n+    correct_lambda_inside_map,\\n     correct_lambda_inside_list,\\n     correct_lambda_inside_set,\\n+    correct_lambda_nested_argument_for,\\n ])\\n def test_correct_lambda_body(\\n     assert_errors,\\n", "problem_statement": "WPS426 triggers on safe lambdas\\n# Bug report\\r\\n\\r\\n<!--\\r\\nHi, thanks for submitting a bug. We appreciate that.\\r\\n\\r\\nBut, we will need some information about what\'s wrong to help you.\\r\\n-->\\r\\n\\r\\n## What\'s wrong\\r\\n\\r\\n<!-- Describe what is not working. Please, attach a traceback. -->\\r\\n\\r\\n```python\\r\\ndef _show_titles(number_of_posts: int) -> Sequence[FutureResultE[str]]:\\r\\n    return [\\r\\n        # Notice how easily we compose async and sync functions:\\r\\n        _fetch_post(post_id).map(lambda post: post[\'title\'])\\r\\n        # TODO: try `for post_id in (2, 1, 0):` to see how async errors work\\r\\n        for post_id in range(1, number_of_posts + 1)\\r\\n    ]\\r\\n```\\r\\n\\r\\nRaises:\\r\\n\\r\\n```\\r\\n  32:9     WPS426 Found `lambda` in loop\'s body\\r\\n  _fetch_post(post_id).map(lambda post: post[\'title\'])\\r\\n  ^\\r\\n```\\r\\n\\r\\nBut it should not!\\r\\n\\r\\n## How is that should be\\r\\n\\r\\nWhy? Because this `lambda` does not use `post_id` in its body. It is safe to be used.\\r\\nSo, if `lambda` uses only its args - we can allow it.\\r\\n\\r\\n<!-- Describe how it should work. -->\\r\\n\\r\\n<!--\\r\\n\\r\\n  Love wemake-python-styleguide? \\u2764\\ufe0f\\r\\n  Do you want your bugs to be solved faster?\\r\\n\\r\\n  Please consider supporting our collective:\\r\\n\\ud83d\\udc49  https://opencollective.com/wemake-python-styleguide/donate\\r\\n\\r\\n-->\\r\\n\\r\\n## System information\\r\\n\\r\\n### flake8 information\\r\\n\\r\\nContents of `flake8 --bug-report`:\\r\\n\\r\\n### pip information\\r\\n\\r\\nContents of `pip freeze`:\\r\\n", "hints_text": "codecov[bot]: # [Codecov](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694?src=pr&el=h1) Report\\n> Merging [#1694](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694?src=pr&el=desc) into [master](https://codecov.io/gh/wemake-services/wemake-python-styleguide/commit/ece40b94d7649cc30a3e8a896513719d28099aed?el=desc) will **not change** coverage.\\n> The diff coverage is `100.00%`.\\n\\n[![Impacted file tree graph](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694/graphs/tree.svg?width=650&height=150&src=pr&token=yMZCKw0D02)](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694?src=pr&el=tree)\\n\\n```diff\\n@@            Coverage Diff            @@\\n##            master     #1694   +/-   ##\\n=========================================\\n  Coverage   100.00%   100.00%           \\n=========================================\\n  Files          106       106           \\n  Lines         5757      5733   -24     \\n  Branches      1286      1282    -4     \\n=========================================\\n- Hits          5757      5733   -24     \\n```\\n\\n\\n| [Impacted Files](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694?src=pr&el=tree) | Coverage \\u0394 | |\\n|---|---|---|\\n| [wemake\\\\_python\\\\_styleguide/visitors/ast/loops.py](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694/diff?src=pr&el=tree#diff-d2VtYWtlX3B5dGhvbl9zdHlsZWd1aWRlL3Zpc2l0b3JzL2FzdC9sb29wcy5weQ==) | `100.00% <100.00%> (\\u00f8)` | |\\n| [wemake\\\\_python\\\\_styleguide/compat/nodes.py](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694/diff?src=pr&el=tree#diff-d2VtYWtlX3B5dGhvbl9zdHlsZWd1aWRlL2NvbXBhdC9ub2Rlcy5weQ==) | `100.00% <0.00%> (\\u00f8)` | |\\n| [wemake\\\\_python\\\\_styleguide/logic/walrus.py](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694/diff?src=pr&el=tree#diff-d2VtYWtlX3B5dGhvbl9zdHlsZWd1aWRlL2xvZ2ljL3dhbHJ1cy5weQ==) | `100.00% <0.00%> (\\u00f8)` | |\\n| [wemake\\\\_python\\\\_styleguide/compat/routing.py](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694/diff?src=pr&el=tree#diff-d2VtYWtlX3B5dGhvbl9zdHlsZWd1aWRlL2NvbXBhdC9yb3V0aW5nLnB5) | `100.00% <0.00%> (\\u00f8)` | |\\n| [wemake\\\\_python\\\\_styleguide/logic/safe\\\\_eval.py](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694/diff?src=pr&el=tree#diff-d2VtYWtlX3B5dGhvbl9zdHlsZWd1aWRlL2xvZ2ljL3NhZmVfZXZhbC5weQ==) | `100.00% <0.00%> (\\u00f8)` | |\\n| [wemake\\\\_python\\\\_styleguide/compat/packaging.py](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694/diff?src=pr&el=tree#diff-d2VtYWtlX3B5dGhvbl9zdHlsZWd1aWRlL2NvbXBhdC9wYWNrYWdpbmcucHk=) | `100.00% <0.00%> (\\u00f8)` | |\\n| [wemake\\\\_python\\\\_styleguide/visitors/ast/functions.py](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694/diff?src=pr&el=tree#diff-d2VtYWtlX3B5dGhvbl9zdHlsZWd1aWRlL3Zpc2l0b3JzL2FzdC9mdW5jdGlvbnMucHk=) | `100.00% <0.00%> (\\u00f8)` | |\\n| [wemake\\\\_python\\\\_styleguide/visitors/ast/operators.py](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694/diff?src=pr&el=tree#diff-d2VtYWtlX3B5dGhvbl9zdHlsZWd1aWRlL3Zpc2l0b3JzL2FzdC9vcGVyYXRvcnMucHk=) | `100.00% <0.00%> (\\u00f8)` | |\\n\\n------\\n\\n[Continue to review full report at Codecov](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694?src=pr&el=continue).\\n> **Legend** - [Click here to learn more](https://docs.codecov.io/docs/codecov-delta)\\n> `\\u0394 = absolute <relative> (impact)`, `\\u00f8 = not affected`, `? = missing data`\\n> Powered by [Codecov](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694?src=pr&el=footer). Last update [ece40b9...4825098](https://codecov.io/gh/wemake-services/wemake-python-styleguide/pull/1694?src=pr&el=lastupdated). Read the [comment docs](https://docs.codecov.io/docs/pull-request-comments).\\n", "created_at": "2020-10-24 16:39:32", "FAIL_TO_PASS": ["tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py::test_correct_lambda_body[regular_wrapper-\\\\ndef", "tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py::test_correct_lambda_body[async_wrapper-\\\\ndef"], "PASS_TO_PASS": ["tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py::test_lambda_body[async_wrapper-\\\\ndef", "tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py::test_lambda_body[regular_wrapper-\\\\ndef", "tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py::test_lambda_body[regular_wrapper-\\\\nwhile", "tests/test_visitors/test_ast/test_loops/test_loops/test_lambda_in_loop.py::test_lambda_body[async_wrapper-\\\\nwhile"], "environment_setup_commit": "faa465eec62a5a9590f121462f532bf40c33dafe", "version": "0.14", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .[dev]", "pre_install": null, "pip_packages": ["pytest", "pytest-cov", "pytest-randomly", "coverage-conditional-plugin"], "reqs_path": ["requirements/base.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 2}}, {"instance_id": "auth0__auth0-python-64", "repo": "auth0/auth0-python", "base_commit": "0d7decce20e04703d11a614bffc8f9d9f6e9723e", "patch": "diff --git a/auth0/v3/authentication/base.py b/auth0/v3/authentication/base.py\\nindex b4ccd1d..778912c 100644\\n--- a/auth0/v3/authentication/base.py\\n+++ b/auth0/v3/authentication/base.py\\n@@ -19,8 +19,8 @@ class AuthenticationBase(object):\\n         except ValueError:\\n             return response.text\\n         else:\\n-            if \'error\' in text:\\n-                raise Auth0Error(status_code=text[\'error\'],\\n-                                 error_code=text[\'error\'],\\n-                                 message=text[\'error_description\'])\\n+            if response.status_code is None or response.status_code >= 400:\\n+                raise Auth0Error(status_code=response.status_code,\\n+                                 error_code=text.get(\'error\', \'\'),\\n+                                 message=text.get(\'error_description\', \'\'))\\n         return text\\n", "test_patch": "diff --git a/auth0/v3/test/authentication/test_base.py b/auth0/v3/test/authentication/test_base.py\\nindex c058864..d6539c8 100644\\n--- a/auth0/v3/test/authentication/test_base.py\\n+++ b/auth0/v3/test/authentication/test_base.py\\n@@ -10,6 +10,7 @@ class TestBase(unittest.TestCase):\\n     def test_post(self, mock_post):\\n         ab = AuthenticationBase()\\n \\n+        mock_post.return_value.status_code = 200\\n         mock_post.return_value.text = \'{\\"x\\": \\"y\\"}\'\\n \\n         data = ab.post(\'the-url\', data={\'a\': \'b\'}, headers={\'c\': \'d\'})\\n@@ -23,12 +24,14 @@ class TestBase(unittest.TestCase):\\n     def test_post_error(self, mock_post):\\n         ab = AuthenticationBase()\\n \\n-        mock_post.return_value.text = \'{\\"error\\": \\"e0\\",\' \\\\\\n-                                      \'\\"error_description\\": \\"desc\\"}\'\\n+        for error_status in [400, 500, None]:\\n+            mock_post.return_value.status_code = error_status\\n+            mock_post.return_value.text = \'{\\"error\\": \\"e0\\",\' \\\\\\n+                                          \'\\"error_description\\": \\"desc\\"}\'\\n \\n-        with self.assertRaises(Auth0Error) as context:\\n-            data = ab.post(\'the-url\', data={\'a\': \'b\'}, headers={\'c\': \'d\'})\\n+            with self.assertRaises(Auth0Error) as context:\\n+                data = ab.post(\'the-url\', data={\'a\': \'b\'}, headers={\'c\': \'d\'})\\n \\n-        self.assertEqual(context.exception.status_code, \'e0\')\\n-        self.assertEqual(context.exception.error_code, \'e0\')\\n-        self.assertEqual(context.exception.message, \'desc\')\\n+            self.assertEqual(context.exception.status_code, error_status)\\n+            self.assertEqual(context.exception.error_code, \'e0\')\\n+            self.assertEqual(context.exception.message, \'desc\')\\n", "problem_statement": "Auth0Error not being raised due to inconsistent API error responses\\nCurrently `Auth0Error` is raised whenever the API response contains an `error` key in the response JSON. Unfortunately at least one endpoint (`/dbconnections/signup`) returns inconsistent error messages (that do not always contain the `error` key) for different scenarios and as a result `Auth0Error` is not raised when an error occurs.\\r\\n\\r\\nExamples of inconsistent responses:\\r\\n* when making a signup request with an email that is already registered: \\r\\n```\\r\\n{\\r\\n    \\"code\\": \\"user_exists\\", \\r\\n    \\"description\\": \\"The user already exists.\\", \\r\\n    \\"name\\": \\"BadRequestError\\", \\r\\n    \\"statusCode\\": 400\\r\\n}\\r\\n```\\r\\n\\r\\n* when making a request with an invalid `client_id` (with public signup disabled)\\r\\n```\\r\\n{\\r\\n    \\"name\\": \\"NotFoundError\\", \\r\\n    \\"statusCode\\": 404\\r\\n}\\r\\n```\\r\\n\\r\\n* when making a request with an invalid password (with password strength enabled)\\r\\n```\\r\\n{\\r\\n    \\"code\\": \\"invalid_password\\", \\r\\n    \\"description\\": {\\r\\n        \\"rules\\": [\\r\\n            {\\r\\n                \\"code\\": \\"lengthAtLeast\\", \\r\\n                \\"format\\": [\\r\\n                    6\\r\\n                ], \\r\\n                \\"message\\": \\"At least %d characters in length\\", \\r\\n                \\"verified\\": false\\r\\n            }\\r\\n        ], \\r\\n        \\"verified\\": false\\r\\n    }, \\r\\n    \\"message\\": \\"Password is too weak\\", \\r\\n    \\"name\\": \\"PasswordStrengthError\\", \\r\\n    \\"policy\\": \\"* At least 6 characters in length\\", \\r\\n    \\"statusCode\\": 400\\r\\n}\\r\\n```\\r\\n\\r\\n* when making a request with missing password\\r\\n```\\r\\n{\\r\\n    \\"error\\": \\"password is required\\"\\r\\n}\\r\\n```\\r\\n\\r\\nThe last example highlights a related issue. Even though there is an `error` key, a `KeyError` exception will ultimately occur because `AuthenticationBase._process_response` assumes the additional existence of an `error_description` key when creating the `Auth0Error` and setting its message.", "hints_text": "", "created_at": "2017-05-19 09:54:07", "FAIL_TO_PASS": ["auth0/v3/test/authentication/test_base.py::TestBase::test_post_error"], "PASS_TO_PASS": ["auth0/v3/test/authentication/test_base.py::TestBase::test_post"], "environment_setup_commit": "a9fd8e39c6aee52bf1654d96c494287121d9e141", "version": "3.1", "license_name": "MIT License", "install_config": {"python": "3.9", "install": "pip install -e .[dev]", "pre_install": null, "pip_packages": ["pytest", "pytest-cov"], "reqs_path": ["requirements.txt"], "test_cmd": "pytest --no-header -rA --tb=line --color=no -p no:cacheprovider -W ignore::DeprecationWarning", "env_vars": null}, "meta": {"difficulty": 1, "files": 1}}]')

In [ ]:
import json, os, time, concurrent.futures as cf
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], check=True)   # Kaggle's `python -m venv` is broken (ensurepip)
exec(WORKER_SRC)   # defines run_task(task, work_root)
WORK = Path('/tmp/spike'); WORK.mkdir(exist_ok=True)
OUTDIR = Path('/kaggle/working'); OUTDIR.mkdir(exist_ok=True)
t0 = time.time(); results = []
def _one(t):
    try: return run_task(t, WORK)
    except Exception as e: return {'id': t['instance_id'], 'verdict': 'harness_error', 'reason': repr(e)[:300]}
with cf.ThreadPoolExecutor(max_workers=4) as ex:
    futs = {ex.submit(_one, t): t['instance_id'] for t in TASKS}
    for f in cf.as_completed(futs):
        r = f.result(); results.append(r)
        print(f"[{time.time() - t0:6.0f}s] {r['id']:55s} {r['verdict']:14s} {r.get('reason', '')[:100]}", flush=True)
        (OUTDIR / 'spike_results.json').write_text(json.dumps(results, indent=1))
import collections
print('\nverdicts:', dict(collections.Counter(r['verdict'] for r in results)), '| wall', round(time.time() - t0), 's')
print('valid tasks:', [r['id'] for r in results if r['verdict'] == 'valid'])